# Disaster Damage Assessment Pipeline
## Stage 1: Building Segmentation + Stage 2: Damage Classification

This notebook implements an end-to-end deep learning pipeline for satellite image analysis:
- **Stage 1**: U-Net++ with EfficientNet-B3 backbone for building segmentation
- **Stage 2**: Dual-input ResNet50 for damage classification
- **Inference**: Predict damage maps on new image pairs

Dataset: xView2 (pre/post disaster satellite imagery)

## Cell 1: Imports and Setup

In [34]:
import os
import json
import cv2
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from glob import glob
from tqdm import tqdm
from sklearn.metrics import f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from torchvision import models

# Check GPU
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

GPU Available: True
GPU Name: NVIDIA A100-PCIE-40GB
GPU Memory: 42.29 GB


## Cell 2: Configuration

In [35]:
# ─────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# ── Data paths (EDIT THIS TO MATCH YOUR DATA LOCATION) ──────────
BASE_DIR = Path.home() / 'Satellite-based disaster damage'

print('BASE_DIR           :', BASE_DIR)
print('train/images exists:', (BASE_DIR / 'train' / 'images').exists())
print('train/labels exists:', (BASE_DIR / 'train' / 'labels').exists())
print('test/images  exists:', (BASE_DIR / 'test'  / 'images').exists())

train_img_dir = BASE_DIR / 'train' / 'images'
test_img_dir  = BASE_DIR / 'test'  / 'images'
train_files   = list(train_img_dir.glob('*')) if train_img_dir.exists() else []
test_files    = list(test_img_dir.glob('*'))  if test_img_dir.exists()  else []

print(f'\nTrain folder — {len(train_files)} total files')
print('  First 6:', [f.name for f in sorted(train_files)[:6]])
print(f'\nTest  folder — {len(test_files)} total files')
print('  First 6:', [f.name for f in sorted(test_files)[:6]])

assert BASE_DIR.exists(), f'DATA ROOT not found: {BASE_DIR}'
print('\n✓ All paths look good')

BASE_DIR           : /home/devnurma/Satellite-based disaster damage
train/images exists: True
train/labels exists: True
test/images  exists: True

Train folder — 5598 total files
  First 6: ['guatemala-volcano_00000000_post_disaster.png', 'guatemala-volcano_00000000_pre_disaster.png', 'guatemala-volcano_00000001_post_disaster.png', 'guatemala-volcano_00000001_pre_disaster.png', 'guatemala-volcano_00000002_post_disaster.png', 'guatemala-volcano_00000002_pre_disaster.png']

Test  folder — 1866 total files
  First 6: ['test_post_00000.png', 'test_post_00001.png', 'test_post_00002.png', 'test_post_00003.png', 'test_post_00004.png', 'test_post_00005.png']

✓ All paths look good


In [36]:
class Config:
    # ── Paths ──────────────────────────────────────────────────────
    DATA_ROOT      = BASE_DIR
    TRAIN_IMG_DIR  = str(DATA_ROOT / 'train' / 'images')
    TRAIN_LBL_DIR  = str(DATA_ROOT / 'train' / 'labels')
    TEST_IMG_DIR   = str(DATA_ROOT / 'test'  / 'images')
    OUTPUT_DIR     = './outputs'
    CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
    LOG_DIR        = os.path.join(OUTPUT_DIR, 'logs')
    VIZ_DIR        = os.path.join(OUTPUT_DIR, 'visualizations')
    PRED_DIR       = os.path.join(OUTPUT_DIR, 'predictions')
    PATCH_DIR      = os.path.join(OUTPUT_DIR, 'patches')

    # ── Segmentation ───────────────────────────────────────────────
    SEG_IMG_SIZE   = 512
    SEG_BATCH_SIZE = 4
    SEG_LR         = 1e-4
    SEG_EPOCHS     = 100
    SEG_THRESHOLD  = 0.5
    SEG_MIN_AREA   = 100      # pixels — blobs smaller than this are removed

    # ── Classification ─────────────────────────────────────────────
    CLS_PATCH_SIZE  = 224
    CLS_BATCH_SIZE  = 16
    CLS_LR          = 1e-4
    CLS_EPOCHS      = 50
    CLS_NUM_CLASSES = 4

    # ── Damage label map (xView2 convention) ───────────────────────
    DAMAGE_CLASSES = {
        0: 'no-damage',
        1: 'minor-damage',
        2: 'major-damage',
        3: 'destroyed',
    }

    # Color overlays for visualization (BGR)
    DAMAGE_COLORS = {
        0: (0,   200,  0),    # green  – no damage
        1: (0,   200, 255),   # yellow – minor
        2: (0,   100, 255),   # orange – major
        3: (0,     0, 255),   # red    – destroyed
    }

    # ── Reproducibility ────────────────────────────────────────────
    SEED = 42

    # ── Device ─────────────────────────────────────────────────────
    DEVICE = 'cuda'   # falls back to cpu in train scripts if needed


def make_dirs():
    """Create all output directories."""
    for d in [Config.OUTPUT_DIR, Config.CHECKPOINT_DIR,
              Config.LOG_DIR, Config.VIZ_DIR, Config.PRED_DIR, Config.PATCH_DIR]:
        os.makedirs(d, exist_ok=True)
    print('✓ Output directories created')

make_dirs()

✓ Output directories created


## Cell 3: Model Definitions

In [37]:
# ─────────────────────────────────────────────────────────────────
# Stage 1 — U-Net++ with EfficientNet-B3 backbone
# ─────────────────────────────────────────────────────────────────

def build_segmentation_model(encoder_name: str = 'efficientnet-b3',
                              encoder_weights: str = 'imagenet',
                              in_channels: int = 3,
                              num_classes: int = 1) -> nn.Module:
    """
    U-Net++ segmentation model.
    Returns logits of shape (B, 1, H, W).
    Apply sigmoid + threshold at inference time.
    """
    model = smp.UnetPlusPlus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,         # raw logits → use BCEWithLogitsLoss
        decoder_attention_type='scse',  # squeeze-and-excitation attention
    )
    return model

print('✓ Segmentation model builder defined')

✓ Segmentation model builder defined


In [38]:
# ─────────────────────────────────────────────────────────────────
# Stage 2 — Dual-input Siamese ResNet50
# ─────────────────────────────────────────────────────────────────

class DualResNet50(nn.Module):
    """
    Siamese ResNet50 for damage classification.
    Both pre and post patches are passed through the SAME ResNet50 backbone
    (shared weights). Features are combined via absolute difference.
    """

    def __init__(self, num_classes: int = 4,
                 pretrained: bool = True,
                 dropout: float = 0.4):
        super().__init__()

        # ── Shared encoder (strip the classification head) ──────────
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet50(weights=weights)
        self.encoder = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )  # output: (B, 2048, 7, 7) for 224×224 input

        self.pool = nn.AdaptiveAvgPool2d(1)   # → (B, 2048, 1, 1)

        # ── Classification head ─────────────────────────────────────
        feat_dim = 2048
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Extract pooled feature vector from a single image."""
        f = self.encoder(x)          # (B, 2048, 7, 7)
        f = self.pool(f)             # (B, 2048, 1, 1)
        f = f.flatten(1)             # (B, 2048)
        return f

    def forward(self, pre: torch.Tensor, post: torch.Tensor) -> torch.Tensor:
        f_pre  = self.encode(pre)
        f_post = self.encode(post)
        # Absolute difference → forces the network to learn *change*
        diff   = torch.abs(f_post - f_pre)   # (B, 2048)
        return self.classifier(diff)          # (B, num_classes)

print('✓ DualResNet50 model defined')

✓ DualResNet50 model defined


In [39]:
# ─────────────────────────────────────────────────────────────────
# Loss Functions
# ─────────────────────────────────────────────────────────────────

class DiceBCELoss(nn.Module):
    """Combined Dice + BCE loss for binary segmentation."""

    def __init__(self, bce_weight: float = 0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        bce_loss  = self.bce(logits, targets)

        probs     = torch.sigmoid(logits)
        smooth    = 1e-6
        inter     = (probs * targets).sum(dim=(2, 3))
        dice_loss = 1 - (2 * inter + smooth) / (
            probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + smooth)
        dice_loss = dice_loss.mean()

        return self.bce_weight * bce_loss + (1 - self.bce_weight) * dice_loss

print('✓ Loss functions defined')

✓ Loss functions defined


In [40]:
# ─────────────────────────────────────────────────────────────────
# Quick sanity check
# ─────────────────────────────────────────────────────────────────

device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

print('\n=== Segmentation model ===')
seg = build_segmentation_model()
x   = torch.randn(2, 3, 512, 512)
out = seg(x)
print(f'  Input : {x.shape}')
print(f'  Output: {out.shape}')   # expect (2, 1, 512, 512)
del seg, x, out

print('\n=== Classification model ===')
cls  = DualResNet50(num_classes=4)
pre  = torch.randn(4, 3, 224, 224)
post = torch.randn(4, 3, 224, 224)
out  = cls(pre, post)
print(f'  Input : pre {pre.shape}, post {post.shape}')
print(f'  Output: {out.shape}')   # expect (4, 4)
del cls, pre, post, out

print('\n✓ Both models initialised successfully')

Using device: cuda

=== Segmentation model ===
  Input : torch.Size([2, 3, 512, 512])
  Output: torch.Size([2, 1, 512, 512])

=== Classification model ===
  Input : pre torch.Size([4, 3, 224, 224]), post torch.Size([4, 3, 224, 224])
  Output: torch.Size([4, 4])

✓ Both models initialised successfully


## Cell 4: Dataset Utilities

In [41]:
# ─────────────────────────────────────────────────────────────────
# Helper: build binary building mask from xView2 GeoJSON label
# ─────────────────────────────────────────────────────────────────

def json_to_mask(json_path: str, height: int = 1024, width: int = 1024) -> np.ndarray:
    """
    Parse an xView2 GeoJSON annotation file and rasterise all building
    polygons into a binary uint8 mask (0 = background, 255 = building).
    """
    with open(json_path) as f:
        data = json.load(f)

    mask = np.zeros((height, width), dtype=np.uint8)

    for feat in data.get('features', {}).get('xy', []):
        geom = feat.get('wkt', '')
        if not geom.startswith('POLYGON'):
            continue
        coords_str = geom.replace('POLYGON ((', '').replace('))', '').strip()
        pts = []
        for pair in coords_str.split(','):
            pair = pair.strip()
            if pair:
                x, y = pair.split()
                pts.append([float(x), float(y)])
        if len(pts) < 3:
            continue
        poly = np.array(pts, dtype=np.int32)
        cv2.fillPoly(mask, [poly], 255)

    return mask


def json_to_damage_mask(json_path: str, height: int = 1024,
                         width: int = 1024) -> np.ndarray:
    """
    Parse a POST-disaster xView2 JSON and rasterise damage subtypes:
        no-damage   → 1
        minor-damage→ 2
        major-damage→ 3
        destroyed   → 4
        (0 = background / unknown)
    """
    subtype_map = {
        'no-damage':    1,
        'minor-damage': 2,
        'major-damage': 3,
        'destroyed':    4,
        'un-classified': 0,
    }

    with open(json_path) as f:
        data = json.load(f)

    mask = np.zeros((height, width), dtype=np.uint8)

    for feat in data.get('features', {}).get('xy', []):
        geom     = feat.get('wkt', '')
        subtype  = feat.get('properties', {}).get('subtype', 'un-classified')
        label    = subtype_map.get(subtype, 0)

        if not geom.startswith('POLYGON') or label == 0:
            continue

        coords_str = geom.replace('POLYGON ((', '').replace('))', '').strip()
        pts = []
        for pair in coords_str.split(','):
            pair = pair.strip()
            if pair:
                x, y = pair.split()
                pts.append([float(x), float(y)])
        if len(pts) < 3:
            continue
        poly = np.array(pts, dtype=np.int32)
        cv2.fillPoly(mask, [poly], label)

    return mask

print('✓ Mask conversion functions defined')

✓ Mask conversion functions defined


In [42]:
# ─────────────────────────────────────────────────────────────────
# Stage 1 — Segmentation Dataset
# ─────────────────────────────────────────────────────────────────

def get_seg_transforms(train: bool = True):
    if train:
        return A.Compose([
            A.Resize(Config.SEG_IMG_SIZE, Config.SEG_IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2,
                                       contrast_limit=0.2, p=0.4),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std =(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(Config.SEG_IMG_SIZE, Config.SEG_IMG_SIZE),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std =(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])


class SegmentationDataset(Dataset):
    """Returns (pre_image_tensor, binary_mask_tensor) pairs."""

    def __init__(self, img_dir: str, lbl_dir: str,
                 transform=None, img_size: int = 1024):
        self.img_dir   = Path(img_dir)
        self.lbl_dir   = Path(lbl_dir)
        self.transform = transform
        self.img_size  = img_size

        self.samples = []
        for img_path in sorted(self.img_dir.glob('*_pre_disaster.png')):
            stem      = img_path.stem
            json_path = self.lbl_dir / f'{stem}.json'
            if json_path.exists():
                self.samples.append((str(img_path), str(json_path)))

        print(f'SegmentationDataset: {len(self.samples)} samples found')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, json_path = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w  = image.shape[:2]

        mask  = json_to_mask(json_path, height=h, width=w)
        mask  = (mask > 0).astype(np.float32)

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug['image']
            mask  = aug['mask'].unsqueeze(0)
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.
            mask  = torch.from_numpy(mask).unsqueeze(0)

        return image, mask

print('✓ Segmentation dataset defined')

✓ Segmentation dataset defined


In [43]:
# ─────────────────────────────────────────────────────────────────
# Stage 2 — Classification Dataset (pre/post patch pairs)
# ─────────────────────────────────────────────────────────────────

def get_cls_transforms(train: bool = True):
    if train:
        return A.Compose([
            A.Resize(Config.CLS_PATCH_SIZE, Config.CLS_PATCH_SIZE),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.1,
                                       contrast_limit=0.1, p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std =(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(Config.CLS_PATCH_SIZE, Config.CLS_PATCH_SIZE),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std =(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])


class PatchClassificationDataset(Dataset):
    """Expects patch_dir/pre/, patch_dir/post/, patch_dir/labels.json"""

    def __init__(self, patch_dir: str, transform=None):
        self.patch_dir  = Path(patch_dir)
        self.transform  = transform
        self.pre_dir    = self.patch_dir / 'pre'
        self.post_dir   = self.patch_dir / 'post'
        labels_file     = self.patch_dir / 'labels.json'

        # ── Guard: raise a clear error if patch extraction was skipped ──
        if not labels_file.exists():
            raise FileNotFoundError(
                f"Patch labels not found at '{labels_file}'.\n"
                "You must run extract_all_patches() before train_classification().\n"
                "Example:\n"
                "  extract_all_patches(\n"
                "      img_dir    = Config.TRAIN_IMG_DIR,\n"
                "      lbl_dir    = Config.TRAIN_LBL_DIR,\n"
                "      output_dir = Config.PATCH_DIR,\n"
                "      use_gt_mask= True,\n"
                "  )"
            )

        with open(labels_file) as f:
            self.labels = json.load(f)

        self.ids = sorted(self.labels.keys())
        print(f'PatchClassificationDataset: {len(self.ids)} patch pairs')

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        patch_id = self.ids[idx]
        label    = int(self.labels[patch_id])

        pre_path  = self.pre_dir  / f'{patch_id}.png'
        post_path = self.post_dir / f'{patch_id}.png'

        pre  = cv2.cvtColor(cv2.imread(str(pre_path)),  cv2.COLOR_BGR2RGB)
        post = cv2.cvtColor(cv2.imread(str(post_path)), cv2.COLOR_BGR2RGB)

        if self.transform:
            pre_aug  = self.transform(image=pre)
            post_aug = self.transform(image=post)
            pre  = pre_aug['image']
            post = post_aug['image']
        else:
            pre  = torch.from_numpy(pre.transpose(2,0,1)).float() / 255.
            post = torch.from_numpy(post.transpose(2,0,1)).float() / 255.

        return pre, post, torch.tensor(label, dtype=torch.long)

print('✓ Classification dataset defined')

✓ Classification dataset defined


In [44]:
# ─────────────────────────────────────────────────────────────────
# Utility — compute class weights for imbalanced classification
# ─────────────────────────────────────────────────────────────────

def compute_class_weights(patch_dir: str,
                          num_classes: int = Config.CLS_NUM_CLASSES) -> torch.Tensor:
    """Inverse-frequency class weights for CrossEntropyLoss."""
    labels_file = Path(patch_dir) / 'labels.json'
    with open(labels_file) as f:
        labels = json.load(f)

    counts = np.zeros(num_classes, dtype=np.float32)
    for v in labels.values():
        counts[int(v)] += 1

    counts = np.where(counts == 0, 1, counts)
    weights = counts.sum() / (num_classes * counts)
    print('Class counts :', counts.astype(int))
    print('Class weights:', np.round(weights, 3))
    return torch.tensor(weights, dtype=torch.float32)

print('✓ Class weight computation defined')

✓ Class weight computation defined


## Cell 5: Patch Extraction (Bridge between Stage 1 and Stage 2)

In [45]:
# ─────────────────────────────────────────────────────────────────
# Core extraction function (single image pair)
# ─────────────────────────────────────────────────────────────────

def extract_patches_from_pair(pre_img_path: str,
                               post_img_path: str,
                               mask: np.ndarray,
                               post_json_path: str = None,
                               patch_size: int = Config.CLS_PATCH_SIZE,
                               min_area: int = Config.SEG_MIN_AREA):
    """
    Given a binary mask and the pre/post image pair, find every connected
    building blob and crop the matching patches.
    """
    pre  = cv2.cvtColor(cv2.imread(pre_img_path),  cv2.COLOR_BGR2RGB)
    post = cv2.cvtColor(cv2.imread(post_img_path), cv2.COLOR_BGR2RGB)
    h, w = pre.shape[:2]

    if mask.shape != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    if post_json_path and Path(post_json_path).exists():
        damage_mask = json_to_damage_mask(post_json_path, height=h, width=w)
    else:
        damage_mask = np.full((h, w), fill_value=255, dtype=np.uint8)

    binary   = (mask > 0).astype(np.uint8) * 255
    num_comp, labels_cc, stats, _ = cv2.connectedComponentsWithStats(binary)

    patches = []

    for comp_id in range(1, num_comp):
        area = stats[comp_id, cv2.CC_STAT_AREA]
        if area < min_area:
            continue

        x = stats[comp_id, cv2.CC_STAT_LEFT]
        y = stats[comp_id, cv2.CC_STAT_TOP]
        bw = stats[comp_id, cv2.CC_STAT_WIDTH]
        bh = stats[comp_id, cv2.CC_STAT_HEIGHT]

        margin = 10
        x1 = max(0, x - margin)
        y1 = max(0, y - margin)
        x2 = min(w, x + bw + margin)
        y2 = min(h, y + bh + margin)

        pre_crop  = pre[y1:y2,  x1:x2]
        post_crop = post[y1:y2, x1:x2]

        if pre_crop.size == 0 or post_crop.size == 0:
            continue

        pre_patch  = cv2.resize(pre_crop,  (patch_size, patch_size))
        post_patch = cv2.resize(post_crop, (patch_size, patch_size))

        building_region = damage_mask[y1:y2, x1:x2]
        comp_mask       = (labels_cc[y1:y2, x1:x2] == comp_id)
        damage_vals     = building_region[comp_mask]
        valid           = damage_vals[(damage_vals >= 1) & (damage_vals <= 4)]

        if len(valid) > 0:
            damage = int(np.bincount(valid).argmax()) - 1
        else:
            damage = -1

        patches.append({
            'pre_patch':  pre_patch,
            'post_patch': post_patch,
            'damage':     damage,
            'bbox':       (x1, y1, x2 - x1, y2 - y1),
        })

    return patches

print('✓ Patch extraction defined')

✓ Patch extraction defined


In [46]:
# ─────────────────────────────────────────────────────────────────
# Batch extractor — runs over entire dataset split
# ─────────────────────────────────────────────────────────────────

def extract_all_patches(img_dir: str,
                         lbl_dir: str,
                         output_dir: str,
                         mask_dir: str = None,
                         use_gt_mask: bool = True):
    """
    Extract building patches for every pre/post pair in img_dir.
    """
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)
    out_dir = Path(output_dir)
    pre_out = out_dir / 'pre'
    post_out = out_dir / 'post'
    pre_out.mkdir(parents=True, exist_ok=True)
    post_out.mkdir(parents=True, exist_ok=True)

    labels_dict = {}
    patch_count = 0

    pre_images = sorted(img_dir.glob('*_pre_disaster.png'))
    print(f'Found {len(pre_images)} pre-disaster images to process...')

    for pre_path in tqdm(pre_images, desc='Extracting patches'):
        stem      = pre_path.stem
        post_name = stem.replace('_pre_', '_post_') + '.png'
        post_path = img_dir / post_name

        if not post_path.exists():
            continue

        if use_gt_mask:
            pre_json = lbl_dir / f'{stem}.json'
            if not pre_json.exists():
                continue
            img_tmp = cv2.imread(str(pre_path))
            h, w    = img_tmp.shape[:2]
            mask    = json_to_mask(str(pre_json), height=h, width=w)
        else:
            mask_path = Path(mask_dir) / f'{stem}_mask.png'
            if not mask_path.exists():
                continue
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        post_json_name = stem.replace('_pre_', '_post_') + '.json'
        post_json      = lbl_dir / post_json_name

        patches = extract_patches_from_pair(
            pre_img_path=str(pre_path),
            post_img_path=str(post_path),
            mask=mask,
            post_json_path=str(post_json) if post_json.exists() else None,
        )

        for i, p in enumerate(patches):
            if p['damage'] == -1:
                continue

            patch_id = f'{stem}_{i:04d}'
            cv2.imwrite(str(pre_out  / f'{patch_id}.png'),
                        cv2.cvtColor(p['pre_patch'],  cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(post_out / f'{patch_id}.png'),
                        cv2.cvtColor(p['post_patch'], cv2.COLOR_RGB2BGR))
            labels_dict[patch_id] = p['damage']
            patch_count += 1

    labels_out = out_dir / 'labels.json'
    with open(labels_out, 'w') as f:
        json.dump(labels_dict, f)

    print(f'✓ Extracted {patch_count} labelled patches → {output_dir}')

    dist = Counter(labels_dict.values())
    for cls_id, name in Config.DAMAGE_CLASSES.items():
        print(f'  class {cls_id} ({name}): {dist.get(cls_id, 0)}')

print('✓ Batch patch extraction defined')

✓ Batch patch extraction defined


## Cell 6: Stage 1 — Segmentation Training

In [47]:
# ─────────────────────────────────────────────────────────────────
# Segmentation Metrics
# ─────────────────────────────────────────────────────────────────

def iou_score(pred_mask: np.ndarray, true_mask: np.ndarray,
              threshold: float = 0.5) -> float:
    pred = (pred_mask > threshold).astype(np.uint8)
    true = (true_mask  > threshold).astype(np.uint8)
    inter = (pred & true).sum()
    union = (pred | true).sum()
    return inter / (union + 1e-6)


def dice_score(pred_mask: np.ndarray, true_mask: np.ndarray,
               threshold: float = 0.5) -> float:
    pred = (pred_mask > threshold).astype(np.uint8)
    true = (true_mask  > threshold).astype(np.uint8)
    inter = (pred & true).sum()
    return (2 * inter) / (pred.sum() + true.sum() + 1e-6)

print('✓ Segmentation metrics defined')

✓ Segmentation metrics defined


In [48]:
# ─────────────────────────────────────────────────────────────────
# Segmentation Training Helpers
# ─────────────────────────────────────────────────────────────────

def train_seg_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0.0

    for batch_idx, (images, masks) in enumerate(loader):
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            logits = model(images)
            loss   = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate_seg(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_iou    = []
    all_dice   = []

    for images, masks in loader:
        images = images.to(device)
        masks  = masks.to(device)

        with torch.cuda.amp.autocast():
            logits = model(images)
            loss   = criterion(logits, masks)

        total_loss += loss.item()

        probs = torch.sigmoid(logits).cpu().numpy()
        gt    = masks.cpu().numpy()

        for p, g in zip(probs, gt):
            all_iou.append(iou_score(p[0], g[0]))
            all_dice.append(dice_score(p[0], g[0]))

    return (total_loss / len(loader),
            float(np.mean(all_iou)),
            float(np.mean(all_dice)))

print('✓ Segmentation training helpers defined')

✓ Segmentation training helpers defined


In [49]:
# ─────────────────────────────────────────────────────────────────
# Main Segmentation Training Loop
# ─────────────────────────────────────────────────────────────────

def train_segmentation():
    device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Training on: {device}')

    # ── Dataset ──────────────────────────────────────────────────
    full_dataset = SegmentationDataset(
        img_dir   = Config.TRAIN_IMG_DIR,
        lbl_dir   = Config.TRAIN_LBL_DIR,
        transform = None,
    )

    n_total = len(full_dataset)
    n_val   = max(1, int(0.15 * n_total))
    n_train = n_total - n_val

    train_ds, val_ds = random_split(
        full_dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(Config.SEED)
    )

    train_ds.dataset.transform = get_seg_transforms(train=True)
    val_ds.dataset.transform   = get_seg_transforms(train=False)

    train_loader = DataLoader(train_ds, batch_size=Config.SEG_BATCH_SIZE,
                              shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=Config.SEG_BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

    # ── Model ────────────────────────────────────────────────────
    model     = build_segmentation_model().to(device)
    criterion = DiceBCELoss(bce_weight=0.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.SEG_LR,
                                  weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.SEG_EPOCHS, eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler()

    writer    = SummaryWriter(log_dir=os.path.join(Config.LOG_DIR, 'segmentation'))

    best_iou      = 0.0
    best_ckpt     = os.path.join(Config.CHECKPOINT_DIR, 'seg_best.pth')
    patience      = 7
    patience_ctr  = 0

    print(f'\n{"Epoch":>6} | {"Train Loss":>11} | {"Val Loss":>9} | '
          f'{"IoU":>6} | {"Dice":>6} | {"LR":>9}')
    print('-' * 65)

    for epoch in range(1, Config.SEG_EPOCHS + 1):
        t0 = time.time()

        train_loss = train_seg_one_epoch(model, train_loader, optimizer,
                                         criterion, device, scaler)
        val_loss, iou, dice = validate_seg(model, val_loader, criterion, device)

        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        elapsed = time.time() - t0
        print(f'{epoch:>6} | {train_loss:>11.4f} | {val_loss:>9.4f} | '
              f'{iou:>6.4f} | {dice:>6.4f} | {lr:>9.2e}  [{elapsed:.0f}s]')

        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val',   val_loss,   epoch)
        writer.add_scalar('Metric/IoU', iou,        epoch)
        writer.add_scalar('Metric/Dice',dice,       epoch)

        if iou > best_iou:
            best_iou     = iou
            patience_ctr = 0
            torch.save({
                'epoch':       epoch,
                'state_dict':  model.state_dict(),
                'optimizer':   optimizer.state_dict(),
                'best_iou':    best_iou,
            }, best_ckpt)
            print(f'  ✓ Best model saved  (IoU={best_iou:.4f})')
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping triggered at epoch {epoch}')
                break

    writer.close()
    print(f'\n✓ Segmentation training complete. Best IoU: {best_iou:.4f}')
    print(f'  Checkpoint: {best_ckpt}')
    return best_ckpt

print('✓ Segmentation training function defined')

✓ Segmentation training function defined


### Run Segmentation Training

Uncomment and run the cell below to start Stage 1 training:

In [50]:
# Uncomment to run Stage 1 training
seg_ckpt = train_segmentation()

Training on: cuda
SegmentationDataset: 2799 samples found
Train: 2380 | Val: 419

 Epoch |  Train Loss |  Val Loss |    IoU |   Dice |        LR
-----------------------------------------------------------------


/tmp/ipykernel_85300/2561978118.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()
/tmp/ipykernel_85300/1683020645.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_85300/1683020645.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


     1 |      0.4531 |    0.3083 | 0.4416 | 0.5546 |  1.00e-04  [110s]
  ✓ Best model saved  (IoU=0.4416)
     2 |      0.2919 |    0.2558 | 0.4841 | 0.5917 |  9.99e-05  [109s]
  ✓ Best model saved  (IoU=0.4841)
     3 |      0.2542 |    0.2396 | 0.5050 | 0.6086 |  9.98e-05  [110s]
  ✓ Best model saved  (IoU=0.5050)
     4 |      0.2372 |    0.2383 | 0.5067 | 0.6108 |  9.96e-05  [110s]
  ✓ Best model saved  (IoU=0.5067)
     5 |      0.2265 |    0.2333 | 0.5152 | 0.6177 |  9.94e-05  [109s]
  ✓ Best model saved  (IoU=0.5152)
     6 |      0.2195 |    0.2294 | 0.5210 | 0.6202 |  9.91e-05  [110s]
  ✓ Best model saved  (IoU=0.5210)
     7 |      0.2129 |    0.2324 | 0.5165 | 0.6140 |  9.88e-05  [110s]
     8 |      0.2106 |    0.2256 | 0.5293 | 0.6278 |  9.84e-05  [110s]
  ✓ Best model saved  (IoU=0.5293)
     9 |      0.2055 |    0.2258 | 0.5288 | 0.6262 |  9.80e-05  [110s]
    10 |      0.2026 |    0.2227 | 0.5340 | 0.6301 |  9.76e-05  [110s]
  ✓ Best model saved  (IoU=0.5340)
    11 |  

## Cell 7: Stage 2 — Classification Training

In [51]:
# ─────────────────────────────────────────────────────────────────
# Classification Metrics
# ─────────────────────────────────────────────────────────────────

def compute_cls_metrics(all_preds, all_labels, num_classes=Config.CLS_NUM_CLASSES):
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    f1     = f1_score(all_labels, all_preds, average='macro',
                      labels=list(range(num_classes)), zero_division=0)
    cm     = confusion_matrix(all_labels, all_preds,
                              labels=list(range(num_classes)))
    acc    = (all_preds == all_labels).mean()
    return f1, acc, cm

print('✓ Classification metrics defined')

✓ Classification metrics defined


In [52]:
# ─────────────────────────────────────────────────────────────────
# Classification Training Helpers
# ─────────────────────────────────────────────────────────────────

def train_cls_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for pre, post, labels in loader:
        pre    = pre.to(device)
        post   = post.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            logits = model(pre, post)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds       = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    f1, acc, _ = compute_cls_metrics(all_preds, all_labels)
    return total_loss / len(loader), f1, acc


@torch.no_grad()
def validate_cls(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for pre, post, labels in loader:
        pre    = pre.to(device)
        post   = post.to(device)
        labels = labels.to(device)

        with torch.cuda.amp.autocast():
            logits = model(pre, post)
            loss   = criterion(logits, labels)

        total_loss += loss.item()
        preds       = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    f1, acc, cm = compute_cls_metrics(all_preds, all_labels)
    return total_loss / len(loader), f1, acc, cm

print('✓ Classification training helpers defined')

✓ Classification training helpers defined


In [53]:
# ─────────────────────────────────────────────────────────────────
# Main Classification Training Loop
# ─────────────────────────────────────────────────────────────────

def train_classification(patch_dir: str = Config.PATCH_DIR):
    device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Training on: {device}')

    # ── Dataset ──────────────────────────────────────────────────
    full_dataset = PatchClassificationDataset(
        patch_dir = patch_dir,
        transform = None,
    )

    n_total = len(full_dataset)
    n_val   = max(1, int(0.15 * n_total))
    n_train = n_total - n_val

    train_ds, val_ds = random_split(
        full_dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(Config.SEED)
    )

    train_ds.dataset.transform = get_cls_transforms(train=True)
    val_ds.dataset.transform   = get_cls_transforms(train=False)

    train_loader = DataLoader(train_ds, batch_size=Config.CLS_BATCH_SIZE,
                              shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=Config.CLS_BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

    # ── Class weights (imbalanced dataset) ───────────────────────
    class_weights = compute_class_weights(patch_dir).to(device)

    # ── Model ────────────────────────────────────────────────────
    model     = DualResNet50(num_classes=Config.CLS_NUM_CLASSES,
                             pretrained=True, dropout=0.4).to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.CLS_LR,
                                  weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.CLS_EPOCHS, eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler()

    writer    = SummaryWriter(
        log_dir=os.path.join(Config.LOG_DIR, 'classification'))

    best_f1      = 0.0
    best_ckpt    = os.path.join(Config.CHECKPOINT_DIR, 'cls_best.pth')
    patience     = 8
    patience_ctr = 0

    print(f'\n{"Epoch":>6} | {"T.Loss":>8} | {"T.F1":>6} | '
          f'{"V.Loss":>8} | {"V.F1":>6} | {"V.Acc":>6} | {"LR":>9}')
    print('-' * 72)

    for epoch in range(1, Config.CLS_EPOCHS + 1):
        t0 = time.time()

        t_loss, t_f1, t_acc = train_cls_one_epoch(
            model, train_loader, optimizer, criterion, device, scaler)
        v_loss, v_f1, v_acc, cm = validate_cls(
            model, val_loader, criterion, device)

        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        elapsed = time.time() - t0
        print(f'{epoch:>6} | {t_loss:>8.4f} | {t_f1:>6.4f} | '
              f'{v_loss:>8.4f} | {v_f1:>6.4f} | {v_acc:>6.4f} | '
              f'{lr:>9.2e}  [{elapsed:.0f}s]')

        writer.add_scalar('Loss/train',    t_loss, epoch)
        writer.add_scalar('Loss/val',      v_loss, epoch)
        writer.add_scalar('F1/train',      t_f1,   epoch)
        writer.add_scalar('F1/val',        v_f1,   epoch)
        writer.add_scalar('Accuracy/val',  v_acc,  epoch)

        if v_f1 > best_f1:
            best_f1      = v_f1
            patience_ctr = 0
            torch.save({
                'epoch':      epoch,
                'state_dict': model.state_dict(),
                'optimizer':  optimizer.state_dict(),
                'best_f1':    best_f1,
            }, best_ckpt)
            print(f'  ✓ Best model saved  (F1={best_f1:.4f})')
            print('  Confusion matrix:')
            for row in cm:
                print('   ', row.tolist())
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping triggered at epoch {epoch}')
                break

    writer.close()
    print(f'\n✓ Classification training complete. Best macro-F1: {best_f1:.4f}')
    print(f'  Checkpoint: {best_ckpt}')
    return best_ckpt

print('✓ Classification training function defined')

✓ Classification training function defined


### Run Classification Training

Uncomment and run the cell below to start Stage 2 training (requires patches first):

In [54]:
# ─────────────────────────────────────────────────────────────────
# Stage 1.5 + Stage 2  (run these together)
# Stage 1.5 must complete before Stage 2 or you will get a
# FileNotFoundError for outputs/patches/labels.json
# ─────────────────────────────────────────────────────────────────

# Step 1 — Extract building patches from training set
# Skip this block only if patches already exist in Config.PATCH_DIR
extract_all_patches(
    img_dir    = Config.TRAIN_IMG_DIR,
    lbl_dir    = Config.TRAIN_LBL_DIR,
    output_dir = Config.PATCH_DIR,
    use_gt_mask= True,   # set False to use seg-model predicted masks
)

# Step 2 — Train Stage 2 classifier on the extracted patches
cls_ckpt = train_classification()

Found 2799 pre-disaster images to process...


Extracting patches: 100%|██████████| 2799/2799 [14:31<00:00,  3.21it/s]


✓ Extracted 102659 labelled patches → ./outputs/patches
  class 0 (no-damage): 71372
  class 1 (minor-damage): 11618
  class 2 (major-damage): 11236
  class 3 (destroyed): 8433
Training on: cuda
PatchClassificationDataset: 102659 patch pairs
Train: 87261 | Val: 15398
Class counts : [71372 11618 11236  8433]
Class weights: [0.36  2.209 2.284 3.043]

 Epoch |   T.Loss |   T.F1 |   V.Loss |   V.F1 |  V.Acc |        LR
------------------------------------------------------------------------


/tmp/ipykernel_85300/3307190716.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()
/tmp/ipykernel_85300/33263858.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/tmp/ipykernel_85300/33263858.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please u

     1 |   0.8276 | 0.5979 |   0.6948 | 0.6603 | 0.7626 |  9.99e-05  [547s]
  ✓ Best model saved  (F1=0.6603)
  Confusion matrix:
    [8530, 1157, 523, 495]
    [318, 933, 352, 103]
    [144, 242, 1189, 140]
    [64, 52, 66, 1090]


...

    12 |   0.1339 | 0.9167 |   1.0599 | 0.6964 | 0.8054 |  8.66e-05  [532s]


/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/tmp/ipykernel_85300/33263858.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware t

    13 |   0.1167 | 0.9285 |   1.0833 | 0.6947 | 0.7946 |  8.44e-05  [541s]


/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/tmp/ipykernel_85300/33263858.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware t

    14 |   0.1030 | 0.9351 |   1.0700 | 0.6962 | 0.7943 |  8.21e-05  [545s]
Early stopping triggered at epoch 14

✓ Classification training complete. Best macro-F1: 0.7139
  Checkpoint: ./outputs/checkpoints/cls_best.pth


## Cell 8: Inference Pipeline

In [55]:
# ─────────────────────────────────────────────────────────────────
# Pre-processing helpers for inference
# ─────────────────────────────────────────────────────────────────

SEG_TRANSFORM = A.Compose([
    A.Resize(Config.SEG_IMG_SIZE, Config.SEG_IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

CLS_TRANSFORM = A.Compose([
    A.Resize(Config.CLS_PATCH_SIZE, Config.CLS_PATCH_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


def preprocess_for_seg(image_rgb: np.ndarray) -> torch.Tensor:
    """(H, W, 3) uint8 → (1, 3, 512, 512) float tensor."""
    aug = SEG_TRANSFORM(image=image_rgb)
    return aug['image'].unsqueeze(0)


def preprocess_patch(patch_rgb: np.ndarray) -> torch.Tensor:
    """(H, W, 3) uint8 → (1, 3, 224, 224) float tensor."""
    aug = CLS_TRANSFORM(image=patch_rgb)
    return aug['image'].unsqueeze(0)

print('✓ Preprocessing helpers defined')

✓ Preprocessing helpers defined


In [56]:
# ─────────────────────────────────────────────────────────────────
# Model loaders
# ─────────────────────────────────────────────────────────────────

def load_seg_model(ckpt_path: str, device: torch.device) -> torch.nn.Module:
    model = build_segmentation_model()
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    print(f'✓ Segmentation model loaded (IoU={ckpt.get("best_iou", "?"):.4f})')
    return model


def load_cls_model(ckpt_path: str, device: torch.device) -> torch.nn.Module:
    model = DualResNet50(num_classes=Config.CLS_NUM_CLASSES, pretrained=False)
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    print(f'✓ Classification model loaded (F1={ckpt.get("best_f1", "?"):.4f})')
    return model

print('✓ Model loaders defined')

✓ Model loaders defined


In [57]:
# ─────────────────────────────────────────────────────────────────
# Stage 1 — predict building mask
# ─────────────────────────────────────────────────────────────────

@torch.no_grad()
def predict_mask(seg_model: torch.nn.Module,
                 image_rgb: np.ndarray,
                 device: torch.device,
                 threshold: float = Config.SEG_THRESHOLD) -> np.ndarray:
    """
    Returns binary mask (H, W) uint8 {0, 255} at original resolution.
    """
    orig_h, orig_w = image_rgb.shape[:2]
    inp    = preprocess_for_seg(image_rgb).to(device)

    with torch.cuda.amp.autocast():
        logit  = seg_model(inp)              # (1, 1, 512, 512)

    prob   = torch.sigmoid(logit).squeeze().cpu().numpy()  # (512, 512)
    mask   = (prob > threshold).astype(np.uint8) * 255

    mask   = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    return mask

print('✓ Building mask prediction defined')

✓ Building mask prediction defined


In [58]:
# ─────────────────────────────────────────────────────────────────
# Stage 2 — classify each building patch
# ─────────────────────────────────────────────────────────────────

@torch.no_grad()
def classify_patches(cls_model: torch.nn.Module,
                     patches: list,
                     device: torch.device) -> list:
    """
    Args:
        patches: list of dicts from extract_patches_from_pair().
    Returns:
        List of predicted damage class ints (0-3) in same order.
    """
    predictions = []

    for p in patches:
        pre_t  = preprocess_patch(p['pre_patch']).to(device)
        post_t = preprocess_patch(p['post_patch']).to(device)

        with torch.cuda.amp.autocast():
            logit = cls_model(pre_t, post_t)   # (1, 4)

        pred = logit.argmax(dim=1).item()
        predictions.append(pred)

    return predictions

print('✓ Patch classification defined')

✓ Patch classification defined


In [59]:
# ─────────────────────────────────────────────────────────────────
# Visualization
# ─────────────────────────────────────────────────────────────────

def draw_predictions(post_image_rgb: np.ndarray,
                     patches: list,
                     predictions: list,
                     alpha: float = 0.5) -> np.ndarray:
    """
    Overlay coloured bounding boxes on the post-disaster image.
    Returns BGR image ready for cv2.imwrite.
    """
    vis = cv2.cvtColor(post_image_rgb, cv2.COLOR_RGB2BGR).copy()

    for p, pred in zip(patches, predictions):
        x, y, bw, bh = p['bbox']
        color = Config.DAMAGE_COLORS.get(pred, (128, 128, 128))
        label = Config.DAMAGE_CLASSES.get(pred, 'unknown')

        # Semi-transparent fill
        overlay = vis.copy()
        cv2.rectangle(overlay, (x, y), (x + bw, y + bh), color, -1)
        cv2.addWeighted(overlay, alpha, vis, 1 - alpha, 0, vis)

        # Solid border
        cv2.rectangle(vis, (x, y), (x + bw, y + bh), color, 2)

        # Label text
        cv2.putText(vis, label, (x, max(0, y - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1,
                    cv2.LINE_AA)

    return vis


def draw_legend(vis: np.ndarray) -> np.ndarray:
    """Add a damage class legend in the bottom-left corner."""
    legend_h = 30 * Config.CLS_NUM_CLASSES + 10
    legend_w = 180
    h, w     = vis.shape[:2]
    y0       = h - legend_h - 10
    x0       = 10

    # Semi-transparent background
    overlay = vis.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + legend_w, y0 + legend_h),
                  (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, vis, 0.4, 0, vis)

    for i, (cls_id, name) in enumerate(Config.DAMAGE_CLASSES.items()):
        color = Config.DAMAGE_COLORS[cls_id]
        cy    = y0 + 10 + i * 30
        cv2.rectangle(vis, (x0 + 5, cy), (x0 + 25, cy + 18), color, -1)
        cv2.putText(vis, name, (x0 + 32, cy + 13),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    return vis

print('✓ Visualization functions defined')

✓ Visualization functions defined


In [60]:
# ─────────────────────────────────────────────────────────────────
# End-to-end single image pair inference
# ─────────────────────────────────────────────────────────────────

def run_inference(pre_path: str,
                  post_path: str,
                  seg_model: torch.nn.Module,
                  cls_model: torch.nn.Module,
                  device: torch.device,
                  output_dir: str = Config.PRED_DIR) -> dict:
    """
    Full pipeline for one image pair.

    Returns:
        {
            'mask':        np.ndarray (H, W) uint8,
            'predictions': list[int],
            'patches':     list[dict],
            'vis_path':    str,
        }
    """
    pre_rgb  = cv2.cvtColor(cv2.imread(pre_path),  cv2.COLOR_BGR2RGB)
    post_rgb = cv2.cvtColor(cv2.imread(post_path), cv2.COLOR_BGR2RGB)

    # Stage 1 — building mask
    print('  [1/3] Predicting building mask...')
    mask = predict_mask(seg_model, pre_rgb, device)

    # Stage 2a — extract patches using predicted mask
    print('  [2/3] Extracting building patches...')
    patches = extract_patches_from_pair(
        pre_img_path  = pre_path,
        post_img_path = post_path,
        mask          = mask,
        post_json_path= None,
    )
    print(f'         {len(patches)} buildings detected')

    if len(patches) == 0:
        print('  WARNING: No buildings found. Check segmentation quality.')
        return {'mask': mask, 'predictions': [], 'patches': [], 'vis_path': None}

    # Stage 2b — classify each building
    print('  [3/3] Classifying damage...')
    predictions = classify_patches(cls_model, patches, device)

    # Tally
    dist = Counter(predictions)
    for cls_id, name in Config.DAMAGE_CLASSES.items():
        print(f'         {name}: {dist.get(cls_id, 0)} buildings')

    # Visualization
    vis = draw_predictions(post_rgb, patches, predictions)
    vis = draw_legend(vis)

    stem     = Path(pre_path).stem.replace('_pre_disaster', '')
    vis_path = os.path.join(output_dir, f'{stem}_damage_map.png')
    cv2.imwrite(vis_path, vis)

    mask_path = os.path.join(output_dir, f'{stem}_building_mask.png')
    cv2.imwrite(mask_path, mask)

    print(f'  ✓ Visualization saved → {vis_path}')

    return {
        'mask':        mask,
        'predictions': predictions,
        'patches':     patches,
        'vis_path':    vis_path,
    }

print('✓ Single pair inference defined')

✓ Single pair inference defined


In [61]:
# ─────────────────────────────────────────────────────────────────
# Batch inference over test set
# ─────────────────────────────────────────────────────────────────

def run_batch_inference(img_dir: str,
                        seg_ckpt: str,
                        cls_ckpt: str,
                        output_dir: str = Config.PRED_DIR):
    """Run inference on all pairs in img_dir."""
    device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')

    seg_model = load_seg_model(seg_ckpt, device)
    cls_model = load_cls_model(cls_ckpt, device)

    img_dir  = Path(img_dir)
    pre_imgs = sorted(img_dir.glob('*_pre_disaster.png'))
    print(f'\nRunning inference on {len(pre_imgs)} image pairs...\n')

    results = {}

    for pre_path in tqdm(pre_imgs, desc='Batch inference'):
        stem      = pre_path.stem
        post_name = stem.replace('_pre_', '_post_') + '.png'
        post_path = img_dir / post_name

        if not post_path.exists():
            continue

        print(f'Processing: {stem}')
        result = run_inference(str(pre_path), str(post_path),
                               seg_model, cls_model, device, output_dir)
        results[stem] = result['predictions']

    summary_path = os.path.join(output_dir, 'predictions_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(results, f)
    print(f'\n✓ Batch inference complete. Summary → {summary_path}')

print('✓ Batch inference defined')

✓ Batch inference defined


### Run Inference on a Single Image Pair

Uncomment and edit the paths below to run inference:

In [64]:
# Example: Run inference on a single image pair
import os
from pathlib import Path

device   = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
seg_ckpt = './outputs/checkpoints/seg_best.pth'
cls_ckpt = './outputs/checkpoints/cls_best.pth'

seg_model = load_seg_model(seg_ckpt, device)
cls_model = load_cls_model(cls_ckpt, device)

# ── Pick a real pair from your training set ──────────────────────
# (test set may only have post images based on your Cell 2 output,
#  so use a training pair instead)
train_img_dir = Path(Config.TRAIN_IMG_DIR)

# Auto-pick the first available pre/post pair
pre_path  = str(sorted(train_img_dir.glob('*_pre_disaster.png'))[0])
post_path = pre_path.replace('_pre_disaster', '_post_disaster')

print(f'pre  → {pre_path}')
print(f'post → {post_path}')
assert os.path.exists(pre_path),  f'pre image not found:  {pre_path}'
assert os.path.exists(post_path), f'post image not found: {post_path}'

result = run_inference(
    pre_path=pre_path,
    post_path=post_path,
    seg_model=seg_model,
    cls_model=cls_model,
    device=device,
)

✓ Segmentation model loaded (IoU=0.5506)
✓ Classification model loaded (F1=0.7139)
pre  → /home/devnurma/Satellite-based disaster damage/train/images/guatemala-volcano_00000000_pre_disaster.png
post → /home/devnurma/Satellite-based disaster damage/train/images/guatemala-volcano_00000000_post_disaster.png
  [1/3] Predicting building mask...


/tmp/ipykernel_85300/1548817967.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  [2/3] Extracting building patches...
         7 buildings detected
  [3/3] Classifying damage...
         no-damage: 2 buildings
         minor-damage: 4 buildings
         major-damage: 1 buildings
         destroyed: 0 buildings


/tmp/ipykernel_85300/335311234.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  ✓ Visualization saved → ./outputs/predictions/guatemala-volcano_00000000_damage_map.png


## Cell 9: Full Pipeline Runner

In [65]:
# ─────────────────────────────────────────────────────────────────
# Full Pipeline Execution
# ─────────────────────────────────────────────────────────────────

def run_full_pipeline(use_gt_mask: bool = True):
    """
    Run the entire pipeline:
    1. Train segmentation (Stage 1)
    2. Extract patches
    3. Train classification (Stage 2)
    4. Run batch inference on test set
    """
    print('\n' + '═' * 60)
    print('  🚀 FULL DISASTER DAMAGE ASSESSMENT PIPELINE')
    print('═' * 60)

    # Stage 1
    print('\n' + '═' * 60)
    print('  STAGE 1 — Segmentation (U-Net++ / EfficientNet-B3)')
    print('═' * 60)
    seg_ckpt = train_segmentation()

    # Patch extraction
    print('\n' + '═' * 60)
    print('  STAGE 1.5 — Patch Extraction')
    print('═' * 60)
    extract_all_patches(
        img_dir    = Config.TRAIN_IMG_DIR,
        lbl_dir    = Config.TRAIN_LBL_DIR,
        output_dir = Config.PATCH_DIR,
        use_gt_mask= use_gt_mask,
    )

    # Stage 2
    print('\n' + '═' * 60)
    print('  STAGE 2 — Classification (Dual ResNet50)')
    print('═' * 60)
    cls_ckpt = train_classification(patch_dir=Config.PATCH_DIR)

    # Pipeline complete
    print('\n' + '═' * 60)
    print('  ✓ PIPELINE COMPLETE')
    print('═' * 60)
    print(f'  Segmentation checkpoint : {seg_ckpt}')
    print(f'  Classification checkpoint: {cls_ckpt}')
    print(f'  Outputs                 : {Config.OUTPUT_DIR}')

    # Batch inference if test dir exists
    if os.path.isdir(Config.TEST_IMG_DIR):
        print('\n  Running inference on test set...')
        run_batch_inference(Config.TEST_IMG_DIR, seg_ckpt, cls_ckpt)

    return seg_ckpt, cls_ckpt

print('✓ Full pipeline runner defined')

✓ Full pipeline runner defined


### Execute Full Pipeline

Uncomment and run the cell below to execute the complete pipeline:

In [66]:
# ⚠️  MAIN EXECUTION — Uncomment to run full pipeline
seg_ckpt, cls_ckpt = run_full_pipeline(use_gt_mask=True)


════════════════════════════════════════════════════════════
  🚀 FULL DISASTER DAMAGE ASSESSMENT PIPELINE
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  STAGE 1 — Segmentation (U-Net++ / EfficientNet-B3)
════════════════════════════════════════════════════════════
Training on: cuda
SegmentationDataset: 2799 samples found
Train: 2380 | Val: 419


/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/tmp/ipykernel_85300/2561978118.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()



 Epoch |  Train Loss |  Val Loss |    IoU |   Dice |        LR
-----------------------------------------------------------------


/tmp/ipykernel_85300/1683020645.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_85300/1683020645.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


     1 |      0.6731 |    0.4748 | 0.3968 | 0.5150 |  1.00e-04  [108s]
  ✓ Best model saved  (IoU=0.3968)
     2 |      0.4202 |    0.3210 | 0.4655 | 0.5755 |  9.99e-05  [108s]
  ✓ Best model saved  (IoU=0.4655)
     3 |      0.3012 |    0.2523 | 0.4978 | 0.6039 |  9.98e-05  [108s]
  ✓ Best model saved  (IoU=0.4978)
     4 |      0.2555 |    0.2477 | 0.5019 | 0.6062 |  9.96e-05  [108s]
  ✓ Best model saved  (IoU=0.5019)
     5 |      0.2339 |    0.2349 | 0.5122 | 0.6141 |  9.94e-05  [108s]
  ✓ Best model saved  (IoU=0.5122)
     6 |      0.2271 |    0.2337 | 0.5137 | 0.6147 |  9.91e-05  [108s]
  ✓ Best model saved  (IoU=0.5137)
     7 |      0.2177 |    0.2291 | 0.5210 | 0.6201 |  9.88e-05  [108s]
  ✓ Best model saved  (IoU=0.5210)
     8 |      0.2115 |    0.2291 | 0.5230 | 0.6225 |  9.84e-05  [108s]
  ✓ Best model saved  (IoU=0.5230)
     9 |      0.2088 |    0.2277 | 0.5246 | 0.6233 |  9.80e-05  [108s]
  ✓ Best model saved  (IoU=0.5246)
    10 |      0.2066 |    0.2242 | 0.5299 | 0.

...

    12 |   0.1437 | 0.9096 |   0.9462 | 0.6928 | 0.7928 |  8.66e-05  [539s]


/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/tmp/ipykernel_85300/33263858.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware t

    13 |   0.1215 | 0.9245 |   1.0362 | 0.6895 | 0.7937 |  8.44e-05  [539s]
Early stopping triggered at epoch 13

✓ Classification training complete. Best macro-F1: 0.7092
  Checkpoint: ./outputs/checkpoints/cls_best.pth

════════════════════════════════════════════════════════════
  ✓ PIPELINE COMPLETE
════════════════════════════════════════════════════════════
  Segmentation checkpoint : ./outputs/checkpoints/seg_best.pth
  Classification checkpoint: ./outputs/checkpoints/cls_best.pth
  Outputs                 : ./outputs

  Running inference on test set...
✓ Segmentation model loaded (IoU=0.5495)
✓ Classification model loaded (F1=0.7092)

Running inference on 0 image pairs...



Batch inference: 0it [00:00, ?it/s]


✓ Batch inference complete. Summary → ./outputs/predictions/predictions_summary.json


## Cell 10: Summary and Next Steps

In [67]:
print("""
╔═══════════════════════════════════════════════════════════════════╗
║     DISASTER DAMAGE ASSESSMENT PIPELINE — NOTEBOOK SUMMARY        ║
╚═══════════════════════════════════════════════════════════════════╝

✓ All components have been defined and are ready to use:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE 1: SEGMENTATION (Building Detection)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Function:    train_segmentation()
  Model:       U-Net++ with EfficientNet-B3 backbone
  Input:       Pre-disaster satellite images (512×512)
  Output:      Binary building mask
  Loss:        Dice + BCE
  Metric:      IoU (Intersection over Union)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 STAGE 1.5: PATCH EXTRACTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Function:    extract_all_patches()
  Process:     Extract individual building patches from pre/post pairs
  Uses:        Building mask + damage labels from post-disaster JSON
  Output:      Organized patch directory with labels.json

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 STAGE 2: CLASSIFICATION (Damage Assessment)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Function:    train_classification()
  Model:       Siamese ResNet50 (dual-input)
  Input:       Pre + post building patches (224×224 each)
  Output:      Damage class (0-3):
               • 0: no-damage    (green)
               • 1: minor-damage (yellow)
               • 2: major-damage (orange)
               • 3: destroyed    (red)
  Loss:        CrossEntropyLoss with class weights
  Metric:      Macro-averaged F1 score

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 INFERENCE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Single pair:      run_inference(pre_path, post_path, seg_model, cls_model, device)
  Batch:            run_batch_inference(img_dir, seg_ckpt, cls_ckpt)
  Full pipeline:    run_full_pipeline(use_gt_mask=True)

  Output:           Damage maps with color-coded overlays + legend

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 QUICK START
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Edit Cell 2 (Config) → Update BASE_DIR to point to your xView2 dataset

2. Run entire notebook sequentially (Cells 1–9 define components)

3. Choose your workflow:

   Option A: Run full pipeline
   ─────────────────────────────
   • Uncomment the last line in Cell 9
   • Executes all stages automatically

   Option B: Run individual stages
   ─────────────────────────────────
   • Cell 6: train_segmentation()
   • Cell 5: extract_all_patches()
   • Cell 7: train_classification()
   • Cell 8: run_batch_inference() or run_inference()

   Option C: Run inference only
   ──────────────────────────────
   • Load pre-trained checkpoints
   • Uncomment inference example in Cell 8
   • Provide pre/post image paths

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 KEY PARAMETERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Segmentation
  • SEG_IMG_SIZE    = 512 (resize before processing)
  • SEG_EPOCHS      = 30
  • SEG_BATCH_SIZE  = 4
  • SEG_THRESHOLD   = 0.5 (mask probability cutoff)

  Classification
  • CLS_PATCH_SIZE  = 224
  • CLS_EPOCHS      = 25
  • CLS_BATCH_SIZE  = 16
  • CLS_NUM_CLASSES = 4

  Patch Extraction
  • SEG_MIN_AREA    = 100 (pixels, ignore smaller blobs)
  • CLS_PATCH_SIZE  = 224 (output patch size)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 EXPECTED DIRECTORY STRUCTURE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ~/Satellite-based disaster damage/
  ├── train/
  │   ├── images/
  │   │   ├── <disaster>_<id>_pre_disaster.png
  │   │   └── <disaster>_<id>_post_disaster.png
  │   └── labels/
  │       ├── <disaster>_<id>_pre_disaster.json
  │       └── <disaster>_<id>_post_disaster.json
  └── test/
      └── images/
          ├── <disaster>_<id>_pre_disaster.png
          └── <disaster>_<id>_post_disaster.png

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 OUTPUT STRUCTURE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  outputs/
  ├── checkpoints/
  │   ├── seg_best.pth
  │   └── cls_best.pth
  ├── logs/
  │   ├── segmentation/
  │   └── classification/
  ├── patches/
  │   ├── pre/     (pre-disaster patches)
  │   ├── post/    (post-disaster patches)
  │   └── labels.json
  ├── predictions/
  │   ├── <id>_damage_map.png
  │   ├── <id>_building_mask.png
  │   └── predictions_summary.json
  └── visualizations/

""")



╔═══════════════════════════════════════════════════════════════════╗
║     DISASTER DAMAGE ASSESSMENT PIPELINE — NOTEBOOK SUMMARY        ║
╚═══════════════════════════════════════════════════════════════════╝

✓ All components have been defined and are ready to use:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE 1: SEGMENTATION (Building Detection)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Function:    train_segmentation()
  Model:       U-Net++ with EfficientNet-B3 backbone
  Input:       Pre-disaster satellite images (512×512)
  Output:      Binary building mask
  Loss:        Dice + BCE
  Metric:      IoU (Intersection over Union)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 STAGE 1.5: PATCH EXTRACTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Function:    extract_all_patches()
  Process:     Extract individual building patches from pre/post pairs
  Uses:        Building mask + damag

### Improved the code

In [1]:
import os
import json
import cv2
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
from scipy.ndimage import distance_transform_edt
from sklearn.metrics import f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from torchvision import models

# ─────────────────────────────────────────────────────────────────
# GPU Check
# ─────────────────────────────────────────────────────────────────
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# ─────────────────────────────────────────────────────────────────
# Data paths
# ─────────────────────────────────────────────────────────────────
BASE_DIR = Path.home() / 'Satellite-based disaster damage'

print('BASE_DIR           :', BASE_DIR)
print('train/images exists:', (BASE_DIR / 'train' / 'images').exists())
print('train/labels exists:', (BASE_DIR / 'train' / 'labels').exists())
print('test/images  exists:', (BASE_DIR / 'test'  / 'images').exists())

train_img_dir = BASE_DIR / 'train' / 'images'
test_img_dir  = BASE_DIR / 'test'  / 'images'
train_files   = list(train_img_dir.glob('*')) if train_img_dir.exists() else []
test_files    = list(test_img_dir.glob('*'))  if test_img_dir.exists()  else []

print(f'\nTrain folder — {len(train_files)} total files')
print('  First 6:', [f.name for f in sorted(train_files)[:6]])
print(f'\nTest  folder — {len(test_files)} total files')
print('  First 6:', [f.name for f in sorted(test_files)[:6]])

assert BASE_DIR.exists(), f'DATA ROOT not found: {BASE_DIR}'
print('\n✓ All paths look good')


# ─────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────
class Config:
    DATA_ROOT      = BASE_DIR
    TRAIN_IMG_DIR  = str(DATA_ROOT / 'train' / 'images')
    TRAIN_LBL_DIR  = str(DATA_ROOT / 'train' / 'labels')
    TEST_IMG_DIR   = str(DATA_ROOT / 'test'  / 'images')
    OUTPUT_DIR     = './outputs'
    CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
    LOG_DIR        = os.path.join(OUTPUT_DIR, 'logs')
    VIZ_DIR        = os.path.join(OUTPUT_DIR, 'visualizations')
    PRED_DIR       = os.path.join(OUTPUT_DIR, 'predictions')
    PATCH_DIR      = os.path.join(OUTPUT_DIR, 'patches')

    # Segmentation
    SEG_IMG_SIZE      = 640
    SEG_CROP_SIZE     = 512
    SEG_BATCH_SIZE    = 4
    SEG_LR            = 3e-4
    SEG_EPOCHS        = 100
    SEG_THRESHOLD     = 0.5
    SEG_MIN_AREA      = 100
    SEG_IN_CHANNELS   = 6
    SEG_WARMUP_EPOCHS = 5
    SEG_FREEZE_EPOCHS = 5

    # FIX 8: validation fraction of disaster scenes (not samples)
    VAL_SCENE_FRACTION = 0.15

    # Classification
    CLS_PATCH_SIZES = [192, 224, 256]   # FIX 6: multi-scale patches
    CLS_PATCH_SIZE  = 224               # used at inference
    CLS_BATCH_SIZE  = 16
    CLS_LR          = 1e-4
    CLS_EPOCHS      = 50
    CLS_NUM_CLASSES = 4

    # FIX 7: hard-negative mining
    HNM_START_EPOCH = 5                 # start HNM after this epoch
    HNM_OVERSAMPLE  = 3                 # repeat misclassified samples N times

    DAMAGE_CLASSES = {0: 'no-damage', 1: 'minor-damage',
                      2: 'major-damage', 3: 'destroyed'}
    DAMAGE_COLORS  = {0: (0, 200, 0), 1: (0, 200, 255),
                      2: (0, 100, 255), 3: (0, 0, 255)}

    SEED   = 42
    DEVICE = 'cuda'


def make_dirs():
    for d in [Config.OUTPUT_DIR, Config.CHECKPOINT_DIR,
              Config.LOG_DIR, Config.VIZ_DIR, Config.PRED_DIR, Config.PATCH_DIR]:
        os.makedirs(d, exist_ok=True)
    print('✓ Output directories created')

make_dirs()


# ─────────────────────────────────────────────────────────────────
# Models
# ─────────────────────────────────────────────────────────────────
def build_segmentation_model(encoder_name='timm-efficientnet-b5',
                              encoder_weights='imagenet',
                              in_channels=Config.SEG_IN_CHANNELS,
                              num_classes=1) -> nn.Module:
    return smp.DeepLabV3Plus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,
    )

print('✓ Segmentation model builder defined')


# ─────────────────────────────────────────────────────────────────
# FIX 4: True Spatial Attention (not channel attention)
# ─────────────────────────────────────────────────────────────────
class DualResNet50(nn.Module):
    """
    Siamese ResNet50 with spatial attention on post-disaster features.
    FIX 4: spatial_attn uses (B,2048,7,7) → (B,1,7,7) attention map
            so the model focuses on *where* damage occurs, not just
            which channels are active.
    Fusion: cat(f_pre_pooled, f_post_attended_pooled, |diff|)
    """

    def __init__(self, num_classes=4, pretrained=True, dropout=0.4):
        super().__init__()
        weights  = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet50(weights=weights)
        self.encoder = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4,
        )  # → (B, 2048, 7, 7)

        # FIX 4: spatial attention — 2048 → 512 → 1 per spatial position
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 1, kernel_size=1),
            nn.Sigmoid(),
        )  # → (B, 1, 7, 7)

        self.pool = nn.AdaptiveAvgPool2d(1)

        feat_dim = 2048 * 3
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout / 2),
            nn.Linear(256, num_classes),
        )

    def encode(self, x):
        return self.encoder(x)   # (B, 2048, 7, 7)

    def forward(self, pre, post):
        f_pre  = self.encode(pre)
        f_post = self.encode(post)

        # FIX 4: spatial attention map → highlights changed regions
        attn   = self.spatial_attn(f_post)      # (B, 1, 7, 7)
        f_post = f_post * attn                   # broadcast over channels

        p_pre  = self.pool(f_pre).flatten(1)     # (B, 2048)
        p_post = self.pool(f_post).flatten(1)    # (B, 2048)
        diff   = torch.abs(p_post - p_pre)       # (B, 2048)

        return self.classifier(torch.cat([p_pre, p_post, diff], dim=1))

print('✓ DualResNet50 with spatial attention defined')


# ─────────────────────────────────────────────────────────────────
# Loss Functions
# ─────────────────────────────────────────────────────────────────
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, smooth=1e-6):
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth

    def forward(self, logits, targets):
        p  = torch.sigmoid(logits)
        TP = (p * targets).sum(dim=(2,3))
        FP = ((1-targets)*p).sum(dim=(2,3))
        FN = (targets*(1-p)).sum(dim=(2,3))
        t  = (TP+self.smooth) / (TP+self.alpha*FP+self.beta*FN+self.smooth)
        return (1 - t).mean()


# FIX 1: Symmetric distance-transform loss (pos + neg distance)
def compute_dist_map(mask_np: np.ndarray) -> np.ndarray:
    """
    FIX 1: symmetric distance map = distance to nearest boundary
    from BOTH foreground and background pixels.
    This gives strong gradients on both sides of every building edge.
    """
    pos = distance_transform_edt(mask_np)           # dist inside buildings
    neg = distance_transform_edt(1.0 - mask_np)    # dist outside buildings
    d   = pos + neg
    return d / (d.max() + 1e-6)                    # normalise to [0,1]


class DistanceTransformLoss(nn.Module):
    """
    FIX 1: symmetric boundary loss.
    weight = 1 + 5 × normalised_dist_map
    Pixels on building boundaries get the highest weight.
    """
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)

        dist_maps = []
        for t in targets.cpu().numpy():
            dist_maps.append(compute_dist_map(t[0]))
        dist_maps = torch.tensor(
            np.stack(dist_maps), dtype=torch.float32
        ).unsqueeze(1).to(logits.device)            # (B,1,H,W)

        return ((probs - targets)**2 * (1 + 5*dist_maps)).mean()


class CombinedSegLoss(nn.Module):
    """0.4·BCE + 0.3·Dice + 0.2·Tversky + 0.1·SymDistBoundary"""
    def __init__(self):
        super().__init__()
        self.bce       = nn.BCEWithLogitsLoss()
        self.tversky   = TverskyLoss(alpha=0.7, beta=0.3)
        self.dist_bdry = DistanceTransformLoss()

    def _dice(self, logits, targets):
        p      = torch.sigmoid(logits)
        smooth = 1e-6
        inter  = (p * targets).sum(dim=(2,3))
        return (1-(2*inter+smooth)/(p.sum(dim=(2,3))+targets.sum(dim=(2,3))+smooth)).mean()

    def forward(self, logits, targets):
        return (0.40 * self.bce(logits, targets) +
                0.30 * self._dice(logits, targets) +
                0.20 * self.tversky(logits, targets) +
                0.10 * self.dist_bdry(logits, targets))


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        p_t = torch.exp(-ce)
        return ((1-p_t)**self.gamma * ce).mean()

print('✓ Loss functions defined')


# ─────────────────────────────────────────────────────────────────
# Sanity check
# ─────────────────────────────────────────────────────────────────
device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

seg = build_segmentation_model()
out = seg(torch.randn(2, 6, 512, 512))
print(f'Seg model — Input: (2,6,512,512)  Output: {out.shape}')
del seg, out

cls = DualResNet50(num_classes=4)
out = cls(torch.randn(4,3,224,224), torch.randn(4,3,224,224))
print(f'Cls model — Output: {out.shape}')
del cls, out
print('✓ Both models initialised successfully')


# ─────────────────────────────────────────────────────────────────
# Mask helpers
# ─────────────────────────────────────────────────────────────────
def json_to_mask(json_path, height=1024, width=1024):
    with open(json_path) as f:
        data = json.load(f)
    mask = np.zeros((height, width), dtype=np.uint8)
    for feat in data.get('features', {}).get('xy', []):
        geom = feat.get('wkt', '')
        if not geom.startswith('POLYGON'):
            continue
        coords = geom.replace('POLYGON ((','').replace('))','').strip()
        pts = []
        for pair in coords.split(','):
            pair = pair.strip()
            if pair:
                x, y = pair.split()
                pts.append([float(x), float(y)])
        if len(pts) >= 3:
            cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], 255)
    return mask


def json_to_damage_mask(json_path, height=1024, width=1024):
    subtype_map = {'no-damage':1,'minor-damage':2,
                   'major-damage':3,'destroyed':4,'un-classified':0}
    with open(json_path) as f:
        data = json.load(f)
    mask = np.zeros((height, width), dtype=np.uint8)
    for feat in data.get('features', {}).get('xy', []):
        geom  = feat.get('wkt', '')
        label = subtype_map.get(
            feat.get('properties', {}).get('subtype','un-classified'), 0)
        if not geom.startswith('POLYGON') or label == 0:
            continue
        coords = geom.replace('POLYGON ((','').replace('))','').strip()
        pts = []
        for pair in coords.split(','):
            pair = pair.strip()
            if pair:
                x, y = pair.split()
                pts.append([float(x), float(y)])
        if len(pts) >= 3:
            cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], label)
    return mask

print('✓ Mask helpers defined')


# ─────────────────────────────────────────────────────────────────
# FIX 2: Split spatial / photometric transforms
# Spatial = ReplayCompose (identical for pre+post)
# Photometric = Compose (independent for pre+post)
# ─────────────────────────────────────────────────────────────────
def get_spatial_transforms(train: bool = True):
    """
    FIX 2: Only spatial ops here so they can be replayed identically
    on both pre and post images.  No photometric ops.
    """
    if train:
        return A.ReplayCompose([
            A.Resize(Config.SEG_IMG_SIZE, Config.SEG_IMG_SIZE),
            A.RandomCrop(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
        ])
    else:
        return A.ReplayCompose([
            A.Resize(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
        ])


def get_photometric_transforms():
    """
    FIX 2: Applied independently to pre and post so each image can
    have different lighting / noise — matches real disaster imagery.
    """
    return A.Compose([
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
            A.CLAHE(clip_limit=4.0),
            A.RandomGamma(gamma_limit=(70, 130)),
        ], p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50)),
            A.MotionBlur(blur_limit=5),
            A.MedianBlur(blur_limit=5),
        ], p=0.3),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                        fill_value=0, p=0.3),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])


_val_norm = A.Compose([
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

print('✓ Split spatial / photometric transforms defined')


# ─────────────────────────────────────────────────────────────────
# FIX 8: Disaster-scene-aware train/val split
# ─────────────────────────────────────────────────────────────────
def disaster_scene_split(img_dir: str, val_fraction: float = 0.15, seed: int = 42):
    """
    FIX 8: extract the disaster-event prefix from each filename
    (e.g. 'guatemala-volcano', 'mexico-earthquake') and split at
    the scene level so no scene leaks across train/val.
    """
    img_dir = Path(img_dir)
    pre_files = sorted(img_dir.glob('*_pre_disaster.png'))

    scenes = defaultdict(list)
    for p in pre_files:
        # filename format: <disaster>_<tile>_pre_disaster.png
        # split on the LAST occurrence of '_' before 'pre'
        parts = p.stem.split('_pre_disaster')[0]
        # scene = everything except the trailing tile id (last token after _)
        tokens = parts.rsplit('_', 1)
        scene  = tokens[0] if len(tokens) > 1 else parts
        scenes[scene].append(str(p))

    scene_names = sorted(scenes.keys())
    rng = random.Random(seed)
    rng.shuffle(scene_names)

    n_val      = max(1, int(len(scene_names) * val_fraction))
    val_scenes = set(scene_names[:n_val])
    trn_scenes = set(scene_names[n_val:])

    train_files = [f for s in trn_scenes for f in scenes[s]]
    val_files   = [f for s in val_scenes  for f in scenes[s]]

    print(f'Scene split — '
          f'{len(trn_scenes)} train scenes ({len(train_files)} samples) | '
          f'{len(val_scenes)} val scenes ({len(val_files)} samples)')
    return sorted(train_files), sorted(val_files)

print('✓ Disaster-scene split defined')


# ─────────────────────────────────────────────────────────────────
# Segmentation Dataset (FIX 2 + FIX 8)
# ─────────────────────────────────────────────────────────────────
class SegmentationDataset(Dataset):
    """
    FIX 2: spatial transforms are replayed identically on pre+post;
            photometric transforms are applied independently.
    FIX 8: receives an explicit list of pre-image paths (from scene split).
    """

    def __init__(self, pre_paths: list, lbl_dir: str,
                 spatial_tf=None, photo_tf=None, is_train: bool = True):
        self.lbl_dir    = Path(lbl_dir)
        self.spatial_tf = spatial_tf
        self.photo_tf   = photo_tf
        self.is_train   = is_train

        self.samples = []
        for pre_path in pre_paths:
            pre_path  = Path(pre_path)
            stem      = pre_path.stem
            post_name = stem.replace('_pre_', '_post_') + '.png'
            post_path = pre_path.parent / post_name
            json_path = self.lbl_dir / f'{stem}.json'
            if post_path.exists() and json_path.exists():
                self.samples.append(
                    (str(pre_path), str(post_path), str(json_path)))

        print(f'SegDataset ({"train" if is_train else "val"}): '
              f'{len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pre_path, post_path, json_path = self.samples[idx]

        pre  = cv2.cvtColor(cv2.imread(pre_path),  cv2.COLOR_BGR2RGB)
        post = cv2.cvtColor(cv2.imread(post_path), cv2.COLOR_BGR2RGB)
        h, w = pre.shape[:2]

        mask = json_to_mask(json_path, height=h, width=w)
        mask = (mask > 0).astype(np.float32)

        if self.spatial_tf:
            # FIX 2 step A: same spatial ops (crop/flip/rotate) on both images
            aug_pre  = self.spatial_tf(image=pre, mask=mask)
            pre_sp   = aug_pre['image']
            mask     = aug_pre['mask']
            post_sp  = A.ReplayCompose.replay(
                aug_pre['replay'], image=post)['image']
        else:
            pre_sp, post_sp = pre, post

        if self.photo_tf and self.is_train:
            # FIX 2 step B: independent photometric ops (different lighting ok)
            pre_t  = self.photo_tf(image=pre_sp)['image']
            post_t = self.photo_tf(image=post_sp)['image']
        else:
            pre_t  = _val_norm(image=pre_sp)['image']
            post_t = _val_norm(image=post_sp)['image']

        mask_t = torch.from_numpy(mask).unsqueeze(0)
        return torch.cat([pre_t, post_t], dim=0), mask_t

print('✓ SegmentationDataset (FIX 2+8) defined')


# ─────────────────────────────────────────────────────────────────
# Classification transforms
# ─────────────────────────────────────────────────────────────────
def get_cls_transforms(train=True):
    norm = A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
    if train:
        return A.Compose([
            A.Resize(Config.CLS_PATCH_SIZE, Config.CLS_PATCH_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.3),
            A.OneOf([A.RandomBrightnessContrast(0.2,0.2),
                     A.CLAHE(), A.RandomGamma()], p=0.4),
            A.OneOf([A.GaussNoise(var_limit=(5,30)),
                     A.MotionBlur(blur_limit=3)], p=0.2),
            norm, ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(Config.CLS_PATCH_SIZE, Config.CLS_PATCH_SIZE),
        norm, ToTensorV2(),
    ])


# ─────────────────────────────────────────────────────────────────
# FIX 6: Multi-scale patch dataset
# FIX 7: Hard-negative mining support
# ─────────────────────────────────────────────────────────────────
class PatchClassificationDataset(Dataset):
    """
    FIX 6: at training time each patch is randomly resized to one of
            CLS_PATCH_SIZES before the standard augmentation pipeline.
    FIX 7: `hard_neg_ids` is a set of patch IDs that will be kept in
            the dataset for this epoch (updated after each epoch).
    """

    def __init__(self, patch_dir: str, transform=None,
                 hard_neg_ids: set = None, multiscale: bool = False):
        self.patch_dir    = Path(patch_dir)
        self.pre_dir      = self.patch_dir / 'pre'
        self.post_dir     = self.patch_dir / 'post'
        self.transform    = transform
        self.multiscale   = multiscale
        self.hard_neg_ids = hard_neg_ids or set()

        labels_file = self.patch_dir / 'labels.json'
        if not labels_file.exists():
            raise FileNotFoundError(
                f"labels.json not found at '{labels_file}'. "
                "Run extract_all_patches() first.")

        with open(labels_file) as f:
            self.labels = json.load(f)

        self.ids = sorted(self.labels.keys())
        print(f'PatchDataset: {len(self.ids)} patches | '
              f'{len(self.hard_neg_ids)} hard negatives')

    def __len__(self):
        return len(self.ids)

    def get_labels(self):
        return [int(self.labels[i]) for i in self.ids]

    def _load(self, patch_id: str):
        pre  = cv2.cvtColor(cv2.imread(str(self.pre_dir  / f'{patch_id}.png')),
                            cv2.COLOR_BGR2RGB)
        post = cv2.cvtColor(cv2.imread(str(self.post_dir / f'{patch_id}.png')),
                            cv2.COLOR_BGR2RGB)
        return pre, post

    def __getitem__(self, idx):
        patch_id = self.ids[idx]
        label    = int(self.labels[patch_id])
        pre, post = self._load(patch_id)

        # FIX 6: random resize to one of the multi-scale sizes
        if self.multiscale:
            s   = random.choice(Config.CLS_PATCH_SIZES)
            pre  = cv2.resize(pre,  (s, s))
            post = cv2.resize(post, (s, s))

        if self.transform:
            pre  = self.transform(image=pre)['image']
            post = self.transform(image=post)['image']
        else:
            pre  = torch.from_numpy(pre.transpose(2,0,1)).float()  / 255.
            post = torch.from_numpy(post.transpose(2,0,1)).float() / 255.

        return pre, post, torch.tensor(label, dtype=torch.long), patch_id

print('✓ PatchClassificationDataset (FIX 6+7) defined')


# ─────────────────────────────────────────────────────────────────
# Class weights & sampler
# ─────────────────────────────────────────────────────────────────
def compute_class_weights(patch_dir: str,
                          num_classes: int = Config.CLS_NUM_CLASSES) -> torch.Tensor:
    with open(Path(patch_dir) / 'labels.json') as f:
        labels = json.load(f)
    counts  = np.zeros(num_classes, dtype=np.float32)
    for v in labels.values():
        counts[int(v)] += 1
    counts  = np.where(counts == 0, 1, counts)
    weights = counts.sum() / (num_classes * counts)
    print('Class counts :', counts.astype(int))
    print('Class weights:', np.round(weights, 3))
    return torch.tensor(weights, dtype=torch.float32)


def build_weighted_sampler(dataset: PatchClassificationDataset,
                           extra_ids: set = None,
                           num_classes: int = Config.CLS_NUM_CLASSES):
    """
    FIX 7: `extra_ids` = hard-negative patch IDs that receive boosted weight.
    Their sampling probability is multiplied by HNM_OVERSAMPLE.
    """
    all_labels = dataset.get_labels()
    counts     = np.bincount(all_labels, minlength=num_classes).astype(float)
    counts     = np.where(counts == 0, 1, counts)
    cls_weight = 1.0 / counts

    sample_w = []
    for pid, lbl in zip(dataset.ids, all_labels):
        w = cls_weight[lbl]
        if extra_ids and pid in extra_ids:
            w *= Config.HNM_OVERSAMPLE   # FIX 7: boost hard negatives
        sample_w.append(w)

    return WeightedRandomSampler(sample_w, num_samples=len(sample_w),
                                 replacement=True)

print('✓ Class weight + sampler (FIX 7) defined')


# ─────────────────────────────────────────────────────────────────
# Segmentation metrics
# ─────────────────────────────────────────────────────────────────
def iou_score(pred, gt, thr=0.5):
    pred  = (pred > thr).astype(np.uint8)
    gt    = (gt   > thr).astype(np.uint8)
    inter = (pred & gt).sum()
    return inter / ((pred | gt).sum() + 1e-6)


def dice_score(pred, gt, thr=0.5):
    pred  = (pred > thr).astype(np.uint8)
    gt    = (gt   > thr).astype(np.uint8)
    inter = (pred & gt).sum()
    return (2*inter) / (pred.sum() + gt.sum() + 1e-6)

print('✓ Segmentation metrics defined')


# ─────────────────────────────────────────────────────────────────
# FIX 3: Watershed instance separation
# ─────────────────────────────────────────────────────────────────
def separate_instances_watershed(mask: np.ndarray) -> np.ndarray:
    """
    FIX 3: watershed-based instance separation for dense urban areas.
    Uses distance transform peaks as seeds, removes boundary pixels
    that cv2.watershed marks as -1, then rebuilds a binary mask.
    """
    binary = (mask > 0).astype(np.uint8)

    dist   = cv2.distanceTransform(binary * 255, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
    sure_fg    = np.uint8(sure_fg)

    # Unknown region (between sure foreground and dilated foreground)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    sure_bg = cv2.dilate(binary * 255, kernel, iterations=2)
    unknown = cv2.subtract(sure_bg, sure_fg)

    _, markers = cv2.connectedComponents(sure_fg)
    markers    = markers + 1              # background = 1, not 0
    markers[unknown == 255] = 0           # unknown region

    img_bgr = cv2.cvtColor(binary * 255, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(img_bgr, markers)

    # Rebuild binary: every marker > 1 is a building instance
    result = np.zeros_like(binary, dtype=np.uint8)
    result[markers > 1] = 255
    return result


def postprocess_mask(mask: np.ndarray,
                     min_area: int = Config.SEG_MIN_AREA,
                     use_watershed: bool = True) -> np.ndarray:
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    # FIX 3: watershed separation before connected-component filtering
    if use_watershed and cleaned.max() > 0:
        cleaned = separate_instances_watershed(cleaned)

    num_cc, labels_cc, stats, _ = cv2.connectedComponentsWithStats(cleaned)
    out = np.zeros_like(cleaned)
    for c in range(1, num_cc):
        if stats[c, cv2.CC_STAT_AREA] >= min_area:
            out[labels_cc == c] = 255
    return out

print('✓ Watershed instance separation (FIX 3) defined')


# ─────────────────────────────────────────────────────────────────
# WarmupCosineScheduler
# ─────────────────────────────────────────────────────────────────
class WarmupCosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, total_epochs,
                 min_lr=1e-6, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.min_lr        = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        ep = self.last_epoch + 1
        if ep <= self.warmup_epochs:
            scale = ep / max(1, self.warmup_epochs)
        else:
            prog  = (ep - self.warmup_epochs) / max(
                1, self.total_epochs - self.warmup_epochs)
            scale = 0.5 * (1 + np.cos(np.pi * prog))
        return [self.min_lr + (b - self.min_lr) * scale
                for b in self.base_lrs]

print('✓ WarmupCosineScheduler defined')


# ─────────────────────────────────────────────────────────────────
# FIX 5: Stronger TTA (5 augmentations: id + hflip + vflip + r90 + r180)
# ─────────────────────────────────────────────────────────────────
def _tta_variants(pre_rgb, post_rgb):
    """
    FIX 5: 5-way TTA covering the full set of axis-aligned symmetries.
    Returns list of (pre, post, undo_fn) tuples.
    undo_fn reverses the spatial transform on the probability map.
    """
    identity = lambda x: x

    variants = [
        # (pre_aug, post_aug, undo_fn for probability map)
        (pre_rgb, post_rgb, identity),
        (pre_rgb[:, ::-1], post_rgb[:, ::-1], lambda x: x[:, ::-1]),
        (pre_rgb[::-1, :], post_rgb[::-1, :], lambda x: x[::-1, :]),
        (np.rot90(pre_rgb, 1), np.rot90(post_rgb, 1), lambda x: np.rot90(x, -1)),
        (np.rot90(pre_rgb, 2), np.rot90(post_rgb, 2), lambda x: np.rot90(x, -2)),
    ]
    return variants


@torch.no_grad()
def predict_mask_tta(seg_model, pre_rgb, post_rgb, device,
                     threshold=Config.SEG_THRESHOLD):
    orig_h, orig_w = pre_rgb.shape[:2]
    seg_tf = A.Compose([
        A.Resize(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

    def _prep(pre, post):
        p = seg_tf(image=np.ascontiguousarray(pre))['image']
        q = seg_tf(image=np.ascontiguousarray(post))['image']
        return torch.cat([p, q], dim=0).unsqueeze(0).to(device)

    prob_sum = None
    for pre_a, post_a, undo in _tta_variants(pre_rgb, post_rgb):
        inp  = _prep(pre_a, post_a)
        with torch.amp.autocast('cuda'):
            logit = seg_model(inp)
        prob = torch.sigmoid(logit).squeeze().cpu().numpy()
        prob = undo(prob)                             # realign to original frame
        prob_sum = prob if prob_sum is None else prob_sum + prob

    avg_prob = prob_sum / 5
    mask     = (avg_prob > threshold).astype(np.uint8) * 255
    mask     = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    return postprocess_mask(mask)

print('✓ 5-way TTA (FIX 5) defined')


# ─────────────────────────────────────────────────────────────────
# Segmentation training helpers
# ─────────────────────────────────────────────────────────────────
def train_seg_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def validate_seg(model, loader, criterion, device):
    model.eval()
    total, all_iou, all_dice = 0.0, [], []
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            total += criterion(logits, masks).item()
        probs = torch.sigmoid(logits).cpu().numpy()
        gt    = masks.cpu().numpy()
        for p, g in zip(probs, gt):
            all_iou.append(iou_score(p[0], g[0]))
            all_dice.append(dice_score(p[0], g[0]))
    return total/len(loader), float(np.mean(all_iou)), float(np.mean(all_dice))

print('✓ Segmentation training helpers defined')


# ─────────────────────────────────────────────────────────────────
# Main Segmentation Training Loop (FIX 8: scene-aware split)
# ─────────────────────────────────────────────────────────────────
def train_segmentation():
    device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Training on: {device}')

    # FIX 8: split by disaster scene, not randomly
    train_paths, val_paths = disaster_scene_split(
        Config.TRAIN_IMG_DIR,
        val_fraction=Config.VAL_SCENE_FRACTION,
        seed=Config.SEED)

    spatial_train = get_spatial_transforms(train=True)
    spatial_val   = get_spatial_transforms(train=False)
    photo         = get_photometric_transforms()

    train_ds = SegmentationDataset(train_paths, Config.TRAIN_LBL_DIR,
                                   spatial_tf=spatial_train,
                                   photo_tf=photo, is_train=True)
    val_ds   = SegmentationDataset(val_paths,   Config.TRAIN_LBL_DIR,
                                   spatial_tf=spatial_val,
                                   photo_tf=None, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=Config.SEG_BATCH_SIZE,
                              shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=Config.SEG_BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    model     = build_segmentation_model().to(device)
    criterion = CombinedSegLoss()
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=Config.SEG_LR, weight_decay=1e-4)
    scheduler = WarmupCosineScheduler(optimizer,
                                      warmup_epochs=Config.SEG_WARMUP_EPOCHS,
                                      total_epochs=Config.SEG_EPOCHS,
                                      min_lr=1e-6)
    scaler    = torch.amp.GradScaler('cuda')
    writer    = SummaryWriter(log_dir=os.path.join(Config.LOG_DIR, 'segmentation'))

    best_iou, best_ckpt   = 0.0, os.path.join(Config.CHECKPOINT_DIR, 'seg_best.pth')
    patience, patience_ctr = 10, 0

    print(f'\n{"Epoch":>6} | {"TLoss":>9} | {"VLoss":>9} | '
          f'{"IoU":>6} | {"Dice":>6} | {"LR":>9} | Enc')
    print('─' * 68)

    for epoch in range(1, Config.SEG_EPOCHS + 1):
        t0 = time.time()

        if epoch == 1:
            for p in model.encoder.parameters(): p.requires_grad = False
            print('  [FIX 8] Encoder frozen')
        elif epoch == Config.SEG_FREEZE_EPOCHS + 1:
            for p in model.encoder.parameters(): p.requires_grad = True
            print('  [FIX 8] Encoder unfrozen')

        enc = 'frz' if epoch <= Config.SEG_FREEZE_EPOCHS else 'act'
        tl  = train_seg_one_epoch(model, train_loader, optimizer,
                                  criterion, device, scaler)
        vl, iou, dice = validate_seg(model, val_loader, criterion, device)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        print(f'{epoch:>6} | {tl:>9.4f} | {vl:>9.4f} | '
              f'{iou:>6.4f} | {dice:>6.4f} | {lr:>9.2e} | {enc}  '
              f'[{time.time()-t0:.0f}s]')

        writer.add_scalars('Loss',   {'train': tl,   'val': vl},   epoch)
        writer.add_scalars('Metric', {'IoU': iou, 'Dice': dice},   epoch)

        if iou > best_iou:
            best_iou, patience_ctr = iou, 0
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'best_iou': best_iou}, best_ckpt)
            print(f'  ✓ Saved (IoU={best_iou:.4f})')
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    writer.close()
    print(f'\n✓ Seg done. Best IoU: {best_iou:.4f}  →  {best_ckpt}')
    return best_ckpt

print('✓ Segmentation training defined')


# ─────────────────────────────────────────────────────────────────
# Classification metrics
# ─────────────────────────────────────────────────────────────────
def compute_cls_metrics(preds, labels, num_classes=Config.CLS_NUM_CLASSES):
    preds, labels = np.array(preds), np.array(labels)
    f1  = f1_score(labels, preds, average='macro',
                   labels=list(range(num_classes)), zero_division=0)
    cm  = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    acc = (preds == labels).mean()
    return f1, acc, cm


# ─────────────────────────────────────────────────────────────────
# Classification training helpers (FIX 7: hard-negative tracking)
# ─────────────────────────────────────────────────────────────────
def train_cls_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total, all_preds, all_labels = 0.0, [], []
    for pre, post, labels, _ in loader:
        pre, post, labels = pre.to(device), post.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(pre, post)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        all_preds.extend(logits.argmax(1).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1, acc, _ = compute_cls_metrics(all_preds, all_labels)
    return total / len(loader), f1, acc


@torch.no_grad()
def validate_cls(model, loader, criterion, device):
    """
    FIX 7: also returns dict {patch_id: bool} of wrong predictions
    so the main loop can identify hard negatives.
    """
    model.eval()
    total, all_preds, all_labels = 0.0, [], []
    wrong_ids = set()

    for pre, post, labels, patch_ids in loader:
        pre, post, labels = pre.to(device), post.to(device), labels.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(pre, post)
            total += criterion(logits, labels).item()
        preds = logits.argmax(1).cpu().numpy()
        gts   = labels.cpu().numpy()

        for pid, pred, gt in zip(patch_ids, preds, gts):
            if pred != gt:
                wrong_ids.add(pid)
        all_preds.extend(preds)
        all_labels.extend(gts)

    f1, acc, cm = compute_cls_metrics(all_preds, all_labels)
    return total / len(loader), f1, acc, cm, wrong_ids

print('✓ Classification training helpers (FIX 7) defined')


# ─────────────────────────────────────────────────────────────────
# Main Classification Training Loop
# FIX 7: Hard-negative mining, FIX 6: multi-scale patches
# ─────────────────────────────────────────────────────────────────
def train_classification(patch_dir: str = Config.PATCH_DIR):
    device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Training on: {device}')

    full_dataset = PatchClassificationDataset(
        patch_dir=patch_dir, multiscale=True)   # FIX 6

    n_total = len(full_dataset)
    n_val   = max(1, int(0.15 * n_total))
    n_train = n_total - n_val

    # FIX 8: simple index split is fine here since scene split
    #        already handled the seg model; patch split by scene
    #        would be ideal but requires patch_id → scene mapping.
    rng         = torch.Generator().manual_seed(Config.SEED)
    indices     = torch.randperm(n_total, generator=rng).tolist()
    train_ids   = indices[:n_train]
    val_ids     = indices[n_train:]

    # Wrap in thin Subset views so transforms can differ
    train_full  = PatchClassificationDataset(
        patch_dir=patch_dir,
        transform=get_cls_transforms(train=True),
        multiscale=True)

    val_full    = PatchClassificationDataset(
        patch_dir=patch_dir,
        transform=get_cls_transforms(train=False),
        multiscale=False)

    train_subset = Subset(train_full, train_ids)
    val_subset   = Subset(val_full,   val_ids)

    class_weights = compute_class_weights(patch_dir).to(device)
    model     = DualResNet50(num_classes=Config.CLS_NUM_CLASSES,
                             pretrained=True, dropout=0.4).to(device)
    criterion = FocalLoss(gamma=2.0, weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=Config.CLS_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.CLS_EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda')
    writer    = SummaryWriter(log_dir=os.path.join(Config.LOG_DIR,'classification'))

    best_f1, best_ckpt   = 0.0, os.path.join(Config.CHECKPOINT_DIR, 'cls_best.pth')
    patience, patience_ctr = 10, 0
    hard_neg_ids: set = set()       # FIX 7: grows after each validation

    print(f'\n{"Epoch":>6} | {"T.Loss":>8} | {"T.F1":>6} | '
          f'{"V.Loss":>8} | {"V.F1":>6} | {"V.Acc":>6} | HN')
    print('─' * 72)

    for epoch in range(1, Config.CLS_EPOCHS + 1):
        t0 = time.time()

        # FIX 7: rebuild sampler each epoch with latest hard negatives
        sampler = build_weighted_sampler(train_full,
                                         extra_ids=hard_neg_ids
                                         if epoch >= Config.HNM_START_EPOCH
                                         else set())
        # Re-wrap Subset so sampler sees correct indices
        train_loader = DataLoader(train_subset, batch_size=Config.CLS_BATCH_SIZE,
                                  sampler=sampler, num_workers=4, pin_memory=True,
                                  collate_fn=_collate_with_ids)
        val_loader   = DataLoader(val_subset,   batch_size=Config.CLS_BATCH_SIZE,
                                  shuffle=False, num_workers=4, pin_memory=True,
                                  collate_fn=_collate_with_ids)

        t_loss, t_f1, t_acc = train_cls_one_epoch(
            model, train_loader, optimizer, criterion, device, scaler)
        v_loss, v_f1, v_acc, cm, new_wrong = validate_cls(
            model, val_loader, criterion, device)

        # FIX 7: update hard negative set
        if epoch >= Config.HNM_START_EPOCH:
            hard_neg_ids = new_wrong
            print(f'  [HNM] {len(hard_neg_ids)} hard negatives identified')

        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        hn = len(hard_neg_ids)
        print(f'{epoch:>6} | {t_loss:>8.4f} | {t_f1:>6.4f} | '
              f'{v_loss:>8.4f} | {v_f1:>6.4f} | {v_acc:>6.4f} | {hn}  '
              f'[{time.time()-t0:.0f}s]')

        writer.add_scalars('Loss', {'train': t_loss, 'val': v_loss}, epoch)
        writer.add_scalars('F1',   {'train': t_f1,   'val': v_f1},   epoch)
        writer.add_scalar('HardNegatives', hn, epoch)

        if v_f1 > best_f1:
            best_f1, patience_ctr = v_f1, 0
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'best_f1': best_f1}, best_ckpt)
            print(f'  ✓ Saved (F1={best_f1:.4f})')
            print('  CM:', [row.tolist() for row in cm])
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    writer.close()
    print(f'\n✓ Cls done. Best F1: {best_f1:.4f}  →  {best_ckpt}')
    return best_ckpt


def _collate_with_ids(batch):
    """Custom collate that keeps patch_id strings alongside tensors."""
    pre    = torch.stack([b[0] for b in batch])
    post   = torch.stack([b[1] for b in batch])
    labels = torch.stack([b[2] for b in batch])
    ids    = [b[3] for b in batch]
    return pre, post, labels, ids

print('✓ Classification training defined')


# ─────────────────────────────────────────────────────────────────
# Patch extraction  (uses watershed postprocessing)
# ─────────────────────────────────────────────────────────────────
def extract_patches_from_pair(pre_img_path, post_img_path, mask,
                               post_json_path=None,
                               patch_size=Config.CLS_PATCH_SIZE,
                               min_area=Config.SEG_MIN_AREA,
                               min_confident_ratio=0.50):
    pre  = cv2.cvtColor(cv2.imread(pre_img_path),  cv2.COLOR_BGR2RGB)
    post = cv2.cvtColor(cv2.imread(post_img_path), cv2.COLOR_BGR2RGB)
    h, w = pre.shape[:2]

    if mask.shape != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    # FIX 3: watershed separation
    mask = postprocess_mask(mask, min_area=min_area, use_watershed=True)

    damage_mask = (json_to_damage_mask(post_json_path, h, w)
                   if post_json_path and Path(post_json_path).exists()
                   else np.full((h, w), 255, dtype=np.uint8))

    binary  = (mask > 0).astype(np.uint8) * 255
    num_cc, labels_cc, stats, _ = cv2.connectedComponentsWithStats(binary)

    patches = []
    for comp_id in range(1, num_cc):
        area = stats[comp_id, cv2.CC_STAT_AREA]
        if area < min_area:
            continue

        x, y   = stats[comp_id, cv2.CC_STAT_LEFT], stats[comp_id, cv2.CC_STAT_TOP]
        bw, bh = stats[comp_id, cv2.CC_STAT_WIDTH], stats[comp_id, cv2.CC_STAT_HEIGHT]
        m = 10
        x1,y1 = max(0,x-m),  max(0,y-m)
        x2,y2 = min(w,x+bw+m), min(h,y+bh+m)

        pre_crop  = pre[y1:y2, x1:x2]
        post_crop = post[y1:y2, x1:x2]
        if pre_crop.size == 0 or post_crop.size == 0:
            continue

        pre_patch  = cv2.resize(pre_crop,  (patch_size, patch_size))
        post_patch = cv2.resize(post_crop, (patch_size, patch_size))

        comp_mask   = (labels_cc[y1:y2, x1:x2] == comp_id)
        damage_vals = damage_mask[y1:y2, x1:x2][comp_mask]
        valid       = damage_vals[(damage_vals >= 1) & (damage_vals <= 4)]

        if len(valid) == 0:
            damage = -1
        elif len(valid) / max(comp_mask.sum(), 1) < min_confident_ratio:
            damage = -1
        else:
            damage = int(np.bincount(valid).argmax()) - 1

        patches.append({'pre_patch':pre_patch,'post_patch':post_patch,
                        'damage':damage,'bbox':(x1,y1,x2-x1,y2-y1)})
    return patches


def extract_all_patches(img_dir, lbl_dir, output_dir,
                        mask_dir=None, use_gt_mask=True):
    img_dir  = Path(img_dir)
    lbl_dir  = Path(lbl_dir)
    out_dir  = Path(output_dir)
    (out_dir/'pre').mkdir(parents=True, exist_ok=True)
    (out_dir/'post').mkdir(parents=True, exist_ok=True)

    labels_dict, patch_count = {}, 0
    pre_images = sorted(img_dir.glob('*_pre_disaster.png'))
    print(f'Found {len(pre_images)} pre-disaster images...')

    for pre_path in tqdm(pre_images, desc='Extracting patches'):
        stem      = pre_path.stem
        post_path = img_dir / (stem.replace('_pre_','_post_')+'.png')
        if not post_path.exists(): continue

        if use_gt_mask:
            pre_json = lbl_dir / f'{stem}.json'
            if not pre_json.exists(): continue
            img_tmp = cv2.imread(str(pre_path))
            h,w = img_tmp.shape[:2]
            mask = json_to_mask(str(pre_json), height=h, width=w)
        else:
            mask_path = Path(mask_dir) / f'{stem}_mask.png'
            if not mask_path.exists(): continue
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        post_json = lbl_dir / (stem.replace('_pre_','_post_')+'.json')
        patches   = extract_patches_from_pair(
            str(pre_path), str(post_path), mask,
            str(post_json) if post_json.exists() else None)

        for i, p in enumerate(patches):
            if p['damage'] == -1: continue
            pid = f'{stem}_{i:04d}'
            cv2.imwrite(str(out_dir/'pre' /f'{pid}.png'),
                        cv2.cvtColor(p['pre_patch'], cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(out_dir/'post'/f'{pid}.png'),
                        cv2.cvtColor(p['post_patch'],cv2.COLOR_RGB2BGR))
            labels_dict[pid] = p['damage']
            patch_count += 1

    with open(out_dir/'labels.json','w') as f:
        json.dump(labels_dict, f)
    print(f'✓ {patch_count} labelled patches → {output_dir}')
    dist = Counter(labels_dict.values())
    for cid, name in Config.DAMAGE_CLASSES.items():
        print(f'  {cid} ({name}): {dist.get(cid,0)}')

print('✓ Patch extraction defined')


# ─────────────────────────────────────────────────────────────────
# Model loaders
# ─────────────────────────────────────────────────────────────────
def load_seg_model(ckpt_path, device):
    model = build_segmentation_model()
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    print(f'✓ Seg model loaded (IoU={ckpt.get("best_iou","?"):.4f})')
    return model


def load_cls_model(ckpt_path, device):
    model = DualResNet50(num_classes=Config.CLS_NUM_CLASSES, pretrained=False)
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    print(f'✓ Cls model loaded (F1={ckpt.get("best_f1","?"):.4f})')
    return model

print('✓ Model loaders defined')


# ─────────────────────────────────────────────────────────────────
# Visualization helpers
# ─────────────────────────────────────────────────────────────────
def draw_predictions(post_rgb, patches, predictions, alpha=0.5):
    vis = cv2.cvtColor(post_rgb, cv2.COLOR_RGB2BGR).copy()
    for p, pred in zip(patches, predictions):
        x, y, bw, bh = p['bbox']
        color = Config.DAMAGE_COLORS.get(pred, (128,128,128))
        label = Config.DAMAGE_CLASSES.get(pred, 'unknown')
        ov = vis.copy()
        cv2.rectangle(ov,  (x,y),(x+bw,y+bh), color, -1)
        cv2.addWeighted(ov, alpha, vis, 1-alpha, 0, vis)
        cv2.rectangle(vis, (x,y),(x+bw,y+bh), color, 2)
        cv2.putText(vis, label, (x, max(0,y-4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return vis


def draw_legend(vis):
    lh = 30*Config.CLS_NUM_CLASSES+10
    h, _  = vis.shape[:2]
    x0,y0 = 10, h-lh-10
    ov = vis.copy()
    cv2.rectangle(ov,(x0,y0),(x0+180,y0+lh),(0,0,0),-1)
    cv2.addWeighted(ov, 0.6, vis, 0.4, 0, vis)
    for i,(cid,name) in enumerate(Config.DAMAGE_CLASSES.items()):
        c  = Config.DAMAGE_COLORS[cid]
        cy = y0+10+i*30
        cv2.rectangle(vis,(x0+5,cy),(x0+25,cy+18),c,-1)
        cv2.putText(vis, name,(x0+32,cy+13),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)
    return vis

print('✓ Visualization helpers defined')


# ─────────────────────────────────────────────────────────────────
# End-to-end inference (5-way TTA by default)
# ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def classify_patches(cls_model, patches, device):
    cls_tf = get_cls_transforms(train=False)
    preds  = []
    for p in patches:
        pre_t  = cls_tf(image=p['pre_patch'])['image'].unsqueeze(0).to(device)
        post_t = cls_tf(image=p['post_patch'])['image'].unsqueeze(0).to(device)
        with torch.amp.autocast('cuda'):
            logit = cls_model(pre_t, post_t)
        preds.append(logit.argmax(1).item())
    return preds


def run_inference(pre_path, post_path, seg_model, cls_model, device,
                  output_dir=Config.PRED_DIR, use_tta=True):
    pre_rgb  = cv2.cvtColor(cv2.imread(pre_path),  cv2.COLOR_BGR2RGB)
    post_rgb = cv2.cvtColor(cv2.imread(post_path), cv2.COLOR_BGR2RGB)

    print(f'  [1/3] Building mask ({"5-way TTA" if use_tta else "single"})...')
    if use_tta:
        mask = predict_mask_tta(seg_model, pre_rgb, post_rgb, device)
    else:
        seg_tf = A.Compose([
            A.Resize(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
            ToTensorV2(),
        ])
        p   = seg_tf(image=pre_rgb)['image']
        q   = seg_tf(image=post_rgb)['image']
        inp = torch.cat([p, q], dim=0).unsqueeze(0).to(device)
        with torch.amp.autocast('cuda'):
            logit = seg_model(inp)
        prob = torch.sigmoid(logit).squeeze().cpu().numpy()
        mask = (prob > Config.SEG_THRESHOLD).astype(np.uint8) * 255
        orig_h, orig_w = pre_rgb.shape[:2]
        mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
        mask = postprocess_mask(mask)

    print('  [2/3] Extracting patches (watershed)...')
    patches = extract_patches_from_pair(pre_path, post_path, mask)
    print(f'         {len(patches)} buildings')

    if not patches:
        print('  WARNING: No buildings found.')
        return {'mask':mask,'predictions':[],'patches':[],'vis_path':None}

    print('  [3/3] Classifying...')
    predictions = classify_patches(cls_model, patches, device)
    dist = Counter(predictions)
    for cid,name in Config.DAMAGE_CLASSES.items():
        print(f'         {name}: {dist.get(cid,0)}')

    vis  = draw_legend(draw_predictions(post_rgb, patches, predictions))
    stem = Path(pre_path).stem.replace('_pre_disaster','')
    vp   = os.path.join(output_dir, f'{stem}_damage_map.png')
    cv2.imwrite(vp, vis)
    cv2.imwrite(os.path.join(output_dir, f'{stem}_building_mask.png'), mask)
    print(f'  ✓ Saved → {vp}')

    return {'mask':mask,'predictions':predictions,'patches':patches,'vis_path':vp}

print('✓ Inference pipeline defined')


# ─────────────────────────────────────────────────────────────────
# Full Pipeline
# ─────────────────────────────────────────────────────────────────
def run_full_pipeline(use_gt_mask=True):
    print('\n'+'═'*60)
    print('  🚀  DISASTER DAMAGE PIPELINE  v4')
    print('═'*60)

    print('\n── STAGE 1 ── DeepLabV3+ dual-input, scene-split, 5-way TTA')
    seg_ckpt = train_segmentation()

    print('\n── STAGE 1.5 ── Patch extraction (watershed + quality filter 0.5)')
    extract_all_patches(img_dir=Config.TRAIN_IMG_DIR,
                        lbl_dir=Config.TRAIN_LBL_DIR,
                        output_dir=Config.PATCH_DIR,
                        use_gt_mask=use_gt_mask)

    print('\n── STAGE 2 ── DualResNet50 (spatial-attn, multi-scale, HNM)')
    cls_ckpt = train_classification(patch_dir=Config.PATCH_DIR)

    print('\n'+'═'*60)
    print('  ✓  PIPELINE COMPLETE')
    print(f'  Seg ckpt : {seg_ckpt}')
    print(f'  Cls ckpt : {cls_ckpt}')
    print(f'  Outputs  : {Config.OUTPUT_DIR}')
    print('═'*60)
    return seg_ckpt, cls_ckpt

print('✓ Full pipeline defined')


# ─────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────
seg_ckpt, cls_ckpt = run_full_pipeline(use_gt_mask=True)

# ── OR run stages individually ────────────────────────────────────
# seg_ckpt = train_segmentation()
#
# extract_all_patches(
#     img_dir    = Config.TRAIN_IMG_DIR,
#     lbl_dir    = Config.TRAIN_LBL_DIR,
#     output_dir = Config.PATCH_DIR,
#     use_gt_mask= True,
# )
#
# cls_ckpt = train_classification()
#
# device    = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
# seg_model = load_seg_model('./outputs/checkpoints/seg_best.pth', device)
# cls_model = load_cls_model('./outputs/checkpoints/cls_best.pth', device)
# train_img  = Path(Config.TRAIN_IMG_DIR)
# pre_path   = str(sorted(train_img.glob('*_pre_disaster.png'))[0])
# post_path  = pre_path.replace('_pre_disaster', '_post_disaster')
# result     = run_inference(pre_path, post_path, seg_model, cls_model, device)

GPU Available: True
GPU Name: NVIDIA A100-PCIE-40GB
GPU Memory: 42.29 GB
BASE_DIR           : /home/devnurma/Satellite-based disaster damage
train/images exists: True
train/labels exists: True
test/images  exists: True

Train folder — 5598 total files
  First 6: ['guatemala-volcano_00000000_post_disaster.png', 'guatemala-volcano_00000000_pre_disaster.png', 'guatemala-volcano_00000001_post_disaster.png', 'guatemala-volcano_00000001_pre_disaster.png', 'guatemala-volcano_00000002_post_disaster.png', 'guatemala-volcano_00000002_pre_disaster.png']

Test  folder — 1866 total files
  First 6: ['test_post_00000.png', 'test_post_00001.png', 'test_post_00002.png', 'test_post_00003.png', 'test_post_00004.png', 'test_post_00005.png']

✓ All paths look good
✓ Output directories created
✓ Segmentation model builder defined
✓ DualResNet50 with spatial attention defined
✓ Loss functions defined
Using device: cuda


Downloading: "https://github.com/huggingface/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_b5-c6949ce9.pth" to /home/devnurma/.cache/torch/hub/checkpoints/tf_efficientnet_b5-c6949ce9.pth
100%|██████████| 117M/117M [00:02<00:00, 61.1MB/s] 


Seg model — Input: (2,6,512,512)  Output: torch.Size([2, 1, 512, 512])
Cls model — Output: torch.Size([4, 4])
✓ Both models initialised successfully
✓ Mask helpers defined
✓ Split spatial / photometric transforms defined
✓ Disaster-scene split defined
✓ SegmentationDataset (FIX 2+8) defined
✓ PatchClassificationDataset (FIX 6+7) defined
✓ Class weight + sampler (FIX 7) defined
✓ Segmentation metrics defined
✓ Watershed instance separation (FIX 3) defined
✓ WarmupCosineScheduler defined
✓ 5-way TTA (FIX 5) defined
✓ Segmentation training helpers defined
✓ Segmentation training defined
✓ Classification training helpers (FIX 7) defined
✓ Classification training defined
✓ Patch extraction defined
✓ Model loaders defined
✓ Visualization helpers defined
✓ Inference pipeline defined
✓ Full pipeline defined

════════════════════════════════════════════════════════════
  🚀  DISASTER DAMAGE PIPELINE  v4
════════════════════════════════════════════════════════════

── STAGE 1 ── DeepLabV3+ dual-i

/tmp/ipykernel_114171/2721715054.py:384: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50)),
/tmp/ipykernel_114171/2721715054.py:388: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32,


SegDataset (train): 2686 samples
SegDataset (val): 113 samples


/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



 Epoch |     TLoss |     VLoss |    IoU |   Dice |        LR | Enc
────────────────────────────────────────────────────────────────────
  [FIX 8] Encoder frozen
     1 |    0.5840 |    0.4398 | 0.3888 | 0.5418 |  1.21e-04 | frz  [249s]
  ✓ Saved (IoU=0.3888)
     2 |    0.4701 |    0.3504 | 0.4358 | 0.5936 |  1.80e-04 | frz  [249s]
  ✓ Saved (IoU=0.4358)
     3 |    0.4335 |    0.3355 | 0.4265 | 0.5800 |  2.40e-04 | frz  [245s]
     4 |    0.4209 |    0.3253 | 0.4444 | 0.5993 |  3.00e-04 | frz  [245s]
  ✓ Saved (IoU=0.4444)
     5 |    0.4096 |    0.3141 | 0.4643 | 0.6202 |  3.00e-04 | frz  [250s]
  ✓ Saved (IoU=0.4643)
  [FIX 8] Encoder unfrozen
     6 |    0.3853 |    0.2780 | 0.4996 | 0.6540 |  3.00e-04 | act  [270s]
  ✓ Saved (IoU=0.4996)
     7 |    0.3474 |    0.2738 | 0.4960 | 0.6455 |  2.99e-04 | act  [278s]
     8 |    0.3297 |    0.2661 | 0.5443 | 0.6935 |  2.99e-04 | act  [271s]
  ✓ Saved (IoU=0.5443)
     9 |    0.3192 |    0.2787 | 0.5353 | 0.6873 |  2.98e-04 | act  [270s

Extracting patches: 100%|██████████| 2799/2799 [13:17<00:00,  3.51it/s]  
/tmp/ipykernel_114171/2721715054.py:525: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.OneOf([A.GaussNoise(var_limit=(5,30)),


✓ 54308 labelled patches → ./outputs/patches
  0 (no-damage): 36042
  1 (minor-damage): 6206
  2 (major-damage): 7129
  3 (destroyed): 4931

── STAGE 2 ── DualResNet50 (spatial-attn, multi-scale, HNM)
Training on: cuda
PatchDataset: 54308 patches | 0 hard negatives
PatchDataset: 54308 patches | 0 hard negatives
PatchDataset: 54308 patches | 0 hard negatives
Class counts : [36042  6206  7129  4931]
Class weights: [0.377 2.188 1.904 2.753]

 Epoch |   T.Loss |   T.F1 |   V.Loss |   V.F1 |  V.Acc | HN
────────────────────────────────────────────────────────────────────────


IndexError: Caught IndexError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 50, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
  File "/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataset.py", line 420, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
  File "/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataset.py", line 420, in <listcomp>
    return [self.dataset[self.indices[idx]] for idx in indices]
IndexError: list index out of range


In [ ]:
import os
import json
import cv2
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
from scipy.ndimage import distance_transform_edt
from sklearn.metrics import f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from torchvision import models

# ─────────────────────────────────────────────────────────────────
# GPU Check
# ─────────────────────────────────────────────────────────────────
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# ─────────────────────────────────────────────────────────────────
# Data paths
# ─────────────────────────────────────────────────────────────────
BASE_DIR = Path.home() / 'Satellite-based disaster damage'

print('BASE_DIR           :', BASE_DIR)
print('train/images exists:', (BASE_DIR / 'train' / 'images').exists())
print('train/labels exists:', (BASE_DIR / 'train' / 'labels').exists())
print('test/images  exists:', (BASE_DIR / 'test'  / 'images').exists())

train_img_dir = BASE_DIR / 'train' / 'images'
test_img_dir  = BASE_DIR / 'test'  / 'images'
train_files   = list(train_img_dir.glob('*')) if train_img_dir.exists() else []
test_files    = list(test_img_dir.glob('*'))  if test_img_dir.exists()  else []

print(f'\nTrain folder — {len(train_files)} total files')
print('  First 6:', [f.name for f in sorted(train_files)[:6]])
print(f'\nTest  folder — {len(test_files)} total files')
print('  First 6:', [f.name for f in sorted(test_files)[:6]])

assert BASE_DIR.exists(), f'DATA ROOT not found: {BASE_DIR}'
print('\n✓ All paths look good')


# ─────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────
class Config:
    DATA_ROOT      = BASE_DIR
    TRAIN_IMG_DIR  = str(DATA_ROOT / 'train' / 'images')
    TRAIN_LBL_DIR  = str(DATA_ROOT / 'train' / 'labels')
    TEST_IMG_DIR   = str(DATA_ROOT / 'test'  / 'images')
    OUTPUT_DIR     = './outputs'
    CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
    LOG_DIR        = os.path.join(OUTPUT_DIR, 'logs')
    VIZ_DIR        = os.path.join(OUTPUT_DIR, 'visualizations')
    PRED_DIR       = os.path.join(OUTPUT_DIR, 'predictions')
    PATCH_DIR      = os.path.join(OUTPUT_DIR, 'patches')

    # ── Segmentation ──────────────────────────────────────────────
    SEG_IMG_SIZE      = 640
    SEG_CROP_SIZE     = 512
    SEG_BATCH_SIZE    = 4
    SEG_LR            = 3e-4
    SEG_EPOCHS        = 100
    SEG_THRESHOLD     = 0.5
    SEG_MIN_AREA      = 100
    SEG_IN_CHANNELS   = 6           # pre(3) + post(3)
    SEG_WARMUP_EPOCHS = 5
    SEG_FREEZE_EPOCHS = 5
    VAL_SCENE_FRACTION = 0.15

    # ── Classification ────────────────────────────────────────────
    CLS_PATCH_SIZES = [192, 224, 256]   # multi-scale training
    CLS_PATCH_SIZE  = 224               # inference size
    CLS_BATCH_SIZE  = 16
    CLS_LR          = 1e-4
    CLS_EPOCHS      = 50
    CLS_NUM_CLASSES = 4
    CLS_VAL_FRAC    = 0.15

    # Hard-negative mining
    HNM_START_EPOCH = 5
    HNM_OVERSAMPLE  = 3

    DAMAGE_CLASSES = {0: 'no-damage', 1: 'minor-damage',
                      2: 'major-damage', 3: 'destroyed'}
    DAMAGE_COLORS  = {0: (0, 200, 0), 1: (0, 200, 255),
                      2: (0, 100, 255), 3: (0, 0, 255)}

    SEED   = 42
    DEVICE = 'cuda'


def make_dirs():
    for d in [Config.OUTPUT_DIR, Config.CHECKPOINT_DIR,
              Config.LOG_DIR, Config.VIZ_DIR, Config.PRED_DIR, Config.PATCH_DIR]:
        os.makedirs(d, exist_ok=True)
    print('✓ Output directories created')

make_dirs()


# ─────────────────────────────────────────────────────────────────
# Models
# ─────────────────────────────────────────────────────────────────
def build_segmentation_model(encoder_name='timm-efficientnet-b5',
                              encoder_weights='imagenet',
                              in_channels=Config.SEG_IN_CHANNELS,
                              num_classes=1) -> nn.Module:
    return smp.DeepLabV3Plus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,
    )

print('✓ Segmentation model builder defined')


class DualResNet50(nn.Module):
    """
    Siamese ResNet50 with spatial attention on post-disaster features.
    Fusion: cat(f_pre_pooled, f_post_attended_pooled, |diff|)
    """

    def __init__(self, num_classes=4, pretrained=True, dropout=0.4):
        super().__init__()
        weights  = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet50(weights=weights)
        self.encoder = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4,
        )  # → (B, 2048, 7, 7)

        # Spatial attention: 2048 → 512 → 1 per spatial position
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 1, kernel_size=1),
            nn.Sigmoid(),
        )  # → (B, 1, 7, 7)

        self.pool = nn.AdaptiveAvgPool2d(1)

        feat_dim = 2048 * 3   # pre + post_attended + |diff|
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout / 2),
            nn.Linear(256, num_classes),
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, pre, post):
        f_pre  = self.encode(pre)
        f_post = self.encode(post)

        attn   = self.spatial_attn(f_post)   # (B, 1, 7, 7)
        f_post = f_post * attn               # spatial weighting

        p_pre  = self.pool(f_pre).flatten(1)
        p_post = self.pool(f_post).flatten(1)
        diff   = torch.abs(p_post - p_pre)

        return self.classifier(torch.cat([p_pre, p_post, diff], dim=1))

print('✓ DualResNet50 with spatial attention defined')


# ─────────────────────────────────────────────────────────────────
# Loss Functions
# ─────────────────────────────────────────────────────────────────
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, smooth=1e-6):
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth

    def forward(self, logits, targets):
        p  = torch.sigmoid(logits)
        TP = (p * targets).sum(dim=(2, 3))
        FP = ((1 - targets) * p).sum(dim=(2, 3))
        FN = (targets * (1 - p)).sum(dim=(2, 3))
        t  = (TP + self.smooth) / (TP + self.alpha*FP + self.beta*FN + self.smooth)
        return (1 - t).mean()


def compute_dist_map(mask_np: np.ndarray) -> np.ndarray:
    """Symmetric distance map: strong weight on both sides of building edges."""
    pos = distance_transform_edt(mask_np)
    neg = distance_transform_edt(1.0 - mask_np)
    d   = pos + neg
    return d / (d.max() + 1e-6)


class DistanceTransformLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        dist_maps = []
        for t in targets.cpu().numpy():
            dist_maps.append(compute_dist_map(t[0]))
        dist_maps = torch.tensor(
            np.stack(dist_maps), dtype=torch.float32
        ).unsqueeze(1).to(logits.device)
        return ((probs - targets) ** 2 * (1 + 5 * dist_maps)).mean()


class CombinedSegLoss(nn.Module):
    """0.4·BCE + 0.3·Dice + 0.2·Tversky + 0.1·SymDistBoundary"""
    def __init__(self):
        super().__init__()
        self.bce       = nn.BCEWithLogitsLoss()
        self.tversky   = TverskyLoss(alpha=0.7, beta=0.3)
        self.dist_bdry = DistanceTransformLoss()

    def _dice(self, logits, targets):
        p      = torch.sigmoid(logits)
        smooth = 1e-6
        inter  = (p * targets).sum(dim=(2, 3))
        return (1 - (2*inter + smooth) /
                (p.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + smooth)).mean()

    def forward(self, logits, targets):
        return (0.40 * self.bce(logits, targets) +
                0.30 * self._dice(logits, targets) +
                0.20 * self.tversky(logits, targets) +
                0.10 * self.dist_bdry(logits, targets))


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma, self.weight = gamma, weight

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        p_t = torch.exp(-ce)
        return ((1 - p_t) ** self.gamma * ce).mean()

print('✓ Loss functions defined')


# ─────────────────────────────────────────────────────────────────
# Sanity check
# ─────────────────────────────────────────────────────────────────
device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

seg = build_segmentation_model()
out = seg(torch.randn(2, 6, 512, 512))
print(f'Seg model — Input: (2,6,512,512)  Output: {out.shape}')
del seg, out

cls = DualResNet50(num_classes=4)
out = cls(torch.randn(4, 3, 224, 224), torch.randn(4, 3, 224, 224))
print(f'Cls model — Output: {out.shape}')
del cls, out
print('✓ Both models initialised successfully')


# ─────────────────────────────────────────────────────────────────
# Mask helpers
# ─────────────────────────────────────────────────────────────────
def json_to_mask(json_path, height=1024, width=1024):
    with open(json_path) as f:
        data = json.load(f)
    mask = np.zeros((height, width), dtype=np.uint8)
    for feat in data.get('features', {}).get('xy', []):
        geom = feat.get('wkt', '')
        if not geom.startswith('POLYGON'):
            continue
        coords = geom.replace('POLYGON ((', '').replace('))', '').strip()
        pts = []
        for pair in coords.split(','):
            pair = pair.strip()
            if pair:
                x, y = pair.split()
                pts.append([float(x), float(y)])
        if len(pts) >= 3:
            cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], 255)
    return mask


def json_to_damage_mask(json_path, height=1024, width=1024):
    subtype_map = {'no-damage': 1, 'minor-damage': 2,
                   'major-damage': 3, 'destroyed': 4, 'un-classified': 0}
    with open(json_path) as f:
        data = json.load(f)
    mask = np.zeros((height, width), dtype=np.uint8)
    for feat in data.get('features', {}).get('xy', []):
        geom  = feat.get('wkt', '')
        label = subtype_map.get(
            feat.get('properties', {}).get('subtype', 'un-classified'), 0)
        if not geom.startswith('POLYGON') or label == 0:
            continue
        coords = geom.replace('POLYGON ((', '').replace('))', '').strip()
        pts = []
        for pair in coords.split(','):
            pair = pair.strip()
            if pair:
                x, y = pair.split()
                pts.append([float(x), float(y)])
        if len(pts) >= 3:
            cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], label)
    return mask

print('✓ Mask helpers defined')


# ─────────────────────────────────────────────────────────────────
# Transforms — spatial (ReplayCompose) + photometric (independent)
# ─────────────────────────────────────────────────────────────────
def get_spatial_transforms(train: bool = True):
    if train:
        return A.ReplayCompose([
            A.Resize(Config.SEG_IMG_SIZE, Config.SEG_IMG_SIZE),
            A.RandomCrop(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
        ])
    return A.ReplayCompose([
        A.Resize(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
    ])


def get_photometric_transforms():
    return A.Compose([
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
            A.CLAHE(clip_limit=4.0),
            A.RandomGamma(gamma_limit=(70, 130)),
        ], p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50)),
            A.MotionBlur(blur_limit=5),
            A.MedianBlur(blur_limit=5),
        ], p=0.3),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                        fill_value=0, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


_val_norm = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print('✓ Split spatial / photometric transforms defined')


# ─────────────────────────────────────────────────────────────────
# Scene-aware train / val split for segmentation
# ─────────────────────────────────────────────────────────────────
def disaster_scene_split(img_dir: str, val_fraction: float = 0.15, seed: int = 42):
    img_dir   = Path(img_dir)
    pre_files = sorted(img_dir.glob('*_pre_disaster.png'))

    scenes = defaultdict(list)
    for p in pre_files:
        parts  = p.stem.split('_pre_disaster')[0]
        tokens = parts.rsplit('_', 1)
        scene  = tokens[0] if len(tokens) > 1 else parts
        scenes[scene].append(str(p))

    scene_names = sorted(scenes.keys())
    rng = random.Random(seed)
    rng.shuffle(scene_names)

    n_val      = max(1, int(len(scene_names) * val_fraction))
    val_scenes = set(scene_names[:n_val])
    trn_scenes = set(scene_names[n_val:])

    train_files = [f for s in trn_scenes for f in scenes[s]]
    val_files   = [f for s in val_scenes  for f in scenes[s]]

    print(f'Scene split — {len(trn_scenes)} train scenes ({len(train_files)} samples) | '
          f'{len(val_scenes)} val scenes ({len(val_files)} samples)')
    return sorted(train_files), sorted(val_files)

print('✓ Scene-aware split defined')


# ─────────────────────────────────────────────────────────────────
# Segmentation Dataset
# ─────────────────────────────────────────────────────────────────
class SegmentationDataset(Dataset):
    def __init__(self, pre_paths: list, lbl_dir: str,
                 spatial_tf=None, photo_tf=None, is_train: bool = True):
        self.lbl_dir    = Path(lbl_dir)
        self.spatial_tf = spatial_tf
        self.photo_tf   = photo_tf
        self.is_train   = is_train

        self.samples = []
        for pre_path in pre_paths:
            pre_path  = Path(pre_path)
            stem      = pre_path.stem
            post_path = pre_path.parent / (stem.replace('_pre_', '_post_') + '.png')
            json_path = self.lbl_dir / f'{stem}.json'
            if post_path.exists() and json_path.exists():
                self.samples.append((str(pre_path), str(post_path), str(json_path)))

        print(f'SegDataset ({"train" if is_train else "val"}): '
              f'{len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pre_path, post_path, json_path = self.samples[idx]

        pre  = cv2.cvtColor(cv2.imread(pre_path),  cv2.COLOR_BGR2RGB)
        post = cv2.cvtColor(cv2.imread(post_path), cv2.COLOR_BGR2RGB)
        h, w = pre.shape[:2]

        mask = json_to_mask(json_path, height=h, width=w)
        mask = (mask > 0).astype(np.float32)

        if self.spatial_tf:
            # Same spatial ops on both images
            aug_pre = self.spatial_tf(image=pre, mask=mask)
            pre_sp  = aug_pre['image']
            mask    = aug_pre['mask']
            post_sp = A.ReplayCompose.replay(aug_pre['replay'], image=post)['image']
        else:
            pre_sp, post_sp = pre, post

        if self.photo_tf and self.is_train:
            # Independent photometric ops (different lighting per image)
            pre_t  = self.photo_tf(image=pre_sp)['image']
            post_t = self.photo_tf(image=post_sp)['image']
        else:
            pre_t  = _val_norm(image=pre_sp)['image']
            post_t = _val_norm(image=post_sp)['image']

        return torch.cat([pre_t, post_t], dim=0), torch.from_numpy(mask).unsqueeze(0)

print('✓ SegmentationDataset defined')


# ─────────────────────────────────────────────────────────────────
# Classification transforms
# ─────────────────────────────────────────────────────────────────
def get_cls_transforms(train=True):
    norm = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    if train:
        return A.Compose([
            A.Resize(Config.CLS_PATCH_SIZE, Config.CLS_PATCH_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.3),
            A.OneOf([A.RandomBrightnessContrast(0.2, 0.2),
                     A.CLAHE(), A.RandomGamma()], p=0.4),
            A.OneOf([A.GaussNoise(var_limit=(5, 30)),
                     A.MotionBlur(blur_limit=3)], p=0.2),
            norm, ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(Config.CLS_PATCH_SIZE, Config.CLS_PATCH_SIZE),
        norm, ToTensorV2(),
    ])


# ─────────────────────────────────────────────────────────────────
# FIX (IndexError root cause):
# PatchClassificationDataset now accepts id_subset so train/val
# datasets are fully independent objects with no Subset wrapping.
# The sampler is then built from the *same* dataset object that the
# DataLoader uses, so indices always stay in range.
# ─────────────────────────────────────────────────────────────────
class PatchClassificationDataset(Dataset):
    """
    Parameters
    ----------
    patch_dir   : directory with pre/, post/, labels.json
    transform   : albumentations Compose applied to each patch
    id_subset   : if given, restrict dataset to these patch IDs only
    multiscale  : randomly resize crop to one of CLS_PATCH_SIZES before
                  transform (training only)
    """

    def __init__(self, patch_dir: str, transform=None,
                 id_subset=None, multiscale: bool = False):
        self.patch_dir  = Path(patch_dir)
        self.pre_dir    = self.patch_dir / 'pre'
        self.post_dir   = self.patch_dir / 'post'
        self.transform  = transform
        self.multiscale = multiscale

        labels_file = self.patch_dir / 'labels.json'
        if not labels_file.exists():
            raise FileNotFoundError(
                f"labels.json not found at '{labels_file}'. "
                "Run extract_all_patches() first.")

        with open(labels_file) as f:
            all_labels = json.load(f)

        # Filter to requested subset if provided
        if id_subset is not None:
            all_labels = {k: v for k, v in all_labels.items() if k in id_subset}

        self.labels = all_labels
        self.ids    = sorted(self.labels.keys())
        print(f'PatchDataset: {len(self.ids)} patches')

    def __len__(self):
        return len(self.ids)

    def get_labels(self) -> list:
        return [int(self.labels[i]) for i in self.ids]

    def __getitem__(self, idx):
        patch_id = self.ids[idx]
        label    = int(self.labels[patch_id])

        pre  = cv2.cvtColor(cv2.imread(str(self.pre_dir  / f'{patch_id}.png')),
                            cv2.COLOR_BGR2RGB)
        post = cv2.cvtColor(cv2.imread(str(self.post_dir / f'{patch_id}.png')),
                            cv2.COLOR_BGR2RGB)

        # Multi-scale: random resize before augmentation
        if self.multiscale:
            s    = random.choice(Config.CLS_PATCH_SIZES)
            pre  = cv2.resize(pre,  (s, s))
            post = cv2.resize(post, (s, s))

        if self.transform:
            pre  = self.transform(image=pre)['image']
            post = self.transform(image=post)['image']
        else:
            pre  = torch.from_numpy(pre.transpose(2, 0, 1)).float()  / 255.
            post = torch.from_numpy(post.transpose(2, 0, 1)).float() / 255.

        # Return patch_id so HNM can track misclassified samples
        return pre, post, torch.tensor(label, dtype=torch.long), patch_id

print('✓ PatchClassificationDataset (id_subset, multi-scale) defined')


# ─────────────────────────────────────────────────────────────────
# Class weights & sampler helpers
# ─────────────────────────────────────────────────────────────────
def compute_class_weights(patch_dir: str,
                          num_classes: int = Config.CLS_NUM_CLASSES) -> torch.Tensor:
    with open(Path(patch_dir) / 'labels.json') as f:
        labels = json.load(f)
    counts  = np.zeros(num_classes, dtype=np.float32)
    for v in labels.values():
        counts[int(v)] += 1
    counts  = np.where(counts == 0, 1, counts)
    weights = counts.sum() / (num_classes * counts)
    print('Class counts :', counts.astype(int))
    print('Class weights:', np.round(weights, 3))
    return torch.tensor(weights, dtype=torch.float32)


def build_weighted_sampler(dataset: PatchClassificationDataset,
                           hard_neg_ids: set = None,
                           num_classes: int = Config.CLS_NUM_CLASSES
                           ) -> WeightedRandomSampler:
    """
    Build a sampler over `dataset`.
    `hard_neg_ids` is a subset of dataset.ids — those samples get
    their weight multiplied by HNM_OVERSAMPLE.

    IMPORTANT: call this with the *exact* dataset object that the
    DataLoader will use so that len(sampler) == len(dataset).
    """
    all_labels = dataset.get_labels()
    counts     = np.bincount(all_labels, minlength=num_classes).astype(float)
    counts     = np.where(counts == 0, 1, counts)
    cls_weight = 1.0 / counts

    sample_w = []
    for pid, lbl in zip(dataset.ids, all_labels):
        w = cls_weight[lbl]
        if hard_neg_ids and pid in hard_neg_ids:
            w *= Config.HNM_OVERSAMPLE
        sample_w.append(w)

    return WeightedRandomSampler(sample_w,
                                 num_samples=len(sample_w),
                                 replacement=True)

print('✓ Class weight + sampler defined')


# ─────────────────────────────────────────────────────────────────
# Collate fn — keeps patch_id strings alongside tensors
# ─────────────────────────────────────────────────────────────────
def _collate_with_ids(batch):
    pre    = torch.stack([b[0] for b in batch])
    post   = torch.stack([b[1] for b in batch])
    labels = torch.stack([b[2] for b in batch])
    ids    = [b[3] for b in batch]
    return pre, post, labels, ids


# ─────────────────────────────────────────────────────────────────
# Segmentation metrics
# ─────────────────────────────────────────────────────────────────
def iou_score(pred, gt, thr=0.5):
    pred  = (pred > thr).astype(np.uint8)
    gt    = (gt   > thr).astype(np.uint8)
    return (pred & gt).sum() / ((pred | gt).sum() + 1e-6)


def dice_score(pred, gt, thr=0.5):
    pred  = (pred > thr).astype(np.uint8)
    gt    = (gt   > thr).astype(np.uint8)
    inter = (pred & gt).sum()
    return (2 * inter) / (pred.sum() + gt.sum() + 1e-6)

print('✓ Segmentation metrics defined')


# ─────────────────────────────────────────────────────────────────
# Watershed instance separation + post-processing
# ─────────────────────────────────────────────────────────────────
def separate_instances_watershed(mask: np.ndarray) -> np.ndarray:
    binary = (mask > 0).astype(np.uint8)
    dist   = cv2.distanceTransform(binary * 255, cv2.DIST_L2, 5)
    if dist.max() == 0:
        return mask

    _, sure_fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
    sure_fg    = np.uint8(sure_fg)

    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    sure_bg = cv2.dilate(binary * 255, kernel, iterations=2)
    unknown = cv2.subtract(sure_bg, sure_fg)

    _, markers = cv2.connectedComponents(sure_fg)
    markers    = markers + 1
    markers[unknown == 255] = 0

    img_bgr = cv2.cvtColor(binary * 255, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(img_bgr, markers)

    result = np.zeros_like(binary, dtype=np.uint8)
    result[markers > 1] = 255
    return result


def postprocess_mask(mask: np.ndarray,
                     min_area: int = Config.SEG_MIN_AREA,
                     use_watershed: bool = True) -> np.ndarray:
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    if use_watershed and cleaned.max() > 0:
        cleaned = separate_instances_watershed(cleaned)

    num_cc, labels_cc, stats, _ = cv2.connectedComponentsWithStats(cleaned)
    out = np.zeros_like(cleaned)
    for c in range(1, num_cc):
        if stats[c, cv2.CC_STAT_AREA] >= min_area:
            out[labels_cc == c] = 255
    return out

print('✓ Watershed post-processing defined')


# ─────────────────────────────────────────────────────────────────
# Warmup + Cosine LR scheduler
# ─────────────────────────────────────────────────────────────────
class WarmupCosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, total_epochs,
                 min_lr=1e-6, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.min_lr        = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        ep = self.last_epoch + 1
        if ep <= self.warmup_epochs:
            scale = ep / max(1, self.warmup_epochs)
        else:
            prog  = (ep - self.warmup_epochs) / max(
                1, self.total_epochs - self.warmup_epochs)
            scale = 0.5 * (1 + np.cos(np.pi * prog))
        return [self.min_lr + (b - self.min_lr) * scale for b in self.base_lrs]

print('✓ WarmupCosineScheduler defined')


# ─────────────────────────────────────────────────────────────────
# 5-way TTA for segmentation inference
# ─────────────────────────────────────────────────────────────────
def _tta_variants(pre_rgb, post_rgb):
    return [
        (pre_rgb,             post_rgb,             lambda x: x),
        (pre_rgb[:, ::-1],    post_rgb[:, ::-1],    lambda x: x[:, ::-1]),
        (pre_rgb[::-1, :],    post_rgb[::-1, :],    lambda x: x[::-1, :]),
        (np.rot90(pre_rgb,1), np.rot90(post_rgb,1), lambda x: np.rot90(x,-1)),
        (np.rot90(pre_rgb,2), np.rot90(post_rgb,2), lambda x: np.rot90(x,-2)),
    ]


@torch.no_grad()
def predict_mask_tta(seg_model, pre_rgb, post_rgb, device,
                     threshold=Config.SEG_THRESHOLD):
    orig_h, orig_w = pre_rgb.shape[:2]
    seg_tf = A.Compose([
        A.Resize(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

    def _prep(pre, post):
        p = seg_tf(image=np.ascontiguousarray(pre))['image']
        q = seg_tf(image=np.ascontiguousarray(post))['image']
        return torch.cat([p, q], dim=0).unsqueeze(0).to(device)

    prob_sum = None
    for pre_a, post_a, undo in _tta_variants(pre_rgb, post_rgb):
        with torch.amp.autocast('cuda'):
            logit = seg_model(_prep(pre_a, post_a))
        prob = undo(torch.sigmoid(logit).squeeze().cpu().numpy())
        prob_sum = prob if prob_sum is None else prob_sum + prob

    mask = ((prob_sum / 5) > threshold).astype(np.uint8) * 255
    mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    return postprocess_mask(mask)

print('✓ 5-way TTA defined')


# ─────────────────────────────────────────────────────────────────
# Segmentation training helpers
# ─────────────────────────────────────────────────────────────────
def train_seg_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def validate_seg(model, loader, criterion, device):
    model.eval()
    total, ious, dices = 0.0, [], []
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            total += criterion(logits, masks).item()
        probs = torch.sigmoid(logits).cpu().numpy()
        gt    = masks.cpu().numpy()
        for p, g in zip(probs, gt):
            ious.append(iou_score(p[0], g[0]))
            dices.append(dice_score(p[0], g[0]))
    return total / len(loader), float(np.mean(ious)), float(np.mean(dices))

print('✓ Segmentation training helpers defined')


# ─────────────────────────────────────────────────────────────────
# Segmentation Training Loop
# ─────────────────────────────────────────────────────────────────
def train_segmentation():
    device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Training on: {device}')

    train_paths, val_paths = disaster_scene_split(
        Config.TRAIN_IMG_DIR, Config.VAL_SCENE_FRACTION, Config.SEED)

    train_ds = SegmentationDataset(
        train_paths, Config.TRAIN_LBL_DIR,
        spatial_tf=get_spatial_transforms(train=True),
        photo_tf=get_photometric_transforms(), is_train=True)
    val_ds = SegmentationDataset(
        val_paths, Config.TRAIN_LBL_DIR,
        spatial_tf=get_spatial_transforms(train=False),
        photo_tf=None, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=Config.SEG_BATCH_SIZE,
                              shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=Config.SEG_BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    model     = build_segmentation_model().to(device)
    criterion = CombinedSegLoss()
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=Config.SEG_LR, weight_decay=1e-4)
    scheduler = WarmupCosineScheduler(optimizer,
                                      Config.SEG_WARMUP_EPOCHS,
                                      Config.SEG_EPOCHS, min_lr=1e-6)
    scaler    = torch.amp.GradScaler('cuda')
    writer    = SummaryWriter(os.path.join(Config.LOG_DIR, 'segmentation'))

    best_iou, best_ckpt   = 0.0, os.path.join(Config.CHECKPOINT_DIR, 'seg_best.pth')
    patience, patience_ctr = 10, 0

    print(f'\n{"Epoch":>6} | {"TLoss":>9} | {"VLoss":>9} | '
          f'{"IoU":>6} | {"Dice":>6} | {"LR":>9} | Enc')
    print('─' * 68)

    for epoch in range(1, Config.SEG_EPOCHS + 1):
        t0 = time.time()

        if epoch == 1:
            for p in model.encoder.parameters(): p.requires_grad = False
            print('  Encoder frozen')
        elif epoch == Config.SEG_FREEZE_EPOCHS + 1:
            for p in model.encoder.parameters(): p.requires_grad = True
            print('  Encoder unfrozen')

        enc = 'frz' if epoch <= Config.SEG_FREEZE_EPOCHS else 'act'
        tl  = train_seg_one_epoch(model, train_loader, optimizer,
                                  criterion, device, scaler)
        vl, iou, dice = validate_seg(model, val_loader, criterion, device)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        print(f'{epoch:>6} | {tl:>9.4f} | {vl:>9.4f} | '
              f'{iou:>6.4f} | {dice:>6.4f} | {lr:>9.2e} | {enc}  '
              f'[{time.time()-t0:.0f}s]')

        writer.add_scalars('Loss',   {'train': tl,   'val': vl},   epoch)
        writer.add_scalars('Metric', {'IoU': iou, 'Dice': dice},   epoch)

        if iou > best_iou:
            best_iou, patience_ctr = iou, 0
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'best_iou': best_iou}, best_ckpt)
            print(f'  ✓ Saved (IoU={best_iou:.4f})')
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    writer.close()
    print(f'\n✓ Seg done. Best IoU: {best_iou:.4f}  →  {best_ckpt}')
    return best_ckpt

print('✓ Segmentation training defined')


# ─────────────────────────────────────────────────────────────────
# Classification metrics
# ─────────────────────────────────────────────────────────────────
def compute_cls_metrics(preds, labels, num_classes=Config.CLS_NUM_CLASSES):
    preds, labels = np.array(preds), np.array(labels)
    f1  = f1_score(labels, preds, average='macro',
                   labels=list(range(num_classes)), zero_division=0)
    cm  = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    acc = (preds == labels).mean()
    return f1, acc, cm


# ─────────────────────────────────────────────────────────────────
# Classification training helpers
# HNM: both train + val wrong IDs are tracked so the sampler can
# boost any hard example that lives in the training set.
# ─────────────────────────────────────────────────────────────────
def _run_one_epoch(model, loader, optimizer, criterion,
                   device, scaler, train: bool):
    """Unified train/val loop. Returns (loss, f1, acc, wrong_ids)."""
    model.train(train)
    total, all_preds, all_labels, wrong_ids = 0.0, [], [], set()

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for pre, post, labels, patch_ids in loader:
            pre, post, labels = (pre.to(device), post.to(device),
                                 labels.to(device))
            if train:
                optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                logits = model(pre, post)
                loss   = criterion(logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total += loss.item()
            preds  = logits.argmax(1).detach().cpu().numpy()
            gts    = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(gts)

            # Collect misclassified ids for HNM
            for pid, pred, gt in zip(patch_ids, preds, gts):
                if pred != gt:
                    wrong_ids.add(pid)

    f1, acc, cm = compute_cls_metrics(all_preds, all_labels)
    return total / len(loader), f1, acc, cm, wrong_ids

print('✓ Classification training helpers defined')


# ─────────────────────────────────────────────────────────────────
# Classification Training Loop
# ─────────────────────────────────────────────────────────────────
def train_classification(patch_dir: str = Config.PATCH_DIR):
    device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Training on: {device}')

    # ── Build a clean train / val split at the ID level ──────────
    # Load all ids once
    with open(Path(patch_dir) / 'labels.json') as f:
        all_labels_dict = json.load(f)

    all_ids = sorted(all_labels_dict.keys())
    rng     = random.Random(Config.SEED)
    rng.shuffle(all_ids)

    n_val      = max(1, int(len(all_ids) * Config.CLS_VAL_FRAC))
    val_id_set = set(all_ids[:n_val])
    trn_id_set = set(all_ids[n_val:])

    print(f'Patch split — {len(trn_id_set)} train | {len(val_id_set)} val')

    # ── Create fully independent train / val dataset objects ──────
    # Each dataset object owns exactly the IDs assigned to it.
    # No Subset wrapping → sampler indices always match dataset size.
    train_ds = PatchClassificationDataset(
        patch_dir, transform=get_cls_transforms(train=True),
        id_subset=trn_id_set, multiscale=True)

    val_ds = PatchClassificationDataset(
        patch_dir, transform=get_cls_transforms(train=False),
        id_subset=val_id_set, multiscale=False)

    class_weights = compute_class_weights(patch_dir).to(device)
    model     = DualResNet50(num_classes=Config.CLS_NUM_CLASSES,
                             pretrained=True, dropout=0.4).to(device)
    criterion = FocalLoss(gamma=2.0, weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=Config.CLS_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.CLS_EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda')
    writer    = SummaryWriter(os.path.join(Config.LOG_DIR, 'classification'))

    best_f1, best_ckpt    = 0.0, os.path.join(Config.CHECKPOINT_DIR, 'cls_best.pth')
    patience, patience_ctr = 10, 0
    hard_neg_ids: set      = set()   # IDs in train_ds that were misclassified

    print(f'\n{"Epoch":>6} | {"T.Loss":>8} | {"T.F1":>6} | '
          f'{"V.Loss":>8} | {"V.F1":>6} | {"V.Acc":>6} | HN')
    print('─' * 72)

    for epoch in range(1, Config.CLS_EPOCHS + 1):
        t0 = time.time()

        # ── Rebuild sampler each epoch (HNM boost after warmup) ───
        # Sampler is built from train_ds: len(sampler) == len(train_ds) ✓
        use_hnm = epoch >= Config.HNM_START_EPOCH
        sampler = build_weighted_sampler(
            train_ds,
            hard_neg_ids=hard_neg_ids if use_hnm else set())

        train_loader = DataLoader(
            train_ds, batch_size=Config.CLS_BATCH_SIZE,
            sampler=sampler, num_workers=4, pin_memory=True,
            collate_fn=_collate_with_ids)

        val_loader = DataLoader(
            val_ds, batch_size=Config.CLS_BATCH_SIZE,
            shuffle=False, num_workers=4, pin_memory=True,
            collate_fn=_collate_with_ids)

        t_loss, t_f1, t_acc, _, train_wrong = _run_one_epoch(
            model, train_loader, optimizer, criterion, device, scaler, train=True)
        v_loss, v_f1, v_acc, cm, _ = _run_one_epoch(
            model, val_loader, optimizer, criterion, device, scaler, train=False)

        # HNM: track training-set misclassifications (not val)
        # so that boosted IDs actually appear in the training sampler
        if use_hnm:
            hard_neg_ids = train_wrong & trn_id_set
            print(f'  [HNM] {len(hard_neg_ids)} train hard negatives')

        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        print(f'{epoch:>6} | {t_loss:>8.4f} | {t_f1:>6.4f} | '
              f'{v_loss:>8.4f} | {v_f1:>6.4f} | {v_acc:>6.4f} | '
              f'{len(hard_neg_ids)}  [{time.time()-t0:.0f}s]')

        writer.add_scalars('Loss', {'train': t_loss, 'val': v_loss}, epoch)
        writer.add_scalars('F1',   {'train': t_f1,   'val': v_f1},   epoch)
        writer.add_scalar('HardNegatives', len(hard_neg_ids), epoch)

        if v_f1 > best_f1:
            best_f1, patience_ctr = v_f1, 0
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'best_f1': best_f1}, best_ckpt)
            print(f'  ✓ Saved (F1={best_f1:.4f})')
            print('  CM:', [row.tolist() for row in cm])
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    writer.close()
    print(f'\n✓ Cls done. Best F1: {best_f1:.4f}  →  {best_ckpt}')
    return best_ckpt

print('✓ Classification training defined')


# ─────────────────────────────────────────────────────────────────
# Patch extraction
# ─────────────────────────────────────────────────────────────────
def extract_patches_from_pair(pre_img_path, post_img_path, mask,
                               post_json_path=None,
                               patch_size=Config.CLS_PATCH_SIZE,
                               min_area=Config.SEG_MIN_AREA,
                               min_confident_ratio=0.50):
    pre  = cv2.cvtColor(cv2.imread(pre_img_path),  cv2.COLOR_BGR2RGB)
    post = cv2.cvtColor(cv2.imread(post_img_path), cv2.COLOR_BGR2RGB)
    h, w = pre.shape[:2]

    if mask.shape != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    mask = postprocess_mask(mask, min_area=min_area, use_watershed=True)

    damage_mask = (json_to_damage_mask(post_json_path, h, w)
                   if post_json_path and Path(post_json_path).exists()
                   else np.full((h, w), 255, dtype=np.uint8))

    binary  = (mask > 0).astype(np.uint8) * 255
    num_cc, labels_cc, stats, _ = cv2.connectedComponentsWithStats(binary)

    patches = []
    for comp_id in range(1, num_cc):
        area = stats[comp_id, cv2.CC_STAT_AREA]
        if area < min_area:
            continue

        x, y   = stats[comp_id, cv2.CC_STAT_LEFT],  stats[comp_id, cv2.CC_STAT_TOP]
        bw, bh = stats[comp_id, cv2.CC_STAT_WIDTH], stats[comp_id, cv2.CC_STAT_HEIGHT]
        m = 10
        x1, y1 = max(0, x-m),   max(0, y-m)
        x2, y2 = min(w, x+bw+m), min(h, y+bh+m)

        pre_crop  = pre[y1:y2, x1:x2]
        post_crop = post[y1:y2, x1:x2]
        if pre_crop.size == 0 or post_crop.size == 0:
            continue

        pre_patch  = cv2.resize(pre_crop,  (patch_size, patch_size))
        post_patch = cv2.resize(post_crop, (patch_size, patch_size))

        comp_mask   = (labels_cc[y1:y2, x1:x2] == comp_id)
        damage_vals = damage_mask[y1:y2, x1:x2][comp_mask]
        valid       = damage_vals[(damage_vals >= 1) & (damage_vals <= 4)]

        if len(valid) == 0:
            damage = -1
        elif len(valid) / max(comp_mask.sum(), 1) < min_confident_ratio:
            damage = -1
        else:
            damage = int(np.bincount(valid).argmax()) - 1

        patches.append({'pre_patch': pre_patch, 'post_patch': post_patch,
                        'damage': damage, 'bbox': (x1, y1, x2-x1, y2-y1)})
    return patches


def extract_all_patches(img_dir, lbl_dir, output_dir,
                        mask_dir=None, use_gt_mask=True):
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)
    out_dir = Path(output_dir)
    (out_dir / 'pre').mkdir(parents=True, exist_ok=True)
    (out_dir / 'post').mkdir(parents=True, exist_ok=True)

    labels_dict, patch_count = {}, 0
    pre_images = sorted(img_dir.glob('*_pre_disaster.png'))
    print(f'Found {len(pre_images)} pre-disaster images...')

    for pre_path in tqdm(pre_images, desc='Extracting patches'):
        stem      = pre_path.stem
        post_path = img_dir / (stem.replace('_pre_', '_post_') + '.png')
        if not post_path.exists():
            continue

        if use_gt_mask:
            pre_json = lbl_dir / f'{stem}.json'
            if not pre_json.exists():
                continue
            img_tmp = cv2.imread(str(pre_path))
            h, w = img_tmp.shape[:2]
            mask = json_to_mask(str(pre_json), height=h, width=w)
        else:
            mask_path = Path(mask_dir) / f'{stem}_mask.png'
            if not mask_path.exists():
                continue
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        post_json = lbl_dir / (stem.replace('_pre_', '_post_') + '.json')
        patches   = extract_patches_from_pair(
            str(pre_path), str(post_path), mask,
            str(post_json) if post_json.exists() else None)

        for i, p in enumerate(patches):
            if p['damage'] == -1:
                continue
            pid = f'{stem}_{i:04d}'
            cv2.imwrite(str(out_dir / 'pre'  / f'{pid}.png'),
                        cv2.cvtColor(p['pre_patch'],  cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(out_dir / 'post' / f'{pid}.png'),
                        cv2.cvtColor(p['post_patch'], cv2.COLOR_RGB2BGR))
            labels_dict[pid] = p['damage']
            patch_count += 1

    with open(out_dir / 'labels.json', 'w') as f:
        json.dump(labels_dict, f)
    print(f'✓ {patch_count} labelled patches → {output_dir}')
    dist = Counter(labels_dict.values())
    for cid, name in Config.DAMAGE_CLASSES.items():
        print(f'  {cid} ({name}): {dist.get(cid, 0)}')

print('✓ Patch extraction defined')


# ─────────────────────────────────────────────────────────────────
# Model loaders
# ─────────────────────────────────────────────────────────────────
def load_seg_model(ckpt_path, device):
    model = build_segmentation_model()
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    print(f'✓ Seg model loaded (IoU={ckpt.get("best_iou", "?"):.4f})')
    return model


def load_cls_model(ckpt_path, device):
    model = DualResNet50(num_classes=Config.CLS_NUM_CLASSES, pretrained=False)
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    print(f'✓ Cls model loaded (F1={ckpt.get("best_f1", "?"):.4f})')
    return model

print('✓ Model loaders defined')


# ─────────────────────────────────────────────────────────────────
# Visualization
# ─────────────────────────────────────────────────────────────────
def draw_predictions(post_rgb, patches, predictions, alpha=0.5):
    vis = cv2.cvtColor(post_rgb, cv2.COLOR_RGB2BGR).copy()
    for p, pred in zip(patches, predictions):
        x, y, bw, bh = p['bbox']
        color = Config.DAMAGE_COLORS.get(pred, (128, 128, 128))
        label = Config.DAMAGE_CLASSES.get(pred, 'unknown')
        ov    = vis.copy()
        cv2.rectangle(ov,  (x, y), (x+bw, y+bh), color, -1)
        cv2.addWeighted(ov, alpha, vis, 1-alpha, 0, vis)
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), color, 2)
        cv2.putText(vis, label, (x, max(0, y-4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return vis


def draw_legend(vis):
    lh = 30 * Config.CLS_NUM_CLASSES + 10
    h, _ = vis.shape[:2]
    x0, y0 = 10, h - lh - 10
    ov = vis.copy()
    cv2.rectangle(ov, (x0, y0), (x0+180, y0+lh), (0, 0, 0), -1)
    cv2.addWeighted(ov, 0.6, vis, 0.4, 0, vis)
    for i, (cid, name) in enumerate(Config.DAMAGE_CLASSES.items()):
        c  = Config.DAMAGE_COLORS[cid]
        cy = y0 + 10 + i * 30
        cv2.rectangle(vis, (x0+5, cy), (x0+25, cy+18), c, -1)
        cv2.putText(vis, name, (x0+32, cy+13),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return vis

print('✓ Visualization helpers defined')


# ─────────────────────────────────────────────────────────────────
# End-to-end inference
# ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def classify_patches(cls_model, patches, device):
    tf    = get_cls_transforms(train=False)
    preds = []
    for p in patches:
        pre_t  = tf(image=p['pre_patch'])['image'].unsqueeze(0).to(device)
        post_t = tf(image=p['post_patch'])['image'].unsqueeze(0).to(device)
        with torch.amp.autocast('cuda'):
            logit = cls_model(pre_t, post_t)
        preds.append(logit.argmax(1).item())
    return preds


def run_inference(pre_path, post_path, seg_model, cls_model, device,
                  output_dir=Config.PRED_DIR, use_tta=True):
    pre_rgb  = cv2.cvtColor(cv2.imread(pre_path),  cv2.COLOR_BGR2RGB)
    post_rgb = cv2.cvtColor(cv2.imread(post_path), cv2.COLOR_BGR2RGB)

    print(f'  [1/3] Building mask ({"5-way TTA" if use_tta else "single"})...')
    if use_tta:
        mask = predict_mask_tta(seg_model, pre_rgb, post_rgb, device)
    else:
        seg_tf = A.Compose([
            A.Resize(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
        p   = seg_tf(image=pre_rgb)['image']
        q   = seg_tf(image=post_rgb)['image']
        inp = torch.cat([p, q], dim=0).unsqueeze(0).to(device)
        with torch.amp.autocast('cuda'):
            logit = seg_model(inp)
        prob = torch.sigmoid(logit).squeeze().cpu().numpy()
        mask = (prob > Config.SEG_THRESHOLD).astype(np.uint8) * 255
        orig_h, orig_w = pre_rgb.shape[:2]
        mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
        mask = postprocess_mask(mask)

    print('  [2/3] Extracting patches (watershed)...')
    patches = extract_patches_from_pair(pre_path, post_path, mask)
    print(f'         {len(patches)} buildings detected')

    if not patches:
        print('  WARNING: No buildings found.')
        return {'mask': mask, 'predictions': [], 'patches': [], 'vis_path': None}

    print('  [3/3] Classifying damage...')
    predictions = classify_patches(cls_model, patches, device)
    dist = Counter(predictions)
    for cid, name in Config.DAMAGE_CLASSES.items():
        print(f'         {name}: {dist.get(cid, 0)} buildings')

    vis  = draw_legend(draw_predictions(post_rgb, patches, predictions))
    stem = Path(pre_path).stem.replace('_pre_disaster', '')
    vp   = os.path.join(output_dir, f'{stem}_damage_map.png')
    cv2.imwrite(vp, vis)
    cv2.imwrite(os.path.join(output_dir, f'{stem}_building_mask.png'), mask)
    print(f'  ✓ Saved → {vp}')

    return {'mask': mask, 'predictions': predictions,
            'patches': patches, 'vis_path': vp}

print('✓ Inference pipeline defined')


# ─────────────────────────────────────────────────────────────────
# Full Pipeline
# ─────────────────────────────────────────────────────────────────
def run_full_pipeline(use_gt_mask=True):
    print('\n' + '═'*60)
    print('  🚀  DISASTER DAMAGE PIPELINE  v5')
    print('═'*60)

    print('\n── STAGE 1 ── DeepLabV3+ dual-input, scene-split, 5-way TTA')
    seg_ckpt = train_segmentation()

    print('\n── STAGE 1.5 ── Patch extraction (watershed + quality filter 0.5)')
    extract_all_patches(img_dir=Config.TRAIN_IMG_DIR,
                        lbl_dir=Config.TRAIN_LBL_DIR,
                        output_dir=Config.PATCH_DIR,
                        use_gt_mask=use_gt_mask)

    print('\n── STAGE 2 ── DualResNet50 (spatial-attn, multi-scale, HNM)')
    cls_ckpt = train_classification(patch_dir=Config.PATCH_DIR)

    print('\n' + '═'*60)
    print('  ✓  PIPELINE COMPLETE')
    print(f'  Seg ckpt : {seg_ckpt}')
    print(f'  Cls ckpt : {cls_ckpt}')
    print(f'  Outputs  : {Config.OUTPUT_DIR}')
    print('═'*60)
    return seg_ckpt, cls_ckpt

print('✓ Full pipeline defined')


# ─────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────
seg_ckpt, cls_ckpt = run_full_pipeline(use_gt_mask=True)

# ── OR run stages individually ────────────────────────────────────
# seg_ckpt = train_segmentation()
#
# extract_all_patches(
#     img_dir    = Config.TRAIN_IMG_DIR,
#     lbl_dir    = Config.TRAIN_LBL_DIR,
#     output_dir = Config.PATCH_DIR,
#     use_gt_mask= True,
# )
#
# cls_ckpt = train_classification()
#
# device    = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
# seg_model = load_seg_model('./outputs/checkpoints/seg_best.pth', device)
# cls_model = load_cls_model('./outputs/checkpoints/cls_best.pth', device)
# train_img = Path(Config.TRAIN_IMG_DIR)
# pre_path  = str(sorted(train_img.glob('*_pre_disaster.png'))[0])
# post_path = pre_path.replace('_pre_disaster', '_post_disaster')
# result    = run_inference(pre_path, post_path, seg_model, cls_model, device)

GPU Available: True
GPU Name: NVIDIA A100-PCIE-40GB
GPU Memory: 42.29 GB
BASE_DIR           : /home/devnurma/Satellite-based disaster damage
train/images exists: True
train/labels exists: True
test/images  exists: True

Train folder — 5598 total files
  First 6: ['guatemala-volcano_00000000_post_disaster.png', 'guatemala-volcano_00000000_pre_disaster.png', 'guatemala-volcano_00000001_post_disaster.png', 'guatemala-volcano_00000001_pre_disaster.png', 'guatemala-volcano_00000002_post_disaster.png', 'guatemala-volcano_00000002_pre_disaster.png']

Test  folder — 1866 total files
  First 6: ['test_post_00000.png', 'test_post_00001.png', 'test_post_00002.png', 'test_post_00003.png', 'test_post_00004.png', 'test_post_00005.png']

✓ All paths look good
✓ Output directories created
✓ Segmentation model builder defined
✓ DualResNet50 with spatial attention defined
✓ Loss functions defined
Using device: cuda
Seg model — Input: (2,6,512,512)  Output: torch.Size([2, 1, 512, 512])
Cls model — Output

/tmp/ipykernel_114171/4101767290.py:353: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50)),
/tmp/ipykernel_114171/4101767290.py:357: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32,


SegDataset (train): 2686 samples
SegDataset (val): 113 samples

 Epoch |     TLoss |     VLoss |    IoU |   Dice |        LR | Enc
────────────────────────────────────────────────────────────────────
  Encoder frozen
     1 |    0.5631 |    0.4240 | 0.3760 | 0.5258 |  1.21e-04 | frz  [254s]
  ✓ Saved (IoU=0.3760)
     2 |    0.4637 |    0.3561 | 0.4211 | 0.5718 |  1.80e-04 | frz  [252s]
  ✓ Saved (IoU=0.4211)
     3 |    0.4314 |    0.3376 | 0.4466 | 0.6010 |  2.40e-04 | frz  [246s]
  ✓ Saved (IoU=0.4466)
     4 |    0.4161 |    0.3522 | 0.4452 | 0.5986 |  3.00e-04 | frz  [239s]
     5 |    0.4091 |    0.3224 | 0.4698 | 0.6245 |  3.00e-04 | frz  [239s]
  ✓ Saved (IoU=0.4698)
  Encoder unfrozen


In [1]:
import os
import json
import cv2
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
from scipy.ndimage import distance_transform_edt
from sklearn.metrics import f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from torchvision import models

# ─────────────────────────────────────────────────────────────────
# GPU Check
# ─────────────────────────────────────────────────────────────────
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# ─────────────────────────────────────────────────────────────────
# Data paths
# ─────────────────────────────────────────────────────────────────
BASE_DIR = Path.home() / 'Satellite-based disaster damage'

print('BASE_DIR           :', BASE_DIR)
print('train/images exists:', (BASE_DIR / 'train' / 'images').exists())
print('train/labels exists:', (BASE_DIR / 'train' / 'labels').exists())
print('test/images  exists:', (BASE_DIR / 'test'  / 'images').exists())

train_img_dir = BASE_DIR / 'train' / 'images'
test_img_dir  = BASE_DIR / 'test'  / 'images'
train_files   = list(train_img_dir.glob('*')) if train_img_dir.exists() else []
test_files    = list(test_img_dir.glob('*'))  if test_img_dir.exists()  else []

print(f'\nTrain folder — {len(train_files)} total files')
print('  First 6:', [f.name for f in sorted(train_files)[:6]])
print(f'\nTest  folder — {len(test_files)} total files')
print('  First 6:', [f.name for f in sorted(test_files)[:6]])

assert BASE_DIR.exists(), f'DATA ROOT not found: {BASE_DIR}'
print('\n✓ All paths look good')


# ─────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────
class Config:
    DATA_ROOT      = BASE_DIR
    TRAIN_IMG_DIR  = str(DATA_ROOT / 'train' / 'images')
    TRAIN_LBL_DIR  = str(DATA_ROOT / 'train' / 'labels')
    TEST_IMG_DIR   = str(DATA_ROOT / 'test'  / 'images')
    OUTPUT_DIR     = './outputs'
    CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
    LOG_DIR        = os.path.join(OUTPUT_DIR, 'logs')
    VIZ_DIR        = os.path.join(OUTPUT_DIR, 'visualizations')
    PRED_DIR       = os.path.join(OUTPUT_DIR, 'predictions')
    PATCH_DIR      = os.path.join(OUTPUT_DIR, 'patches')

    # ── Segmentation ──────────────────────────────────────────────
    SEG_IMG_SIZE      = 640
    SEG_CROP_SIZE     = 512
    SEG_BATCH_SIZE    = 4
    SEG_LR            = 3e-4
    SEG_EPOCHS        = 50
    SEG_THRESHOLD     = 0.5
    SEG_MIN_AREA      = 100
    SEG_IN_CHANNELS   = 6           # pre(3) + post(3)
    SEG_WARMUP_EPOCHS = 5
    SEG_FREEZE_EPOCHS = 5
    VAL_SCENE_FRACTION = 0.15

    # ── Classification ────────────────────────────────────────────
    CLS_PATCH_SIZES = [192, 224, 256]   # multi-scale training
    CLS_PATCH_SIZE  = 224               # inference size
    CLS_BATCH_SIZE  = 16
    CLS_LR          = 1e-4
    CLS_EPOCHS      = 50
    CLS_NUM_CLASSES = 4
    CLS_VAL_FRAC    = 0.15

    # Hard-negative mining
    HNM_START_EPOCH = 5
    HNM_OVERSAMPLE  = 3

    DAMAGE_CLASSES = {0: 'no-damage', 1: 'minor-damage',
                      2: 'major-damage', 3: 'destroyed'}
    DAMAGE_COLORS  = {0: (0, 200, 0), 1: (0, 200, 255),
                      2: (0, 100, 255), 3: (0, 0, 255)}

    SEED   = 42
    DEVICE = 'cuda'


def make_dirs():
    for d in [Config.OUTPUT_DIR, Config.CHECKPOINT_DIR,
              Config.LOG_DIR, Config.VIZ_DIR, Config.PRED_DIR, Config.PATCH_DIR]:
        os.makedirs(d, exist_ok=True)
    print('✓ Output directories created')

make_dirs()


GPU Available: True
GPU Name: NVIDIA A100-PCIE-40GB
GPU Memory: 42.29 GB
BASE_DIR           : /home/devnurma/Satellite-based disaster damage
train/images exists: True
train/labels exists: True
test/images  exists: True

Train folder — 5598 total files
  First 6: ['guatemala-volcano_00000000_post_disaster.png', 'guatemala-volcano_00000000_pre_disaster.png', 'guatemala-volcano_00000001_post_disaster.png', 'guatemala-volcano_00000001_pre_disaster.png', 'guatemala-volcano_00000002_post_disaster.png', 'guatemala-volcano_00000002_pre_disaster.png']

Test  folder — 1866 total files
  First 6: ['test_post_00000.png', 'test_post_00001.png', 'test_post_00002.png', 'test_post_00003.png', 'test_post_00004.png', 'test_post_00005.png']

✓ All paths look good
✓ Output directories created


In [2]:
# ─────────────────────────────────────────────────────────────────
# Models
# ─────────────────────────────────────────────────────────────────
def build_segmentation_model(encoder_name='timm-efficientnet-b5',
                              encoder_weights='imagenet',
                              in_channels=Config.SEG_IN_CHANNELS,
                              num_classes=1) -> nn.Module:
    return smp.DeepLabV3Plus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,
    )

print('✓ Segmentation model builder defined')


class DualResNet50(nn.Module):
    """
    Siamese ResNet50 with spatial attention on post-disaster features.
    Fusion: cat(f_pre_pooled, f_post_attended_pooled, |diff|)
    """

    def __init__(self, num_classes=4, pretrained=True, dropout=0.4):
        super().__init__()
        weights  = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet50(weights=weights)
        self.encoder = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4,
        )  # → (B, 2048, 7, 7)

        # Spatial attention: 2048 → 512 → 1 per spatial position
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 1, kernel_size=1),
            nn.Sigmoid(),
        )  # → (B, 1, 7, 7)

        self.pool = nn.AdaptiveAvgPool2d(1)

        feat_dim = 2048 * 3   # pre + post_attended + |diff|
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout / 2),
            nn.Linear(256, num_classes),
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, pre, post):
        f_pre  = self.encode(pre)
        f_post = self.encode(post)

        attn   = self.spatial_attn(f_post)   # (B, 1, 7, 7)
        f_post = f_post * attn               # spatial weighting

        p_pre  = self.pool(f_pre).flatten(1)
        p_post = self.pool(f_post).flatten(1)
        diff   = torch.abs(p_post - p_pre)

        return self.classifier(torch.cat([p_pre, p_post, diff], dim=1))

print('✓ DualResNet50 with spatial attention defined')


✓ Segmentation model builder defined
✓ DualResNet50 with spatial attention defined


In [3]:
# ─────────────────────────────────────────────────────────────────
# Loss Functions
# ─────────────────────────────────────────────────────────────────
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, smooth=1e-6):
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth

    def forward(self, logits, targets):
        p  = torch.sigmoid(logits)
        TP = (p * targets).sum(dim=(2, 3))
        FP = ((1 - targets) * p).sum(dim=(2, 3))
        FN = (targets * (1 - p)).sum(dim=(2, 3))
        t  = (TP + self.smooth) / (TP + self.alpha*FP + self.beta*FN + self.smooth)
        return (1 - t).mean()


def compute_dist_map(mask_np: np.ndarray) -> np.ndarray:
    """Symmetric distance map: strong weight on both sides of building edges."""
    pos = distance_transform_edt(mask_np)
    neg = distance_transform_edt(1.0 - mask_np)
    d   = pos + neg
    return d / (d.max() + 1e-6)


class DistanceTransformLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        dist_maps = []
        for t in targets.cpu().numpy():
            dist_maps.append(compute_dist_map(t[0]))
        dist_maps = torch.tensor(
            np.stack(dist_maps), dtype=torch.float32
        ).unsqueeze(1).to(logits.device)
        return ((probs - targets) ** 2 * (1 + 5 * dist_maps)).mean()


class CombinedSegLoss(nn.Module):
    """0.4·BCE + 0.3·Dice + 0.2·Tversky + 0.1·SymDistBoundary"""
    def __init__(self):
        super().__init__()
        self.bce       = nn.BCEWithLogitsLoss()
        self.tversky   = TverskyLoss(alpha=0.7, beta=0.3)
        self.dist_bdry = DistanceTransformLoss()

    def _dice(self, logits, targets):
        p      = torch.sigmoid(logits)
        smooth = 1e-6
        inter  = (p * targets).sum(dim=(2, 3))
        return (1 - (2*inter + smooth) /
                (p.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + smooth)).mean()

    def forward(self, logits, targets):
        return (0.40 * self.bce(logits, targets) +
                0.30 * self._dice(logits, targets) +
                0.20 * self.tversky(logits, targets) +
                0.10 * self.dist_bdry(logits, targets))


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma, self.weight = gamma, weight

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        p_t = torch.exp(-ce)
        return ((1 - p_t) ** self.gamma * ce).mean()

print('✓ Loss functions defined')


✓ Loss functions defined


In [4]:
# ─────────────────────────────────────────────────────────────────
# Sanity check
# ─────────────────────────────────────────────────────────────────
device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

seg = build_segmentation_model()
out = seg(torch.randn(2, 6, 512, 512))
print(f'Seg model — Input: (2,6,512,512)  Output: {out.shape}')
del seg, out

cls = DualResNet50(num_classes=4)
out = cls(torch.randn(4, 3, 224, 224), torch.randn(4, 3, 224, 224))
print(f'Cls model — Output: {out.shape}')
del cls, out
print('✓ Both models initialised successfully')


# ─────────────────────────────────────────────────────────────────
# Mask helpers
# ─────────────────────────────────────────────────────────────────
def json_to_mask(json_path, height=1024, width=1024):
    with open(json_path) as f:
        data = json.load(f)
    mask = np.zeros((height, width), dtype=np.uint8)
    for feat in data.get('features', {}).get('xy', []):
        geom = feat.get('wkt', '')
        if not geom.startswith('POLYGON'):
            continue
        coords = geom.replace('POLYGON ((', '').replace('))', '').strip()
        pts = []
        for pair in coords.split(','):
            pair = pair.strip()
            if pair:
                x, y = pair.split()
                pts.append([float(x), float(y)])
        if len(pts) >= 3:
            cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], 255)
    return mask


def json_to_damage_mask(json_path, height=1024, width=1024):
    subtype_map = {'no-damage': 1, 'minor-damage': 2,
                   'major-damage': 3, 'destroyed': 4, 'un-classified': 0}
    with open(json_path) as f:
        data = json.load(f)
    mask = np.zeros((height, width), dtype=np.uint8)
    for feat in data.get('features', {}).get('xy', []):
        geom  = feat.get('wkt', '')
        label = subtype_map.get(
            feat.get('properties', {}).get('subtype', 'un-classified'), 0)
        if not geom.startswith('POLYGON') or label == 0:
            continue
        coords = geom.replace('POLYGON ((', '').replace('))', '').strip()
        pts = []
        for pair in coords.split(','):
            pair = pair.strip()
            if pair:
                x, y = pair.split()
                pts.append([float(x), float(y)])
        if len(pts) >= 3:
            cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], label)
    return mask

print('✓ Mask helpers defined')


# ─────────────────────────────────────────────────────────────────
# Transforms — spatial (ReplayCompose) + photometric (independent)
# ─────────────────────────────────────────────────────────────────
def get_spatial_transforms(train: bool = True):
    if train:
        return A.ReplayCompose([
            A.Resize(Config.SEG_IMG_SIZE, Config.SEG_IMG_SIZE),
            A.RandomCrop(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
        ])
    return A.ReplayCompose([
        A.Resize(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
    ])


def get_photometric_transforms():
    return A.Compose([
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
            A.CLAHE(clip_limit=4.0),
            A.RandomGamma(gamma_limit=(70, 130)),
        ], p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50)),
            A.MotionBlur(blur_limit=5),
            A.MedianBlur(blur_limit=5),
        ], p=0.3),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                        fill_value=0, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


_val_norm = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print('✓ Split spatial / photometric transforms defined')


Using device: cuda
Seg model — Input: (2,6,512,512)  Output: torch.Size([2, 1, 512, 512])
Cls model — Output: torch.Size([4, 4])
✓ Both models initialised successfully
✓ Mask helpers defined
✓ Split spatial / photometric transforms defined


In [5]:
# ─────────────────────────────────────────────────────────────────
# Scene-aware train / val split for segmentation
# ─────────────────────────────────────────────────────────────────
def disaster_scene_split(img_dir: str, val_fraction: float = 0.15, seed: int = 42):
    img_dir   = Path(img_dir)
    pre_files = sorted(img_dir.glob('*_pre_disaster.png'))

    scenes = defaultdict(list)
    for p in pre_files:
        parts  = p.stem.split('_pre_disaster')[0]
        tokens = parts.rsplit('_', 1)
        scene  = tokens[0] if len(tokens) > 1 else parts
        scenes[scene].append(str(p))

    scene_names = sorted(scenes.keys())
    rng = random.Random(seed)
    rng.shuffle(scene_names)

    n_val      = max(1, int(len(scene_names) * val_fraction))
    val_scenes = set(scene_names[:n_val])
    trn_scenes = set(scene_names[n_val:])

    train_files = [f for s in trn_scenes for f in scenes[s]]
    val_files   = [f for s in val_scenes  for f in scenes[s]]

    print(f'Scene split — {len(trn_scenes)} train scenes ({len(train_files)} samples) | '
          f'{len(val_scenes)} val scenes ({len(val_files)} samples)')
    return sorted(train_files), sorted(val_files)

print('✓ Scene-aware split defined')


# ─────────────────────────────────────────────────────────────────
# Segmentation Dataset
# ─────────────────────────────────────────────────────────────────
class SegmentationDataset(Dataset):
    def __init__(self, pre_paths: list, lbl_dir: str,
                 spatial_tf=None, photo_tf=None, is_train: bool = True):
        self.lbl_dir    = Path(lbl_dir)
        self.spatial_tf = spatial_tf
        self.photo_tf   = photo_tf
        self.is_train   = is_train

        self.samples = []
        for pre_path in pre_paths:
            pre_path  = Path(pre_path)
            stem      = pre_path.stem
            post_path = pre_path.parent / (stem.replace('_pre_', '_post_') + '.png')
            json_path = self.lbl_dir / f'{stem}.json'
            if post_path.exists() and json_path.exists():
                self.samples.append((str(pre_path), str(post_path), str(json_path)))

        print(f'SegDataset ({"train" if is_train else "val"}): '
              f'{len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pre_path, post_path, json_path = self.samples[idx]

        pre  = cv2.cvtColor(cv2.imread(pre_path),  cv2.COLOR_BGR2RGB)
        post = cv2.cvtColor(cv2.imread(post_path), cv2.COLOR_BGR2RGB)
        h, w = pre.shape[:2]

        mask = json_to_mask(json_path, height=h, width=w)
        mask = (mask > 0).astype(np.float32)

        if self.spatial_tf:
            # Same spatial ops on both images
            aug_pre = self.spatial_tf(image=pre, mask=mask)
            pre_sp  = aug_pre['image']
            mask    = aug_pre['mask']
            post_sp = A.ReplayCompose.replay(aug_pre['replay'], image=post)['image']
        else:
            pre_sp, post_sp = pre, post

        if self.photo_tf and self.is_train:
            # Independent photometric ops (different lighting per image)
            pre_t  = self.photo_tf(image=pre_sp)['image']
            post_t = self.photo_tf(image=post_sp)['image']
        else:
            pre_t  = _val_norm(image=pre_sp)['image']
            post_t = _val_norm(image=post_sp)['image']

        return torch.cat([pre_t, post_t], dim=0), torch.from_numpy(mask).unsqueeze(0)

print('✓ SegmentationDataset defined')


✓ Scene-aware split defined
✓ SegmentationDataset defined


In [6]:
# ─────────────────────────────────────────────────────────────────
# Classification transforms
# ─────────────────────────────────────────────────────────────────
def get_cls_transforms(train=True):
    norm = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    if train:
        return A.Compose([
            A.Resize(Config.CLS_PATCH_SIZE, Config.CLS_PATCH_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.3),
            A.OneOf([A.RandomBrightnessContrast(0.2, 0.2),
                     A.CLAHE(), A.RandomGamma()], p=0.4),
            A.OneOf([A.GaussNoise(var_limit=(5, 30)),
                     A.MotionBlur(blur_limit=3)], p=0.2),
            norm, ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(Config.CLS_PATCH_SIZE, Config.CLS_PATCH_SIZE),
        norm, ToTensorV2(),
    ])


# ─────────────────────────────────────────────────────────────────
# FIX (IndexError root cause):
# PatchClassificationDataset now accepts id_subset so train/val
# datasets are fully independent objects with no Subset wrapping.
# The sampler is then built from the *same* dataset object that the
# DataLoader uses, so indices always stay in range.
# ─────────────────────────────────────────────────────────────────
class PatchClassificationDataset(Dataset):
    """
    Parameters
    ----------
    patch_dir   : directory with pre/, post/, labels.json
    transform   : albumentations Compose applied to each patch
    id_subset   : if given, restrict dataset to these patch IDs only
    multiscale  : randomly resize crop to one of CLS_PATCH_SIZES before
                  transform (training only)
    """

    def __init__(self, patch_dir: str, transform=None,
                 id_subset=None, multiscale: bool = False):
        self.patch_dir  = Path(patch_dir)
        self.pre_dir    = self.patch_dir / 'pre'
        self.post_dir   = self.patch_dir / 'post'
        self.transform  = transform
        self.multiscale = multiscale

        labels_file = self.patch_dir / 'labels.json'
        if not labels_file.exists():
            raise FileNotFoundError(
                f"labels.json not found at '{labels_file}'. "
                "Run extract_all_patches() first.")

        with open(labels_file) as f:
            all_labels = json.load(f)

        # Filter to requested subset if provided
        if id_subset is not None:
            all_labels = {k: v for k, v in all_labels.items() if k in id_subset}

        self.labels = all_labels
        self.ids    = sorted(self.labels.keys())
        print(f'PatchDataset: {len(self.ids)} patches')

    def __len__(self):
        return len(self.ids)

    def get_labels(self) -> list:
        return [int(self.labels[i]) for i in self.ids]

    def __getitem__(self, idx):
        patch_id = self.ids[idx]
        label    = int(self.labels[patch_id])

        pre  = cv2.cvtColor(cv2.imread(str(self.pre_dir  / f'{patch_id}.png')),
                            cv2.COLOR_BGR2RGB)
        post = cv2.cvtColor(cv2.imread(str(self.post_dir / f'{patch_id}.png')),
                            cv2.COLOR_BGR2RGB)

        # Multi-scale: random resize before augmentation
        if self.multiscale:
            s    = random.choice(Config.CLS_PATCH_SIZES)
            pre  = cv2.resize(pre,  (s, s))
            post = cv2.resize(post, (s, s))

        if self.transform:
            pre  = self.transform(image=pre)['image']
            post = self.transform(image=post)['image']
        else:
            pre  = torch.from_numpy(pre.transpose(2, 0, 1)).float()  / 255.
            post = torch.from_numpy(post.transpose(2, 0, 1)).float() / 255.

        # Return patch_id so HNM can track misclassified samples
        return pre, post, torch.tensor(label, dtype=torch.long), patch_id

print('✓ PatchClassificationDataset (id_subset, multi-scale) defined')


# ─────────────────────────────────────────────────────────────────
# Class weights & sampler helpers
# ─────────────────────────────────────────────────────────────────
def compute_class_weights(patch_dir: str,
                          num_classes: int = Config.CLS_NUM_CLASSES) -> torch.Tensor:
    with open(Path(patch_dir) / 'labels.json') as f:
        labels = json.load(f)
    counts  = np.zeros(num_classes, dtype=np.float32)
    for v in labels.values():
        counts[int(v)] += 1
    counts  = np.where(counts == 0, 1, counts)
    weights = counts.sum() / (num_classes * counts)
    print('Class counts :', counts.astype(int))
    print('Class weights:', np.round(weights, 3))
    return torch.tensor(weights, dtype=torch.float32)


def build_weighted_sampler(dataset: PatchClassificationDataset,
                           hard_neg_ids: set = None,
                           num_classes: int = Config.CLS_NUM_CLASSES
                           ) -> WeightedRandomSampler:
    """
    Build a sampler over `dataset`.
    `hard_neg_ids` is a subset of dataset.ids — those samples get
    their weight multiplied by HNM_OVERSAMPLE.

    IMPORTANT: call this with the *exact* dataset object that the
    DataLoader will use so that len(sampler) == len(dataset).
    """
    all_labels = dataset.get_labels()
    counts     = np.bincount(all_labels, minlength=num_classes).astype(float)
    counts     = np.where(counts == 0, 1, counts)
    cls_weight = 1.0 / counts

    sample_w = []
    for pid, lbl in zip(dataset.ids, all_labels):
        w = cls_weight[lbl]
        if hard_neg_ids and pid in hard_neg_ids:
            w *= Config.HNM_OVERSAMPLE
        sample_w.append(w)

    return WeightedRandomSampler(sample_w,
                                 num_samples=len(sample_w),
                                 replacement=True)

print('✓ Class weight + sampler defined')


✓ PatchClassificationDataset (id_subset, multi-scale) defined
✓ Class weight + sampler defined


In [7]:
# ─────────────────────────────────────────────────────────────────
# Collate fn — keeps patch_id strings alongside tensors
# ─────────────────────────────────────────────────────────────────
def _collate_with_ids(batch):
    pre    = torch.stack([b[0] for b in batch])
    post   = torch.stack([b[1] for b in batch])
    labels = torch.stack([b[2] for b in batch])
    ids    = [b[3] for b in batch]
    return pre, post, labels, ids


# ─────────────────────────────────────────────────────────────────
# Segmentation metrics
# ─────────────────────────────────────────────────────────────────
def iou_score(pred, gt, thr=0.5):
    pred  = (pred > thr).astype(np.uint8)
    gt    = (gt   > thr).astype(np.uint8)
    return (pred & gt).sum() / ((pred | gt).sum() + 1e-6)


def dice_score(pred, gt, thr=0.5):
    pred  = (pred > thr).astype(np.uint8)
    gt    = (gt   > thr).astype(np.uint8)
    inter = (pred & gt).sum()
    return (2 * inter) / (pred.sum() + gt.sum() + 1e-6)

print('✓ Segmentation metrics defined')


# ─────────────────────────────────────────────────────────────────
# Watershed instance separation + post-processing
# ─────────────────────────────────────────────────────────────────
def separate_instances_watershed(mask: np.ndarray) -> np.ndarray:
    binary = (mask > 0).astype(np.uint8)
    dist   = cv2.distanceTransform(binary * 255, cv2.DIST_L2, 5)
    if dist.max() == 0:
        return mask

    _, sure_fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
    sure_fg    = np.uint8(sure_fg)

    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    sure_bg = cv2.dilate(binary * 255, kernel, iterations=2)
    unknown = cv2.subtract(sure_bg, sure_fg)

    _, markers = cv2.connectedComponents(sure_fg)
    markers    = markers + 1
    markers[unknown == 255] = 0

    img_bgr = cv2.cvtColor(binary * 255, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(img_bgr, markers)

    result = np.zeros_like(binary, dtype=np.uint8)
    result[markers > 1] = 255
    return result


def postprocess_mask(mask: np.ndarray,
                     min_area: int = Config.SEG_MIN_AREA,
                     use_watershed: bool = True) -> np.ndarray:
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    if use_watershed and cleaned.max() > 0:
        cleaned = separate_instances_watershed(cleaned)

    num_cc, labels_cc, stats, _ = cv2.connectedComponentsWithStats(cleaned)
    out = np.zeros_like(cleaned)
    for c in range(1, num_cc):
        if stats[c, cv2.CC_STAT_AREA] >= min_area:
            out[labels_cc == c] = 255
    return out

print('✓ Watershed post-processing defined')


✓ Segmentation metrics defined
✓ Watershed post-processing defined


In [8]:
# ─────────────────────────────────────────────────────────────────
# Warmup + Cosine LR scheduler
# ─────────────────────────────────────────────────────────────────
class WarmupCosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, total_epochs,
                 min_lr=1e-6, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.min_lr        = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        ep = self.last_epoch + 1
        if ep <= self.warmup_epochs:
            scale = ep / max(1, self.warmup_epochs)
        else:
            prog  = (ep - self.warmup_epochs) / max(
                1, self.total_epochs - self.warmup_epochs)
            scale = 0.5 * (1 + np.cos(np.pi * prog))
        return [self.min_lr + (b - self.min_lr) * scale for b in self.base_lrs]

print('✓ WarmupCosineScheduler defined')


# ─────────────────────────────────────────────────────────────────
# 5-way TTA for segmentation inference
# ─────────────────────────────────────────────────────────────────
def _tta_variants(pre_rgb, post_rgb):
    return [
        (pre_rgb,             post_rgb,             lambda x: x),
        (pre_rgb[:, ::-1],    post_rgb[:, ::-1],    lambda x: x[:, ::-1]),
        (pre_rgb[::-1, :],    post_rgb[::-1, :],    lambda x: x[::-1, :]),
        (np.rot90(pre_rgb,1), np.rot90(post_rgb,1), lambda x: np.rot90(x,-1)),
        (np.rot90(pre_rgb,2), np.rot90(post_rgb,2), lambda x: np.rot90(x,-2)),
    ]


@torch.no_grad()
def predict_mask_tta(seg_model, pre_rgb, post_rgb, device,
                     threshold=Config.SEG_THRESHOLD):
    orig_h, orig_w = pre_rgb.shape[:2]
    seg_tf = A.Compose([
        A.Resize(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

    def _prep(pre, post):
        p = seg_tf(image=np.ascontiguousarray(pre))['image']
        q = seg_tf(image=np.ascontiguousarray(post))['image']
        return torch.cat([p, q], dim=0).unsqueeze(0).to(device)

    prob_sum = None
    for pre_a, post_a, undo in _tta_variants(pre_rgb, post_rgb):
        with torch.amp.autocast('cuda'):
            logit = seg_model(_prep(pre_a, post_a))
        prob = undo(torch.sigmoid(logit).squeeze().cpu().numpy())
        prob_sum = prob if prob_sum is None else prob_sum + prob

    mask = ((prob_sum / 5) > threshold).astype(np.uint8) * 255
    mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    return postprocess_mask(mask)

print('✓ 5-way TTA defined')


✓ WarmupCosineScheduler defined
✓ 5-way TTA defined


In [9]:
# ─────────────────────────────────────────────────────────────────
# Segmentation training helpers
# ─────────────────────────────────────────────────────────────────
def train_seg_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def validate_seg(model, loader, criterion, device):
    model.eval()
    total, ious, dices = 0.0, [], []
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            total += criterion(logits, masks).item()
        probs = torch.sigmoid(logits).cpu().numpy()
        gt    = masks.cpu().numpy()
        for p, g in zip(probs, gt):
            ious.append(iou_score(p[0], g[0]))
            dices.append(dice_score(p[0], g[0]))
    return total / len(loader), float(np.mean(ious)), float(np.mean(dices))

print('✓ Segmentation training helpers defined')


# ─────────────────────────────────────────────────────────────────
# Segmentation Training Loop
# ─────────────────────────────────────────────────────────────────
def train_segmentation():
    device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Training on: {device}')

    train_paths, val_paths = disaster_scene_split(
        Config.TRAIN_IMG_DIR, Config.VAL_SCENE_FRACTION, Config.SEED)

    train_ds = SegmentationDataset(
        train_paths, Config.TRAIN_LBL_DIR,
        spatial_tf=get_spatial_transforms(train=True),
        photo_tf=get_photometric_transforms(), is_train=True)
    val_ds = SegmentationDataset(
        val_paths, Config.TRAIN_LBL_DIR,
        spatial_tf=get_spatial_transforms(train=False),
        photo_tf=None, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=Config.SEG_BATCH_SIZE,
                              shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=Config.SEG_BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    model     = build_segmentation_model().to(device)
    criterion = CombinedSegLoss()
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=Config.SEG_LR, weight_decay=1e-4)
    scheduler = WarmupCosineScheduler(optimizer,
                                      Config.SEG_WARMUP_EPOCHS,
                                      Config.SEG_EPOCHS, min_lr=1e-6)
    scaler    = torch.amp.GradScaler('cuda')
    writer    = SummaryWriter(os.path.join(Config.LOG_DIR, 'segmentation'))

    best_iou, best_ckpt   = 0.0, os.path.join(Config.CHECKPOINT_DIR, 'seg_best.pth')
    patience, patience_ctr = 10, 0

    print(f'\n{"Epoch":>6} | {"TLoss":>9} | {"VLoss":>9} | '
          f'{"IoU":>6} | {"Dice":>6} | {"LR":>9} | Enc')
    print('─' * 68)

    for epoch in range(1, Config.SEG_EPOCHS + 1):
        t0 = time.time()

        if epoch == 1:
            for p in model.encoder.parameters(): p.requires_grad = False
            print('  Encoder frozen')
        elif epoch == Config.SEG_FREEZE_EPOCHS + 1:
            for p in model.encoder.parameters(): p.requires_grad = True
            print('  Encoder unfrozen')

        enc = 'frz' if epoch <= Config.SEG_FREEZE_EPOCHS else 'act'
        tl  = train_seg_one_epoch(model, train_loader, optimizer,
                                  criterion, device, scaler)
        vl, iou, dice = validate_seg(model, val_loader, criterion, device)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        print(f'{epoch:>6} | {tl:>9.4f} | {vl:>9.4f} | '
              f'{iou:>6.4f} | {dice:>6.4f} | {lr:>9.2e} | {enc}  '
              f'[{time.time()-t0:.0f}s]')

        writer.add_scalars('Loss',   {'train': tl,   'val': vl},   epoch)
        writer.add_scalars('Metric', {'IoU': iou, 'Dice': dice},   epoch)

        if iou > best_iou:
            best_iou, patience_ctr = iou, 0
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'best_iou': best_iou}, best_ckpt)
            print(f'  ✓ Saved (IoU={best_iou:.4f})')
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    writer.close()
    print(f'\n✓ Seg done. Best IoU: {best_iou:.4f}  →  {best_ckpt}')
    return best_ckpt

print('✓ Segmentation training defined')

✓ Segmentation training helpers defined
✓ Segmentation training defined


In [10]:
# ─────────────────────────────────────────────────────────────────
# Classification metrics
# ─────────────────────────────────────────────────────────────────
def compute_cls_metrics(preds, labels, num_classes=Config.CLS_NUM_CLASSES):
    preds, labels = np.array(preds), np.array(labels)
    f1  = f1_score(labels, preds, average='macro',
                   labels=list(range(num_classes)), zero_division=0)
    cm  = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    acc = (preds == labels).mean()
    return f1, acc, cm


# ─────────────────────────────────────────────────────────────────
# Classification training helpers
# HNM: both train + val wrong IDs are tracked so the sampler can
# boost any hard example that lives in the training set.
# ─────────────────────────────────────────────────────────────────
def _run_one_epoch(model, loader, optimizer, criterion,
                   device, scaler, train: bool):
    """Unified train/val loop. Returns (loss, f1, acc, wrong_ids)."""
    model.train(train)
    total, all_preds, all_labels, wrong_ids = 0.0, [], [], set()

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for pre, post, labels, patch_ids in loader:
            pre, post, labels = (pre.to(device), post.to(device),
                                 labels.to(device))
            if train:
                optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                logits = model(pre, post)
                loss   = criterion(logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total += loss.item()
            preds  = logits.argmax(1).detach().cpu().numpy()
            gts    = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(gts)

            # Collect misclassified ids for HNM
            for pid, pred, gt in zip(patch_ids, preds, gts):
                if pred != gt:
                    wrong_ids.add(pid)

    f1, acc, cm = compute_cls_metrics(all_preds, all_labels)
    return total / len(loader), f1, acc, cm, wrong_ids

print('✓ Classification training helpers defined')


# ─────────────────────────────────────────────────────────────────
# Classification Training Loop
# ─────────────────────────────────────────────────────────────────
def train_classification(patch_dir: str = Config.PATCH_DIR):
    device = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Training on: {device}')

    # ── Build a clean train / val split at the ID level ──────────
    # Load all ids once
    with open(Path(patch_dir) / 'labels.json') as f:
        all_labels_dict = json.load(f)

    all_ids = sorted(all_labels_dict.keys())
    rng     = random.Random(Config.SEED)
    rng.shuffle(all_ids)

    n_val      = max(1, int(len(all_ids) * Config.CLS_VAL_FRAC))
    val_id_set = set(all_ids[:n_val])
    trn_id_set = set(all_ids[n_val:])

    print(f'Patch split — {len(trn_id_set)} train | {len(val_id_set)} val')

    # ── Create fully independent train / val dataset objects ──────
    # Each dataset object owns exactly the IDs assigned to it.
    # No Subset wrapping → sampler indices always match dataset size.
    train_ds = PatchClassificationDataset(
        patch_dir, transform=get_cls_transforms(train=True),
        id_subset=trn_id_set, multiscale=True)

    val_ds = PatchClassificationDataset(
        patch_dir, transform=get_cls_transforms(train=False),
        id_subset=val_id_set, multiscale=False)

    class_weights = compute_class_weights(patch_dir).to(device)
    model     = DualResNet50(num_classes=Config.CLS_NUM_CLASSES,
                             pretrained=True, dropout=0.4).to(device)
    criterion = FocalLoss(gamma=2.0, weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=Config.CLS_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.CLS_EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda')
    writer    = SummaryWriter(os.path.join(Config.LOG_DIR, 'classification'))

    best_f1, best_ckpt    = 0.0, os.path.join(Config.CHECKPOINT_DIR, 'cls_best.pth')
    patience, patience_ctr = 10, 0
    hard_neg_ids: set      = set()   # IDs in train_ds that were misclassified

    print(f'\n{"Epoch":>6} | {"T.Loss":>8} | {"T.F1":>6} | '
          f'{"V.Loss":>8} | {"V.F1":>6} | {"V.Acc":>6} | HN')
    print('─' * 72)

    for epoch in range(1, Config.CLS_EPOCHS + 1):
        t0 = time.time()

        # ── Rebuild sampler each epoch (HNM boost after warmup) ───
        # Sampler is built from train_ds: len(sampler) == len(train_ds) ✓
        use_hnm = epoch >= Config.HNM_START_EPOCH
        sampler = build_weighted_sampler(
            train_ds,
            hard_neg_ids=hard_neg_ids if use_hnm else set())

        train_loader = DataLoader(
            train_ds, batch_size=Config.CLS_BATCH_SIZE,
            sampler=sampler, num_workers=4, pin_memory=True,
            collate_fn=_collate_with_ids)

        val_loader = DataLoader(
            val_ds, batch_size=Config.CLS_BATCH_SIZE,
            shuffle=False, num_workers=4, pin_memory=True,
            collate_fn=_collate_with_ids)

        t_loss, t_f1, t_acc, _, train_wrong = _run_one_epoch(
            model, train_loader, optimizer, criterion, device, scaler, train=True)
        v_loss, v_f1, v_acc, cm, _ = _run_one_epoch(
            model, val_loader, optimizer, criterion, device, scaler, train=False)

        # HNM: track training-set misclassifications (not val)
        # so that boosted IDs actually appear in the training sampler
        if use_hnm:
            hard_neg_ids = train_wrong & trn_id_set
            print(f'  [HNM] {len(hard_neg_ids)} train hard negatives')

        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        print(f'{epoch:>6} | {t_loss:>8.4f} | {t_f1:>6.4f} | '
              f'{v_loss:>8.4f} | {v_f1:>6.4f} | {v_acc:>6.4f} | '
              f'{len(hard_neg_ids)}  [{time.time()-t0:.0f}s]')

        writer.add_scalars('Loss', {'train': t_loss, 'val': v_loss}, epoch)
        writer.add_scalars('F1',   {'train': t_f1,   'val': v_f1},   epoch)
        writer.add_scalar('HardNegatives', len(hard_neg_ids), epoch)

        if v_f1 > best_f1:
            best_f1, patience_ctr = v_f1, 0
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'best_f1': best_f1}, best_ckpt)
            print(f'  ✓ Saved (F1={best_f1:.4f})')
            print('  CM:', [row.tolist() for row in cm])
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    writer.close()
    print(f'\n✓ Cls done. Best F1: {best_f1:.4f}  →  {best_ckpt}')
    return best_ckpt

print('✓ Classification training defined')

✓ Classification training helpers defined
✓ Classification training defined


In [11]:
# ─────────────────────────────────────────────────────────────────
# Patch extraction
# ─────────────────────────────────────────────────────────────────
def extract_patches_from_pair(pre_img_path, post_img_path, mask,
                               post_json_path=None,
                               patch_size=Config.CLS_PATCH_SIZE,
                               min_area=Config.SEG_MIN_AREA,
                               min_confident_ratio=0.50):
    pre  = cv2.cvtColor(cv2.imread(pre_img_path),  cv2.COLOR_BGR2RGB)
    post = cv2.cvtColor(cv2.imread(post_img_path), cv2.COLOR_BGR2RGB)
    h, w = pre.shape[:2]

    if mask.shape != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    mask = postprocess_mask(mask, min_area=min_area, use_watershed=True)

    damage_mask = (json_to_damage_mask(post_json_path, h, w)
                   if post_json_path and Path(post_json_path).exists()
                   else np.full((h, w), 255, dtype=np.uint8))

    binary  = (mask > 0).astype(np.uint8) * 255
    num_cc, labels_cc, stats, _ = cv2.connectedComponentsWithStats(binary)

    patches = []
    for comp_id in range(1, num_cc):
        area = stats[comp_id, cv2.CC_STAT_AREA]
        if area < min_area:
            continue

        x, y   = stats[comp_id, cv2.CC_STAT_LEFT],  stats[comp_id, cv2.CC_STAT_TOP]
        bw, bh = stats[comp_id, cv2.CC_STAT_WIDTH], stats[comp_id, cv2.CC_STAT_HEIGHT]
        m = 10
        x1, y1 = max(0, x-m),   max(0, y-m)
        x2, y2 = min(w, x+bw+m), min(h, y+bh+m)

        pre_crop  = pre[y1:y2, x1:x2]
        post_crop = post[y1:y2, x1:x2]
        if pre_crop.size == 0 or post_crop.size == 0:
            continue

        pre_patch  = cv2.resize(pre_crop,  (patch_size, patch_size))
        post_patch = cv2.resize(post_crop, (patch_size, patch_size))

        comp_mask   = (labels_cc[y1:y2, x1:x2] == comp_id)
        damage_vals = damage_mask[y1:y2, x1:x2][comp_mask]
        valid       = damage_vals[(damage_vals >= 1) & (damage_vals <= 4)]

        if len(valid) == 0:
            damage = -1
        elif len(valid) / max(comp_mask.sum(), 1) < min_confident_ratio:
            damage = -1
        else:
            damage = int(np.bincount(valid).argmax()) - 1

        patches.append({'pre_patch': pre_patch, 'post_patch': post_patch,
                        'damage': damage, 'bbox': (x1, y1, x2-x1, y2-y1)})
    return patches


def extract_all_patches(img_dir, lbl_dir, output_dir,
                        mask_dir=None, use_gt_mask=True):
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)
    out_dir = Path(output_dir)
    (out_dir / 'pre').mkdir(parents=True, exist_ok=True)
    (out_dir / 'post').mkdir(parents=True, exist_ok=True)

    labels_dict, patch_count = {}, 0
    pre_images = sorted(img_dir.glob('*_pre_disaster.png'))
    print(f'Found {len(pre_images)} pre-disaster images...')

    for pre_path in tqdm(pre_images, desc='Extracting patches'):
        stem      = pre_path.stem
        post_path = img_dir / (stem.replace('_pre_', '_post_') + '.png')
        if not post_path.exists():
            continue

        if use_gt_mask:
            pre_json = lbl_dir / f'{stem}.json'
            if not pre_json.exists():
                continue
            img_tmp = cv2.imread(str(pre_path))
            h, w = img_tmp.shape[:2]
            mask = json_to_mask(str(pre_json), height=h, width=w)
        else:
            mask_path = Path(mask_dir) / f'{stem}_mask.png'
            if not mask_path.exists():
                continue
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        post_json = lbl_dir / (stem.replace('_pre_', '_post_') + '.json')
        patches   = extract_patches_from_pair(
            str(pre_path), str(post_path), mask,
            str(post_json) if post_json.exists() else None)

        for i, p in enumerate(patches):
            if p['damage'] == -1:
                continue
            pid = f'{stem}_{i:04d}'
            cv2.imwrite(str(out_dir / 'pre'  / f'{pid}.png'),
                        cv2.cvtColor(p['pre_patch'],  cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(out_dir / 'post' / f'{pid}.png'),
                        cv2.cvtColor(p['post_patch'], cv2.COLOR_RGB2BGR))
            labels_dict[pid] = p['damage']
            patch_count += 1

    with open(out_dir / 'labels.json', 'w') as f:
        json.dump(labels_dict, f)
    print(f'✓ {patch_count} labelled patches → {output_dir}')
    dist = Counter(labels_dict.values())
    for cid, name in Config.DAMAGE_CLASSES.items():
        print(f'  {cid} ({name}): {dist.get(cid, 0)}')

print('✓ Patch extraction defined')

✓ Patch extraction defined


In [12]:
# ─────────────────────────────────────────────────────────────────
# Model loaders
# ─────────────────────────────────────────────────────────────────
def load_seg_model(ckpt_path, device):
    model = build_segmentation_model()
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    print(f'✓ Seg model loaded (IoU={ckpt.get("best_iou", "?"):.4f})')
    return model


def load_cls_model(ckpt_path, device):
    model = DualResNet50(num_classes=Config.CLS_NUM_CLASSES, pretrained=False)
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    print(f'✓ Cls model loaded (F1={ckpt.get("best_f1", "?"):.4f})')
    return model

print('✓ Model loaders defined')


# ─────────────────────────────────────────────────────────────────
# Visualization
# ─────────────────────────────────────────────────────────────────
def draw_predictions(post_rgb, patches, predictions, alpha=0.5):
    vis = cv2.cvtColor(post_rgb, cv2.COLOR_RGB2BGR).copy()
    for p, pred in zip(patches, predictions):
        x, y, bw, bh = p['bbox']
        color = Config.DAMAGE_COLORS.get(pred, (128, 128, 128))
        label = Config.DAMAGE_CLASSES.get(pred, 'unknown')
        ov    = vis.copy()
        cv2.rectangle(ov,  (x, y), (x+bw, y+bh), color, -1)
        cv2.addWeighted(ov, alpha, vis, 1-alpha, 0, vis)
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), color, 2)
        cv2.putText(vis, label, (x, max(0, y-4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return vis


def draw_legend(vis):
    lh = 30 * Config.CLS_NUM_CLASSES + 10
    h, _ = vis.shape[:2]
    x0, y0 = 10, h - lh - 10
    ov = vis.copy()
    cv2.rectangle(ov, (x0, y0), (x0+180, y0+lh), (0, 0, 0), -1)
    cv2.addWeighted(ov, 0.6, vis, 0.4, 0, vis)
    for i, (cid, name) in enumerate(Config.DAMAGE_CLASSES.items()):
        c  = Config.DAMAGE_COLORS[cid]
        cy = y0 + 10 + i * 30
        cv2.rectangle(vis, (x0+5, cy), (x0+25, cy+18), c, -1)
        cv2.putText(vis, name, (x0+32, cy+13),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return vis

print('✓ Visualization helpers defined')


# ─────────────────────────────────────────────────────────────────
# End-to-end inference
# ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def classify_patches(cls_model, patches, device):
    tf    = get_cls_transforms(train=False)
    preds = []
    for p in patches:
        pre_t  = tf(image=p['pre_patch'])['image'].unsqueeze(0).to(device)
        post_t = tf(image=p['post_patch'])['image'].unsqueeze(0).to(device)
        with torch.amp.autocast('cuda'):
            logit = cls_model(pre_t, post_t)
        preds.append(logit.argmax(1).item())
    return preds


def run_inference(pre_path, post_path, seg_model, cls_model, device,
                  output_dir=Config.PRED_DIR, use_tta=True):
    pre_rgb  = cv2.cvtColor(cv2.imread(pre_path),  cv2.COLOR_BGR2RGB)
    post_rgb = cv2.cvtColor(cv2.imread(post_path), cv2.COLOR_BGR2RGB)

    print(f'  [1/3] Building mask ({"5-way TTA" if use_tta else "single"})...')
    if use_tta:
        mask = predict_mask_tta(seg_model, pre_rgb, post_rgb, device)
    else:
        seg_tf = A.Compose([
            A.Resize(Config.SEG_CROP_SIZE, Config.SEG_CROP_SIZE),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
        p   = seg_tf(image=pre_rgb)['image']
        q   = seg_tf(image=post_rgb)['image']
        inp = torch.cat([p, q], dim=0).unsqueeze(0).to(device)
        with torch.amp.autocast('cuda'):
            logit = seg_model(inp)
        prob = torch.sigmoid(logit).squeeze().cpu().numpy()
        mask = (prob > Config.SEG_THRESHOLD).astype(np.uint8) * 255
        orig_h, orig_w = pre_rgb.shape[:2]
        mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
        mask = postprocess_mask(mask)

    print('  [2/3] Extracting patches (watershed)...')
    patches = extract_patches_from_pair(pre_path, post_path, mask)
    print(f'         {len(patches)} buildings detected')

    if not patches:
        print('  WARNING: No buildings found.')
        return {'mask': mask, 'predictions': [], 'patches': [], 'vis_path': None}

    print('  [3/3] Classifying damage...')
    predictions = classify_patches(cls_model, patches, device)
    dist = Counter(predictions)
    for cid, name in Config.DAMAGE_CLASSES.items():
        print(f'         {name}: {dist.get(cid, 0)} buildings')

    vis  = draw_legend(draw_predictions(post_rgb, patches, predictions))
    stem = Path(pre_path).stem.replace('_pre_disaster', '')
    vp   = os.path.join(output_dir, f'{stem}_damage_map.png')
    cv2.imwrite(vp, vis)
    cv2.imwrite(os.path.join(output_dir, f'{stem}_building_mask.png'), mask)
    print(f'  ✓ Saved → {vp}')

    return {'mask': mask, 'predictions': predictions,
            'patches': patches, 'vis_path': vp}

print('✓ Inference pipeline defined')

✓ Model loaders defined
✓ Visualization helpers defined
✓ Inference pipeline defined


In [13]:
# ─────────────────────────────────────────────────────────────────
# Full Pipeline
# ─────────────────────────────────────────────────────────────────
def run_full_pipeline(use_gt_mask=True):
    print('\n' + '═'*60)
    print('  🚀  DISASTER DAMAGE PIPELINE  v5')
    print('═'*60)

    print('\n── STAGE 1 ── DeepLabV3+ dual-input, scene-split, 5-way TTA')
    seg_ckpt = train_segmentation()

    print('\n── STAGE 1.5 ── Patch extraction (watershed + quality filter 0.5)')
    extract_all_patches(img_dir=Config.TRAIN_IMG_DIR,
                        lbl_dir=Config.TRAIN_LBL_DIR,
                        output_dir=Config.PATCH_DIR,
                        use_gt_mask=use_gt_mask)

    print('\n── STAGE 2 ── DualResNet50 (spatial-attn, multi-scale, HNM)')
    cls_ckpt = train_classification(patch_dir=Config.PATCH_DIR)

    print('\n' + '═'*60)
    print('  ✓  PIPELINE COMPLETE')
    print(f'  Seg ckpt : {seg_ckpt}')
    print(f'  Cls ckpt : {cls_ckpt}')
    print(f'  Outputs  : {Config.OUTPUT_DIR}')
    print('═'*60)
    return seg_ckpt, cls_ckpt

print('✓ Full pipeline defined')


# ─────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────
seg_ckpt, cls_ckpt = run_full_pipeline(use_gt_mask=True)

# ── OR run stages individually ────────────────────────────────────
# seg_ckpt = train_segmentation()
#
# extract_all_patches(
#     img_dir    = Config.TRAIN_IMG_DIR,
#     lbl_dir    = Config.TRAIN_LBL_DIR,
#     output_dir = Config.PATCH_DIR,
#     use_gt_mask= True,
# )
#
# cls_ckpt = train_classification()
#
# device    = torch.device(Config.DEVICE if torch.cuda.is_available() else 'cpu')
# seg_model = load_seg_model('./outputs/checkpoints/seg_best.pth', device)
# cls_model = load_cls_model('./outputs/checkpoints/cls_best.pth', device)
# train_img = Path(Config.TRAIN_IMG_DIR)
# pre_path  = str(sorted(train_img.glob('*_pre_disaster.png'))[0])
# post_path = pre_path.replace('_pre_disaster', '_post_disaster')
# result    = run_inference(pre_path, post_path, seg_model, cls_model, device)

✓ Full pipeline defined

════════════════════════════════════════════════════════════
  🚀  DISASTER DAMAGE PIPELINE  v5
════════════════════════════════════════════════════════════

── STAGE 1 ── DeepLabV3+ dual-input, scene-split, 5-way TTA
Training on: cuda
Scene split — 9 train scenes (2686 samples) | 1 val scenes (113 samples)


/tmp/ipykernel_39466/2808308981.py:93: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50)),
/tmp/ipykernel_39466/2808308981.py:97: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32,


SegDataset (train): 2686 samples
SegDataset (val): 113 samples


/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



 Epoch |     TLoss |     VLoss |    IoU |   Dice |        LR | Enc
────────────────────────────────────────────────────────────────────
  Encoder frozen
     1 |    0.5761 |    0.4309 | 0.3782 | 0.5311 |  1.21e-04 | frz  [264s]
  ✓ Saved (IoU=0.3782)
     2 |    0.4657 |    0.3477 | 0.4267 | 0.5811 |  1.80e-04 | frz  [252s]
  ✓ Saved (IoU=0.4267)
     3 |    0.4295 |    0.3290 | 0.4535 | 0.6074 |  2.40e-04 | frz  [253s]
  ✓ Saved (IoU=0.4535)
     4 |    0.4119 |    0.3115 | 0.4712 | 0.6267 |  3.00e-04 | frz  [254s]
  ✓ Saved (IoU=0.4712)
     5 |    0.4080 |    0.3134 | 0.4598 | 0.6155 |  3.00e-04 | frz  [254s]
  Encoder unfrozen
     6 |    0.3865 |    0.2727 | 0.5140 | 0.6657 |  2.99e-04 | act  [278s]
  ✓ Saved (IoU=0.5140)
     7 |    0.3469 |    0.2506 | 0.5387 | 0.6883 |  2.97e-04 | act  [281s]
  ✓ Saved (IoU=0.5387)
     8 |    0.3294 |    0.2649 | 0.5419 | 0.6914 |  2.94e-04 | act  [281s]
  ✓ Saved (IoU=0.5419)
     9 |    0.3179 |    0.2689 | 0.5389 | 0.6873 |  2.91e-04 | act

...

  [HNM] 6350 train hard negatives
    48 |   0.0542 | 0.8232 |   0.8277 | 0.6243 | 0.6458 | 6350  [353s]


/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


  [HNM] 6441 train hard negatives
    49 |   0.0556 | 0.8218 |   0.8272 | 0.6261 | 0.6473 | 6441  [350s]


/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/devnurma/.conda/envs/damage-detection/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


  [HNM] 6409 train hard negatives
    50 |   0.0586 | 0.8231 |   0.8469 | 0.6252 | 0.6348 | 6409  [350s]

✓ Cls done. Best F1: 0.6408  →  ./outputs/checkpoints/cls_best.pth

════════════════════════════════════════════════════════════
  ✓  PIPELINE COMPLETE
  Seg ckpt : ./outputs/checkpoints/seg_best.pth
  Cls ckpt : ./outputs/checkpoints/cls_best.pth
  Outputs  : ./outputs
════════════════════════════════════════════════════════════
